In [ ]:
# %% CELL: CONFIG
# ============================================================
# CONFIG — EDIT THIS SECTION ONLY WHEN NECESSARY
# ============================================================

from pathlib import Path
from datetime import date

PROJECT_DIR_HINT = Path("/content/drive/MyDrive/Academic/RecycleTech")
ZIP_FILENAME = None
PROJECT_PATH_HINTS = ["آکادمیک", "ریسایکل", "recycle", "recycling", "academic"]

TARGET_TECHNOLOGY = (
    "design for disassembly and disassembly-enabled component separation, "
    "including non-destructive, tool-free, active, reversible, and modular disassembly"
)
FIELD_OF_INTEREST = "end-of-life recycling and product recyclability"
EMBEDDING_MODEL = "AI-Growth-Lab/PatentSBERTa"

RANDOM_SEED = 42
RESET_PIPELINE = False
ALLOW_CONFIG_MISMATCH = True

REPLAY_ALL_PREVIOUS_OUTPUTS_ON_START = True
REPLAY_ALL_OUTPUTS_AT_END = True
PRINT_FULL_ANALYTICAL_TABLES = True
PRINT_FULL_RAW_EXTRACTIONS = True
PRINT_CHUNK_LINES = 250

EMBED_BATCH_SIZE = 64
EMBED_SHARD_SIZE = 5000

UMAP_N_NEIGHBORS = 15
UMAP_N_COMPONENTS = 5
UMAP_MIN_DIST = 0.0
HDBSCAN_MIN_CLUSTER_SIZE_GRID = None

N_REP_REFERENCE_FIELDS = None

FUNCTION_TEXT_USES_CLAIMS_IF_PRESENT = True
FUNCTION_EXTRACTION_SHARD_SIZE = 1000
FUNCTION_MIN_FREQUENCY = 2
FUNCTION_NORMALIZATION_SIM_THRESHOLD = 0.70
MAX_FUNCTION_PHRASES_FOR_NORMALIZATION = 20000
N_REP_FUNCTIONS = 8

ZEROSHOT_THRESHOLD_OVERRIDE = None
ZEROSHOT_THRESHOLD_GRID = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]
ZEROSHOT_TARGET_ASSIGNMENT_RATE = 0.73

CIRCULARITY_ANCHORS = [
    "material recovery for recycling",
    "recyclability and recycling-oriented product design",
    "component separation at end of life",
    "closed-loop recycling",
    "recovery of materials from products",
    "design enabling reuse and remanufacturing",
    "end-of-life disassembly for recycling",
    "sorting and separation of recyclable materials",
]
SERVICE_ANCHORS = [
    "maintenance and servicing access",
    "repair of a product",
    "replacement of a failed component",
    "serviceability and maintenance disassembly",
    "assembly access for repair",
]

FIELD_TERMS = [
    "recycl", "recyclability", "recyclable", "material recovery",
    "closed-loop", "closed loop", "end-of-life", "end of life",
    "waste recovery", "material separation", "secondary material"
]

KEY_PATENTS_PER_CLUSTER = 10
KEY_SENTENCES_PER_PATENT = 5
NETWORK_PLOT_TOP_NODES = 30
NETWORK_PLOT_TOP_EDGES = 90
ANALYZE_ALL_KEY_CLUSTERS = True

ENABLE_LOCAL_LLM_IDEATION = False
LOCAL_LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
N_LLM_CLUSTERS = 5
N_SOLUTIONS_PER_CLUSTER = 3

ANALYSIS_DATE_OVERRIDE = None


# %% CELL: INITIALIZATION
# ============================================================
# INITIALIZATION, DRIVE, REPORTING, CHECKPOINTS
# ============================================================

import os, sys, json, re, csv, math, time, hashlib, shutil, zipfile, warnings, subprocess
from datetime import datetime, timezone
from itertools import combinations
from collections import Counter, defaultdict
from io import BytesIO
from urllib.parse import urljoin

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import networkx as nx
import joblib
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as e:
    raise RuntimeError("This pipeline is designed for Google Colab with Google Drive.") from e

DRIVE_ROOT = Path("/content/drive/MyDrive")

def normalize_path_text(s):
    return re.sub(r"\s+", " ", str(s).lower()).strip()

def resolve_project_dir():
    if PROJECT_DIR_HINT.exists():
        return PROJECT_DIR_HINT
    candidates = []
    for p in DRIVE_ROOT.rglob("*"):
        if p.is_dir():
            t = normalize_path_text(p)
            score = sum(1 for h in PROJECT_PATH_HINTS if normalize_path_text(h) in t)
            if score:
                candidates.append((score, len(str(p)), p))
    if not candidates:
        raise FileNotFoundError(
            f"Project folder not found. Expected: {PROJECT_DIR_HINT}. "
            "Edit PROJECT_DIR_HINT in CONFIG."
        )
    candidates.sort(key=lambda x: (-x[0], x[1]))
    return candidates[0][2]

PROJECT_DIR = resolve_project_dir()

zip_candidates = sorted(PROJECT_DIR.glob("*.zip"), key=lambda p: p.stat().st_mtime, reverse=True)
if ZIP_FILENAME:
    ZIP_PATH = PROJECT_DIR / ZIP_FILENAME
    if not ZIP_PATH.exists():
        raise FileNotFoundError(f"Configured ZIP not found: {ZIP_PATH}")
else:
    if not zip_candidates:
        zip_candidates = sorted(PROJECT_DIR.rglob("*.zip"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not zip_candidates:
        raise FileNotFoundError(f"No ZIP file found in {PROJECT_DIR}")
    ZIP_PATH = zip_candidates[0]

OUTPUT_ROOT = PROJECT_DIR / "TOA_Recycle_Results"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
DATA_DIR = OUTPUT_ROOT / "data"
CACHE_DIR = OUTPUT_ROOT / "cache"
EMBED_DIR = OUTPUT_ROOT / "embeddings"
TABLE_DIR = OUTPUT_ROOT / "tables"
TABLE_TEXT_DIR = OUTPUT_ROOT / "tables_text"
FIG_DIR = OUTPUT_ROOT / "figures"
REPORT_DIR = OUTPUT_ROOT / "reports"
NETWORK_DIR = OUTPUT_ROOT / "networks"
LLM_DIR = OUTPUT_ROOT / "llm"
ENV_DIR = OUTPUT_ROOT / "environment"
ARCHIVE_DIR = OUTPUT_ROOT / "archives"

for d in [
    OUTPUT_ROOT, CHECKPOINT_DIR, DATA_DIR, CACHE_DIR, EMBED_DIR, TABLE_DIR,
    TABLE_TEXT_DIR, FIG_DIR, REPORT_DIR, NETWORK_DIR, LLM_DIR, ENV_DIR, ARCHIVE_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(CACHE_DIR / "huggingface")
os.environ["TRANSFORMERS_CACHE"] = str(CACHE_DIR / "huggingface" / "transformers")

MANIFEST_PATH = CHECKPOINT_DIR / "manifest.json"
MASTER_REPORT_PATH = REPORT_DIR / "MASTER_REPORT.md"
CONFIG_PATH = CHECKPOINT_DIR / "run_config.json"
RUN_LOG_PATH = REPORT_DIR / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

def atomic_write_text(path, text):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(str(text), encoding="utf-8")
    os.replace(tmp, path)

def atomic_write_json(path, obj):
    atomic_write_text(path, json.dumps(obj, ensure_ascii=False, indent=2, default=str))

class Reporter:
    def __init__(self):
        self.log_path = RUN_LOG_PATH

    def emit(self, text=""):
        text = str(text)
        print(text)
        with self.log_path.open("a", encoding="utf-8") as f:
            f.write(text + "\n")

    def update_section(self, key, title, body):
        start = f"<!-- SECTION:{key}:START -->"
        end = f"<!-- SECTION:{key}:END -->"
        block = f"{start}\n## {title}\n\n{body.strip()}\n{end}\n"
        old = MASTER_REPORT_PATH.read_text(encoding="utf-8") if MASTER_REPORT_PATH.exists() else "# MASTER TOA REPORT\n\n"
        pattern = re.escape(start) + r".*?" + re.escape(end)
        if re.search(pattern, old, flags=re.S):
            new = re.sub(pattern, block.strip(), old, flags=re.S)
        else:
            new = old.rstrip() + "\n\n" + block
        atomic_write_text(MASTER_REPORT_PATH, new)

    def replay_all_saved(self):
        self.emit("\n" + "="*110)
        self.emit("REPLAYING ALL PREVIOUSLY SAVED ANALYTICAL OUTPUTS")
        self.emit("="*110)
        if MASTER_REPORT_PATH.exists():
            self.emit(MASTER_REPORT_PATH.read_text(encoding="utf-8"))
        for p in sorted(TABLE_TEXT_DIR.glob("*.txt")):
            self.emit(f"\n===== SAVED TABLE: {p.name} =====")
            self.emit(p.read_text(encoding="utf-8", errors="replace"))
        for p in sorted(FIG_DIR.glob("*.txt")):
            self.emit(f"\n===== SAVED FIGURE TEXT: {p.name} =====")
            self.emit(p.read_text(encoding="utf-8", errors="replace"))
        self.emit("="*110 + "\n")

R = Reporter()

def chunked_print_text(text, lines_per_chunk=PRINT_CHUNK_LINES):
    lines = str(text).splitlines()
    for i in range(0, len(lines), lines_per_chunk):
        R.emit("\n".join(lines[i:i+lines_per_chunk]))

def save_table(df, name, print_full=None, index=False):
    if print_full is None:
        print_full = PRINT_FULL_ANALYTICAL_TABLES
    csv_path = TABLE_DIR / f"{name}.csv"
    pq_path = TABLE_DIR / f"{name}.parquet"
    txt_path = TABLE_TEXT_DIR / f"{name}.txt"
    df.to_csv(csv_path, index=index)
    try:
        df.to_parquet(pq_path, index=index)
    except Exception as e:
        R.emit(f"[WARN] Parquet save failed for {name}: {e}")
    text = df.to_string(index=index)
    atomic_write_text(txt_path, text)
    R.emit(f"\nTABLE SAVED: {name} | rows={len(df):,} cols={len(df.columns):,}")
    R.emit(f"CSV: {csv_path}")
    R.emit(f"PARQUET: {pq_path}")
    R.emit(f"TEXT: {txt_path}")
    if print_full:
        chunked_print_text(text)
    return csv_path

def save_figure(fig, name, text_summary, data=None):
    png = FIG_DIR / f"{name}.png"
    pdf = FIG_DIR / f"{name}.pdf"
    txt = FIG_DIR / f"{name}.txt"
    fig.savefig(png, dpi=320, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    plt.close(fig)
    atomic_write_text(txt, text_summary.strip())
    if data is not None:
        data.to_csv(FIG_DIR / f"{name}_data.csv", index=False)
    R.emit(f"\nFIGURE SAVED: {name}")
    R.emit(f"PNG: {png}")
    R.emit(f"PDF: {pdf}")
    R.emit("TEXTUAL FIGURE SUMMARY:")
    R.emit(text_summary.strip())

def load_manifest():
    if MANIFEST_PATH.exists():
        return json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    return {"stages": {}, "created_at": datetime.now(timezone.utc).isoformat()}

MANIFEST = load_manifest()

def mark_stage_done(stage, outputs=None, extra=None):
    MANIFEST["stages"][stage] = {
        "done": True,
        "completed_at": datetime.now(timezone.utc).isoformat(),
        "outputs": [str(x) for x in (outputs or [])],
        "extra": extra or {},
    }
    atomic_write_json(MANIFEST_PATH, MANIFEST)

def stage_done(stage, required=None):
    info = MANIFEST.get("stages", {}).get(stage, {})
    if not info.get("done"):
        return False
    required = required or []
    return all(Path(p).exists() for p in required)

def stage_banner(name):
    R.emit("\n" + "#"*110)
    R.emit(f"STAGE: {name}")
    R.emit("#"*110)

def safe_z(x):
    s = pd.Series(x, dtype=float)
    sd = s.std(ddof=0)
    if not np.isfinite(sd) or sd == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.mean()) / sd

def l2_normalize(x, axis=1, eps=1e-12):
    x = np.asarray(x, dtype=np.float32)
    n = np.linalg.norm(x, axis=axis, keepdims=True)
    return x / np.maximum(n, eps)

if CONFIG_PATH.exists():
    old_cfg = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    persisted_analysis_date = old_cfg.get("analysis_date")
else:
    old_cfg = None
    persisted_analysis_date = None

ANALYSIS_DATE = ANALYSIS_DATE_OVERRIDE or persisted_analysis_date or date.today().isoformat()

CORE_CONFIG = {
    "target_technology": TARGET_TECHNOLOGY,
    "field_of_interest": FIELD_OF_INTEREST,
    "embedding_model": EMBEDDING_MODEL,
    "random_seed": RANDOM_SEED,
    "umap_n_neighbors": UMAP_N_NEIGHBORS,
    "umap_n_components": UMAP_N_COMPONENTS,
    "umap_min_dist": UMAP_MIN_DIST,
    "function_normalization_sim_threshold": FUNCTION_NORMALIZATION_SIM_THRESHOLD,
    "n_rep_functions": N_REP_FUNCTIONS,
    "zeroshot_threshold_override": ZEROSHOT_THRESHOLD_OVERRIDE,
    "analysis_date": ANALYSIS_DATE,
    "zip_path": str(ZIP_PATH),
    "zip_size": ZIP_PATH.stat().st_size,
    "zip_mtime": ZIP_PATH.stat().st_mtime,
}
CONFIG_HASH = hashlib.sha256(json.dumps(CORE_CONFIG, sort_keys=True).encode()).hexdigest()

if old_cfg and old_cfg.get("config_hash") != CONFIG_HASH:
    msg = (
        "Core analysis configuration differs from the saved run.\n"
        f"Saved hash: {old_cfg.get('config_hash')}\nCurrent hash: {CONFIG_HASH}\n"
        "Set RESET_PIPELINE=True for a clean checkpoint reset, or "
        "ALLOW_CONFIG_MISMATCH=True only if artifacts remain compatible."
    )
    if RESET_PIPELINE:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        backup = ARCHIVE_DIR / f"checkpoint_backup_{ts}"
        backup.mkdir(parents=True, exist_ok=True)
        if MANIFEST_PATH.exists():
            shutil.copy2(MANIFEST_PATH, backup / MANIFEST_PATH.name)
        if CONFIG_PATH.exists():
            shutil.copy2(CONFIG_PATH, backup / CONFIG_PATH.name)
        MANIFEST = {"stages": {}, "created_at": datetime.now(timezone.utc).isoformat()}
        atomic_write_json(MANIFEST_PATH, MANIFEST)
        R.emit("RESET_PIPELINE=True: checkpoint manifest reset; existing files are preserved.")
    elif not ALLOW_CONFIG_MISMATCH:
        raise RuntimeError(msg)
    else:
        R.emit("[WARNING] " + msg)

atomic_write_json(CONFIG_PATH, {**CORE_CONFIG, "config_hash": CONFIG_HASH})

R.emit(f"PROJECT_DIR: {PROJECT_DIR}")
R.emit(f"ZIP_PATH: {ZIP_PATH}")
R.emit(f"OUTPUT_ROOT: {OUTPUT_ROOT}")
R.emit(f"ANALYSIS_DATE: {ANALYSIS_DATE}")
R.emit(f"CUDA AVAILABLE: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    R.emit(f"GPU: {torch.cuda.get_device_name(0)}")
    R.emit(f"CUDA: {torch.version.cuda}")

if REPLAY_ALL_PREVIOUS_OUTPUTS_ON_START:
    R.replay_all_saved()


# %% CELL: UTILITIES
# ============================================================
# DATA / EMBEDDING / TEXT UTILITIES
# ============================================================

def normalize_colname(c):
    c = str(c).strip().lower()
    c = re.sub(r"[^a-z0-9]+", "_", c).strip("_")
    return c

def detect_csv_settings(path):
    encodings = ["utf-8-sig", "utf-8", "cp1252", "latin-1"]
    raw = Path(path).read_bytes()[:200_000]
    for enc in encodings:
        try:
            sample = raw.decode(enc)
            try:
                dialect = csv.Sniffer().sniff(sample, delimiters=[",", "\t", ";", "|"])
                return enc, dialect.delimiter
            except Exception:
                return enc, ","
        except Exception:
            continue
    return "latin-1", ","

def read_tabular(path, nrows=None):
    path = Path(path)
    ext = path.suffix.lower()
    if ext in [".xlsx", ".xls"]:
        return pd.read_excel(path, nrows=nrows)
    if ext == ".parquet":
        df = pd.read_parquet(path)
        return df.head(nrows) if nrows else df
    if ext in [".csv", ".tsv", ".txt"]:
        enc, sep = detect_csv_settings(path)
        return pd.read_csv(path, encoding=enc, sep=sep, low_memory=False, nrows=nrows)
    raise ValueError(f"Unsupported data file: {path}")

def choose_and_extract_data_from_zip(zip_path):
    extract_dir = CACHE_DIR / "raw_zip"
    extract_dir.mkdir(parents=True, exist_ok=True)
    choice_meta = CHECKPOINT_DIR / "zip_member_choice.json"
    if choice_meta.exists():
        meta = json.loads(choice_meta.read_text(encoding="utf-8"))
        chosen = extract_dir / meta["member_basename"]
        if chosen.exists():
            return chosen

    valid_ext = {".csv", ".tsv", ".txt", ".xlsx", ".xls", ".parquet"}
    with zipfile.ZipFile(zip_path) as zf:
        infos = [i for i in zf.infolist() if Path(i.filename).suffix.lower() in valid_ext and not i.is_dir()]
        if not infos:
            raise FileNotFoundError("No CSV/XLSX/Parquet-like file found inside ZIP.")
        scored = []
        for info in infos:
            score = 0
            name = normalize_path_text(info.filename)
            for kw in ["patent", "lens", "export", "recycl"]:
                score += int(kw in name)
            scored.append((score, info.file_size, info))
        scored.sort(key=lambda x: (-x[0], -x[1]))
        info = scored[0][2]
        target = extract_dir / Path(info.filename).name
        if not target.exists() or target.stat().st_size != info.file_size:
            with zf.open(info) as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst)

    atomic_write_json(choice_meta, {
        "zip": str(zip_path),
        "chosen_member": info.filename,
        "member_basename": target.name,
        "member_size": info.file_size,
    })
    return target

def split_multi(v):
    if pd.isna(v):
        return []
    s = str(v).strip()
    if not s:
        return []
    parts = re.split(r"[\n\r;|]+", s)
    return [p.strip() for p in parts if p.strip()]

def canonical_family_seed(v):
    items = split_multi(v)
    return "||".join(sorted(set(items))) if items else ""

CPC_RE = re.compile(r"\b([A-HY]\d{2}[A-Z](?:\s*\d+(?:/\d+)?)?|[A-HY]\d{2})\b", flags=re.I)

def parse_cpc_codes(v):
    if pd.isna(v):
        return []
    codes = []
    for m in CPC_RE.findall(str(v).upper()):
        codes.append(re.sub(r"\s+", "", m))
    return sorted(set(codes))

def cpc_class(code):
    code = str(code).strip().upper()
    return code[:3] if re.match(r"^[A-HY]\d{2}", code) else None

def cpc_subclass(code):
    code = str(code).strip().upper()
    return code[:4] if re.match(r"^[A-HY]\d{2}[A-Z]", code) else None

_EMBEDDER = None

def get_embedder():
    global _EMBEDDER
    if _EMBEDDER is None:
        from sentence_transformers import SentenceTransformer
        device = "cuda" if torch.cuda.is_available() else "cpu"
        R.emit(f"Loading {EMBEDDING_MODEL} on {device} ...")
        _EMBEDDER = SentenceTransformer(EMBEDDING_MODEL, device=device)
        try:
            _EMBEDDER.max_seq_length = 512
        except Exception:
            pass
    return _EMBEDDER

def encode_texts_sharded(texts, prefix, shard_size=EMBED_SHARD_SIZE, batch_size=EMBED_BATCH_SIZE):
    texts = list(texts)
    if len(texts) == 0:
        raise ValueError(f"No texts supplied for embedding prefix={prefix}")
    shard_dir = EMBED_DIR / f"{prefix}_shards"
    shard_dir.mkdir(parents=True, exist_ok=True)
    model = get_embedder()
    shards = []

    for start in range(0, len(texts), shard_size):
        end = min(start + shard_size, len(texts))
        path = shard_dir / f"{prefix}_{start:09d}_{end:09d}.npy"
        shards.append((start, end, path))
        if path.exists():
            try:
                arr = np.load(path, mmap_mode="r")
                if arr.shape[0] == end-start:
                    R.emit(f"[embedding shard cached] {path.name}")
                    continue
            except Exception:
                pass

        R.emit(f"Encoding {prefix}: rows {start:,}..{end-1:,}")
        arr = model.encode(
            texts[start:end],
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype("float32")
        tmp = path.with_suffix(".tmp.npy")
        np.save(tmp, arr)
        os.replace(tmp, path)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    final = EMBED_DIR / f"{prefix}.npy"
    first = np.load(shards[0][2], mmap_mode="r")
    dim = first.shape[1]
    mm = np.lib.format.open_memmap(final, mode="w+", dtype="float32", shape=(len(texts), dim))
    for start, end, path in shards:
        mm[start:end] = np.load(path, mmap_mode="r")
    del mm
    return final

def sentence_split_simple(text):
    text = re.sub(r"\s+", " ", str(text or "")).strip()
    if not text:
        return []
    parts = re.split(r"(?<=[.!?;])\s+(?=[A-Z0-9])", text)
    return [p.strip() for p in parts if len(p.strip()) >= 20]

def package_environment_snapshot():
    try:
        freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
        atomic_write_text(ENV_DIR / "pip_freeze.txt", freeze)
    except Exception as e:
        R.emit(f"[WARN] pip freeze failed: {e}")
    try:
        smi = subprocess.check_output(["nvidia-smi"], text=True)
        atomic_write_text(ENV_DIR / "nvidia_smi.txt", smi)
        R.emit(smi)
    except Exception as e:
        R.emit(f"[WARN] nvidia-smi unavailable: {e}")


# %% CELL: STAGES_01_02
# ============================================================
# INGEST, FAMILY DEDUP, DESCRIPTIVES
# ============================================================

RAW_PARQUET = DATA_DIR / "01_raw_normalized.parquet"
FAMILY_PARQUET = DATA_DIR / "02_family_deduplicated.parquet"
FAMILY_MAP_PARQUET = DATA_DIR / "02_family_row_map.parquet"

def stage_01_ingest():
    stage = "01_ingest"
    stage_banner(stage)
    if stage_done(stage, [RAW_PARQUET]):
        R.emit("Checkpoint found — ingest skipped.")
        return

    src = choose_and_extract_data_from_zip(ZIP_PATH)
    R.emit(f"Selected data file: {src}")
    df = read_tabular(src)
    original_cols = list(df.columns)
    normalized_cols = [normalize_colname(c) for c in original_cols]
    df.columns = normalized_cols

    colmap = pd.DataFrame({"original_column": original_cols, "normalized_column": normalized_cols})
    save_table(colmap, "01_column_mapping", print_full=True)

    df.insert(0, "source_row_id", np.arange(len(df), dtype=np.int64))
    df.to_parquet(RAW_PARQUET, index=False)

    null_summary = pd.DataFrame({
        "column": df.columns,
        "non_null": [df[c].notna().sum() for c in df.columns],
        "null": [df[c].isna().sum() for c in df.columns],
    })
    null_summary["null_pct"] = null_summary["null"] / max(len(df), 1)
    save_table(null_summary, "01_missingness", print_full=True)

    body = f"""
Raw Lens export was loaded from `{src.name}`. The normalized dataset contains **{len(df):,} rows**
and **{len(df.columns):,} columns**. Column names were converted to snake_case for reproducibility.

The persisted working copy is `{RAW_PARQUET.name}`.
"""
    R.update_section("01_ingest", "Data ingestion", body)
    R.emit(body)
    mark_stage_done(stage, [RAW_PARQUET])

def stage_02_family_dedup():
    stage = "02_family_dedup"
    stage_banner(stage)
    if stage_done(stage, [FAMILY_PARQUET, FAMILY_MAP_PARQUET]):
        R.emit("Checkpoint found — family deduplication skipped.")
        return

    df = pd.read_parquet(RAW_PARQUET)

    for c in ["lens_id", "title", "abstract", "simple_family_members", "cpc_classifications"]:
        if c not in df.columns:
            df[c] = np.nan

    for c in ["publication_date", "application_date", "earliest_priority_date"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")

    numeric_cols = [
        "cited_by_patent_count", "cites_patent_count", "simple_family_size",
        "extended_family_size", "npl_citation_count", "npl_resolved_citation_count"
    ]
    for c in numeric_cols:
        if c not in df.columns:
            df[c] = np.nan
        df[c] = pd.to_numeric(df[c], errors="coerce")

    seeds = []
    for _, row in df.iterrows():
        fam_seed = canonical_family_seed(row.get("simple_family_members"))
        if not fam_seed:
            fam_seed = f"LENS::{row.get('lens_id')}" if pd.notna(row.get("lens_id")) else f"ROW::{row['source_row_id']}"
        seeds.append(hashlib.sha1(fam_seed.encode("utf-8")).hexdigest())
    df["family_id"] = seeds

    title_s = df["title"].fillna("").astype(str)
    abs_s = df["abstract"].fillna("").astype(str)
    df["_text_len"] = title_s.str.len() + abs_s.str.len()
    df["_has_abstract"] = (abs_s.str.len() > 40).astype(int)
    df["_rep_score"] = df["_has_abstract"] * 10_000_000 + df["_text_len"]

    rep = (
        df.sort_values(["family_id", "_rep_score"], ascending=[True, False])
          .drop_duplicates("family_id", keep="first")
          .copy()
    )

    g = df.groupby("family_id", sort=False)
    agg = pd.DataFrame(index=rep["family_id"].values)
    agg.index.name = "family_id"
    agg["source_row_count"] = g.size()
    agg["family_cited_by_max"] = g["cited_by_patent_count"].max()
    agg["family_simple_size_max"] = g["simple_family_size"].max()

    if "publication_date" in df.columns:
        agg["family_earliest_publication_date"] = g["publication_date"].min()
    else:
        agg["family_earliest_publication_date"] = pd.NaT
    if "earliest_priority_date" in df.columns:
        agg["family_earliest_priority_date"] = g["earliest_priority_date"].min()

    def union_cpc(series):
        vals = set()
        for v in series.dropna():
            vals.update(parse_cpc_codes(v))
        return " ; ".join(sorted(vals))

    def union_multivalue(series):
        vals = set()
        for v in series.dropna():
            vals.update(split_multi(v))
        return " ; ".join(sorted(vals))

    agg["family_cpc_classifications"] = g["cpc_classifications"].apply(union_cpc)
    if "simple_family_member_jurisdictions" in df.columns:
        agg["family_jurisdictions"] = g["simple_family_member_jurisdictions"].apply(union_multivalue)
    else:
        agg["family_jurisdictions"] = ""

    rep = rep.set_index("family_id")
    fam = rep.join(agg, how="left").reset_index()

    fam["cited_by_patent_count"] = fam["family_cited_by_max"].fillna(fam["cited_by_patent_count"]).fillna(0)
    fam["simple_family_size"] = fam["family_simple_size_max"].fillna(fam["simple_family_size"]).fillna(1)
    fam["cpc_classifications"] = np.where(
        fam["family_cpc_classifications"].fillna("").astype(str).str.len() > 0,
        fam["family_cpc_classifications"],
        fam["cpc_classifications"].fillna("")
    )

    fam["title"] = fam["title"].fillna("").astype(str).str.strip()
    fam["abstract"] = fam["abstract"].fillna("").astype(str).str.strip()
    fam["document_text"] = (fam["title"] + ". " + fam["abstract"]).str.strip(" .")

    claims_col = next((c for c in ["claims", "claim", "claims_text"] if c in fam.columns), None)
    if FUNCTION_TEXT_USES_CLAIMS_IF_PRESENT and claims_col:
        fam["function_text"] = (
            fam["title"].fillna("").astype(str) + ". " +
            fam["abstract"].fillna("").astype(str) + ". " +
            fam[claims_col].fillna("").astype(str)
        ).str.strip(" .")
        function_source = f"title + abstract + {claims_col}"
    else:
        fam["function_text"] = fam["document_text"]
        function_source = "title + abstract"

    before_filter = len(fam)
    fam = fam[fam["document_text"].str.len() >= 20].copy().reset_index(drop=True)
    fam.insert(0, "doc_idx", np.arange(len(fam), dtype=np.int64))

    fmap = df[["source_row_id", "lens_id", "family_id"]].copy()
    fmap.to_parquet(FAMILY_MAP_PARQUET, index=False)
    fam.to_parquet(FAMILY_PARQUET, index=False)

    reduction = 1 - len(fam) / max(len(df), 1)
    summary = pd.DataFrame([{
        "raw_rows": len(df),
        "simple_family_representatives_before_text_filter": before_filter,
        "analysis_documents_after_text_filter": len(fam),
        "dedup_reduction_pct": reduction * 100,
        "function_text_source": function_source,
    }])
    save_table(summary, "02_family_dedup_summary", print_full=True)

    body = f"""
The semantic unit of analysis was collapsed to **simple patent families**. From **{len(df):,} raw rows**,
**{len(fam):,} family representatives** with usable title/abstract text remained, a reduction of
**{reduction*100:.1f}%**.

The representative family record is the row with the richest available title/abstract text. Forward citations
use the family maximum, simple-family size uses the family maximum, CPC codes are unioned across family members,
and function extraction uses **{function_source}**.
"""
    R.update_section("02_family", "Family-level deduplication and text preparation", body)
    R.emit(body)
    mark_stage_done(stage, [FAMILY_PARQUET, FAMILY_MAP_PARQUET])

def stage_02b_descriptives():
    stage = "02b_descriptives"
    stage_banner(stage)
    marker = FIG_DIR / "02b_descriptives.done"
    if stage_done(stage, [marker]):
        R.emit("Checkpoint found — descriptive outputs skipped.")
        return

    df = pd.read_parquet(FAMILY_PARQUET)
    years = pd.to_datetime(df["family_earliest_publication_date"], errors="coerce").dt.year
    trend = years.dropna().astype(int).value_counts().sort_index().rename_axis("year").reset_index(name="patent_families")
    if len(trend):
        fig, ax = plt.subplots(figsize=(10, 5.5))
        ax.plot(trend["year"], trend["patent_families"], marker="o", linewidth=2)
        ax.set_title("Patent-family publication trend")
        ax.set_xlabel("Publication year")
        ax.set_ylabel("Patent families")
        ax.grid(alpha=.25)
        peak = trend.loc[trend["patent_families"].idxmax()]
        summary = (
            f"Publication trend contains {len(trend)} observed years. Peak year: {int(peak['year'])} "
            f"with {int(peak['patent_families']):,} patent families. "
            f"Observed range: {int(trend['year'].min())}–{int(trend['year'].max())}."
        )
        save_figure(fig, "02_publication_trend", summary, trend)

    if "jurisdiction" in df.columns:
        jur = df["jurisdiction"].fillna("Unknown").astype(str).value_counts().head(25).rename_axis("jurisdiction").reset_index(name="count")
        fig, ax = plt.subplots(figsize=(10, 7))
        ax.barh(jur["jurisdiction"][::-1], jur["count"][::-1])
        ax.set_title("Top jurisdictions")
        ax.set_xlabel("Patent families")
        save_figure(fig, "02_top_jurisdictions", "Top jurisdictions:\n" + jur.to_string(index=False), jur)

    if "legal_status" in df.columns:
        legal = df["legal_status"].fillna("Unknown").astype(str).value_counts().head(20).rename_axis("legal_status").reset_index(name="count")
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.barh(legal["legal_status"][::-1], legal["count"][::-1])
        ax.set_title("Legal-status distribution")
        ax.set_xlabel("Patent families")
        save_figure(fig, "02_legal_status", "Legal-status distribution:\n" + legal.to_string(index=False), legal)

    marker.write_text("done", encoding="utf-8")
    mark_stage_done(stage, [marker])


# %% CELL: STAGES_03_06
# ============================================================
# EMBEDDINGS, UMAP, HDBSCAN TUNING, BASE BERTopic
# ============================================================

PATENT_EMB = EMBED_DIR / "patent_embeddings.npy"
UMAP5_PATH = EMBED_DIR / "umap_5d.npy"
UMAP2_PATH = EMBED_DIR / "umap_2d.npy"
BASE_ASSIGN = DATA_DIR / "06_base_cluster_assignments.parquet"
BASE_TOPIC_INFO = TABLE_DIR / "06_base_topic_info.csv"

def stage_03_patent_embeddings():
    stage = "03_patent_embeddings"
    stage_banner(stage)
    if stage_done(stage, [PATENT_EMB]):
        R.emit("Checkpoint found — patent embeddings skipped.")
        return

    df = pd.read_parquet(FAMILY_PARQUET)
    path = encode_texts_sharded(df["document_text"].tolist(), "patent_embeddings")
    if path != PATENT_EMB:
        shutil.copy2(path, PATENT_EMB)

    emb = np.load(PATENT_EMB, mmap_mode="r")
    norms = np.linalg.norm(np.asarray(emb[:min(len(emb), 5000)]), axis=1)
    stats = pd.DataFrame([{
        "documents": len(df),
        "embedding_dimension": emb.shape[1],
        "sample_mean_l2_norm": norms.mean(),
        "sample_min_l2_norm": norms.min(),
        "sample_max_l2_norm": norms.max(),
        "model": EMBEDDING_MODEL,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
    }])
    save_table(stats, "03_embedding_summary", print_full=True)

    body = f"""
Patent title–abstract texts were embedded with **{EMBEDDING_MODEL}**. The matrix contains
**{emb.shape[0]:,} × {emb.shape[1]:,}** L2-normalized values. Embedding inference ran on
**{'GPU (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'CPU'}**.
"""
    R.update_section("03_embeddings", "Patent semantic embeddings", body)
    R.emit(body)
    mark_stage_done(stage, [PATENT_EMB])

def stage_04_umap():
    stage = "04_umap"
    stage_banner(stage)
    if stage_done(stage, [UMAP5_PATH, UMAP2_PATH]):
        R.emit("Checkpoint found — UMAP skipped.")
        return

    from umap import UMAP
    emb = np.asarray(np.load(PATENT_EMB, mmap_mode="r"), dtype=np.float32)

    umap5 = UMAP(
        n_neighbors=UMAP_N_NEIGHBORS, n_components=UMAP_N_COMPONENTS,
        min_dist=UMAP_MIN_DIST, metric="cosine",
        random_state=RANDOM_SEED, low_memory=True
    )
    red5 = umap5.fit_transform(emb).astype("float32")
    np.save(UMAP5_PATH, red5)
    joblib.dump(umap5, CHECKPOINT_DIR / "umap_5d.joblib")

    umap2 = UMAP(
        n_neighbors=UMAP_N_NEIGHBORS, n_components=2,
        min_dist=0.05, metric="cosine",
        random_state=RANDOM_SEED, low_memory=True
    )
    red2 = umap2.fit_transform(emb).astype("float32")
    np.save(UMAP2_PATH, red2)
    joblib.dump(umap2, CHECKPOINT_DIR / "umap_2d.joblib")

    save_table(pd.DataFrame([{
        "n_neighbors": UMAP_N_NEIGHBORS,
        "components_for_clustering": UMAP_N_COMPONENTS,
        "min_dist": UMAP_MIN_DIST,
        "metric": "cosine",
        "random_seed": RANDOM_SEED,
    }]), "04_umap_parameters", print_full=True)
    mark_stage_done(stage, [UMAP5_PATH, UMAP2_PATH])

def dynamic_hdbscan_grid(n):
    vals = [15, 25, 50, 75, 100]
    vals += [int(round(n*p)) for p in [0.0025, 0.005, 0.01, 0.02, 0.03]]
    upper = max(5, n // 5)
    return sorted(set(max(5, min(v, upper)) for v in vals if v < n))

def stage_05_hdbscan_tuning():
    stage = "05_hdbscan_tuning"
    stage_banner(stage)
    out = TABLE_DIR / "05_hdbscan_tuning.csv"
    choice = CHECKPOINT_DIR / "hdbscan_choice.json"
    if stage_done(stage, [out, choice]):
        R.emit("Checkpoint found — HDBSCAN tuning skipped.")
        return

    import hdbscan
    from sklearn.preprocessing import minmax_scale

    X = np.load(UMAP5_PATH, mmap_mode="r")
    grid = HDBSCAN_MIN_CLUSTER_SIZE_GRID or dynamic_hdbscan_grid(len(X))
    rows = []

    for mcs in grid:
        R.emit(f"Testing min_cluster_size={mcs}")
        model = hdbscan.HDBSCAN(
            min_cluster_size=int(mcs),
            min_samples=None,
            metric="euclidean",
            cluster_selection_method="eom",
            prediction_data=True,
            gen_min_span_tree=True,
        )
        labels = model.fit_predict(X)
        valid = labels >= 0
        n_clusters = len(set(labels[valid]))
        noise_rate = 1 - valid.mean()
        counts = Counter(labels[valid])
        pers = np.asarray(getattr(model, "cluster_persistence_", []), dtype=float)
        if len(pers) and n_clusters:
            weights = np.array([counts.get(i, 0) for i in range(len(pers))], dtype=float)
            weighted_persistence = np.average(pers, weights=np.maximum(weights, 1))
        else:
            weighted_persistence = np.nan
        try:
            relative_validity = float(model.relative_validity_)
        except Exception:
            relative_validity = np.nan
        rows.append({
            "min_cluster_size": mcs,
            "n_clusters": n_clusters,
            "noise_rate": noise_rate,
            "weighted_cluster_persistence": weighted_persistence,
            "relative_validity": relative_validity,
        })

    res = pd.DataFrame(rows)
    for c in ["weighted_cluster_persistence", "relative_validity"]:
        vals = res[c].fillna(res[c].min() if res[c].notna().any() else 0)
        res[f"{c}_scaled"] = minmax_scale(vals) if len(res) > 1 else 1.0
    res["non_noise_scaled"] = 1 - res["noise_rate"]
    res["selection_score"] = (
        0.45 * res["weighted_cluster_persistence_scaled"]
        + 0.45 * res["relative_validity_scaled"]
        + 0.10 * res["non_noise_scaled"]
    )
    chosen = res.sort_values("selection_score", ascending=False).iloc[0]
    save_table(res, "05_hdbscan_tuning", print_full=True)
    atomic_write_json(choice, chosen.to_dict())

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(res["min_cluster_size"], res["weighted_cluster_persistence_scaled"], marker="o", label="Persistence")
    ax.plot(res["min_cluster_size"], res["relative_validity_scaled"], marker="s", label="Relative validity")
    ax.plot(res["min_cluster_size"], res["non_noise_scaled"], marker="^", label="Non-noise share")
    ax.axvline(chosen["min_cluster_size"], linestyle="--", alpha=.7, label=f"Selected={int(chosen['min_cluster_size'])}")
    ax.set_xlabel("HDBSCAN min_cluster_size")
    ax.set_ylabel("Scaled diagnostic")
    ax.set_title("HDBSCAN parameter tuning")
    ax.legend()
    ax.grid(alpha=.25)
    text = (
        f"Selected min_cluster_size={int(chosen['min_cluster_size'])}; "
        f"n_clusters={int(chosen['n_clusters'])}; noise_rate={chosen['noise_rate']:.3f}; "
        f"weighted persistence={chosen['weighted_cluster_persistence']:.4f}; "
        f"relative validity={chosen['relative_validity']:.4f}; "
        f"selection_score={chosen['selection_score']:.4f}."
    )
    save_figure(fig, "05_hdbscan_tuning", text, res)

    body = f"""
HDBSCAN `min_cluster_size` was tuned from the observed data rather than copied from the source case study.
The selected value was **{int(chosen['min_cluster_size'])}**, based jointly on cluster persistence,
relative validity, and a smaller non-noise-retention term. It produced **{int(chosen['n_clusters'])}**
preliminary clusters with **{chosen['noise_rate']:.1%}** noise.
"""
    R.update_section("05_hdbscan", "HDBSCAN tuning", body)
    R.emit(body)
    mark_stage_done(stage, [out, choice])

def stage_06_base_bertopic():
    stage = "06_base_bertopic"
    stage_banner(stage)
    if stage_done(stage, [BASE_ASSIGN, BASE_TOPIC_INFO]):
        R.emit("Checkpoint found — base BERTopic skipped.")
        return

    from bertopic import BERTopic
    from umap import UMAP
    import hdbscan
    from sklearn.feature_extraction.text import CountVectorizer

    df = pd.read_parquet(FAMILY_PARQUET)
    docs = df["document_text"].tolist()
    emb = np.asarray(np.load(PATENT_EMB, mmap_mode="r"), dtype=np.float32)
    chosen = json.loads((CHECKPOINT_DIR / "hdbscan_choice.json").read_text())
    mcs = int(chosen["min_cluster_size"])

    umap_model = UMAP(
        n_neighbors=UMAP_N_NEIGHBORS, n_components=UMAP_N_COMPONENTS,
        min_dist=UMAP_MIN_DIST, metric="cosine",
        random_state=RANDOM_SEED, low_memory=True
    )
    hdbscan_model = hdbscan.HDBSCAN(
        min_cluster_size=mcs, min_samples=None, metric="euclidean",
        cluster_selection_method="eom", prediction_data=True, gen_min_span_tree=True
    )
    vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1,2), min_df=2)

    topic_model = BERTopic(
        embedding_model=None,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        calculate_probabilities=False,
        verbose=True,
    )
    topics, _ = topic_model.fit_transform(docs, emb)
    assignments = df[["doc_idx", "family_id", "lens_id", "title"]].copy()
    assignments["base_topic"] = topics
    assignments.to_parquet(BASE_ASSIGN, index=False)

    info = topic_model.get_topic_info()
    save_table(info, "06_base_topic_info", print_full=True)

    model_dir = CHECKPOINT_DIR / "bertopic_base_model"
    try:
        topic_model.save(str(model_dir), serialization="safetensors", save_ctfidf=True)
        R.emit(f"BERTopic model saved: {model_dir}")
    except Exception as e:
        R.emit(f"[WARN] BERTopic safe serialization failed; assignments remain saved. {e}")

    xy = np.load(UMAP2_PATH)
    plot_df = assignments.copy()
    plot_df["x"] = xy[:,0]
    plot_df["y"] = xy[:,1]
    counts = plot_df["base_topic"].value_counts()
    major = set(counts[counts >= max(5, int(len(plot_df)*0.01))].index)

    fig, ax = plt.subplots(figsize=(11, 8))
    for topic, grp in plot_df.groupby("base_topic"):
        alpha = .55 if topic in major and topic != -1 else .12
        label = f"Topic {topic}" if topic in major else None
        ax.scatter(grp["x"], grp["y"], s=12, alpha=alpha, label=label)
    ax.set_title("Base semantic patent clusters (2D UMAP)")
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    if len(major) <= 15:
        ax.legend(fontsize=8, ncol=2)

    n_noise = int((assignments["base_topic"] == -1).sum())
    n_clusters = assignments.loc[assignments["base_topic"] >= 0, "base_topic"].nunique()
    text = (
        f"Base semantic map: {len(assignments):,} documents, {n_clusters} non-noise clusters, "
        f"{n_noise:,} outliers ({n_noise/len(assignments):.1%}). "
        f"Clustering used {UMAP_N_COMPONENTS} dimensions; this figure uses a separate 2D projection."
    )
    save_figure(fig, "06_base_semantic_map", text, plot_df[["family_id","base_topic","x","y"]])

    body = f"""
The PatentSBERTa representations were organized with BERTopic using the tuned HDBSCAN model.
The base solution contained **{n_clusters} non-noise clusters** and **{n_noise:,} outliers
({n_noise/len(assignments):.1%})**. These base clusters are used to derive representative CPC reference fields.
"""
    R.update_section("06_base_clusters", "Base semantic clustering", body)
    R.emit(body)
    mark_stage_done(stage, [BASE_ASSIGN, BASE_TOPIC_INFO])


# %% CELL: STAGE_07
# ============================================================
# OFFICIAL CPC TITLES + REPRESENTATIVE REFERENCE FIELDS
# ============================================================

CPC_TITLES_PATH = DATA_DIR / "07_cpc_titles.parquet"
REF_FIELD_SCORES = DATA_DIR / "07_reference_field_scores.parquet"
REP_FIELDS_PATH = DATA_DIR / "07_representative_reference_fields.parquet"

def download_and_parse_cpc_titles():
    import requests
    from bs4 import BeautifulSoup

    bulk_url = "https://www.cooperativepatentclassification.org/cpcSchemeAndDefinitions/bulk"
    html = requests.get(bulk_url, timeout=60)
    html.raise_for_status()
    soup = BeautifulSoup(html.text, "html.parser")

    link = None
    for a in soup.find_all("a"):
        if "CPC Title List" in a.get_text(" ", strip=True):
            link = urljoin(bulk_url, a.get("href"))
            break
    if not link:
        raise RuntimeError("Could not locate official CPC Title List.")

    content = requests.get(link, timeout=120).content
    suffix = ".zip" if content[:2] == b"PK" else ".txt"
    raw_path = CACHE_DIR / f"cpc_title_list_download{suffix}"
    raw_path.write_bytes(content)

    files = []
    if suffix == ".zip":
        ext_dir = CACHE_DIR / "cpc_title_list_extracted"
        ext_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(raw_path) as zf:
            zf.extractall(ext_dir)
        files = [p for p in ext_dir.rglob("*") if p.is_file()]
    else:
        files = [raw_path]

    records = []
    for p in files:
        try:
            text = p.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            continue
        for line in text.splitlines():
            line_clean = re.sub(r"<[^>]+>", " ", line)
            line_clean = re.sub(r"\s+", " ", line_clean).strip()
            m = re.match(r"^([A-HY]\d{2}[A-Z]?(?:\d+(?:/\d+)?)?)\s+(.+)$", line_clean)
            if m:
                symbol = re.sub(r"\s+", "", m.group(1).upper())
                title = m.group(2).strip(" \t;|")
                if len(title) > 1:
                    records.append((symbol, title))

    if not records:
        raise RuntimeError("Official CPC Title List downloaded but parsing returned no records.")

    cpc = pd.DataFrame(records, columns=["cpc_symbol","cpc_title"]).drop_duplicates("cpc_symbol")
    cpc["source_url"] = link
    cpc.to_parquet(CPC_TITLES_PATH, index=False)
    atomic_write_text(CHECKPOINT_DIR / "cpc_title_source_url.txt", link)
    return cpc

def stage_07_reference_fields():
    stage = "07_reference_fields"
    stage_banner(stage)
    if stage_done(stage, [CPC_TITLES_PATH, REF_FIELD_SCORES, REP_FIELDS_PATH]):
        R.emit("Checkpoint found — reference-field extraction skipped.")
        return

    df = pd.read_parquet(FAMILY_PARQUET)
    assign = pd.read_parquet(BASE_ASSIGN)
    emb = np.asarray(np.load(PATENT_EMB, mmap_mode="r"), dtype=np.float32)

    cpc_titles = pd.read_parquet(CPC_TITLES_PATH) if CPC_TITLES_PATH.exists() else download_and_parse_cpc_titles()
    class_titles = cpc_titles[cpc_titles["cpc_symbol"].str.match(r"^[A-HY]\d{2}$", na=False)].copy()
    title_map = dict(zip(class_titles["cpc_symbol"], class_titles["cpc_title"]))

    work = df[["doc_idx","family_id","cpc_classifications"]].merge(
        assign[["doc_idx","base_topic"]], on="doc_idx", how="left"
    )
    rows = []
    for _, r in work.iterrows():
        if r["base_topic"] < 0:
            continue
        classes = sorted(set(filter(None, (cpc_class(x) for x in parse_cpc_codes(r["cpc_classifications"])))))
        for cc in classes:
            rows.append((int(r["doc_idx"]), r["family_id"], int(r["base_topic"]), cc))
    long = pd.DataFrame(rows, columns=["doc_idx","family_id","base_topic","cpc_class"]).drop_duplicates()
    save_table(long, "07_doc_cpc_class_long", print_full=True)

    candidate_classes = sorted(long["cpc_class"].unique())
    cpc_texts = [f"{cc}: {title_map.get(cc, 'CPC class '+cc)}" for cc in candidate_classes]
    cpc_emb_path = encode_texts_sharded(cpc_texts, "cpc_class_title_embeddings", shard_size=1000)
    cpc_emb = np.asarray(np.load(cpc_emb_path), dtype=np.float32)
    cpc_vec = {cc: cpc_emb[i] for i,cc in enumerate(candidate_classes)}

    score_rows = []
    for topic, grp in long.groupby("base_topic"):
        doc_ids = grp["doc_idx"].unique().astype(int)
        centroid = l2_normalize(emb[doc_ids].mean(axis=0, keepdims=True))[0]
        counts = grp.groupby("cpc_class")["family_id"].nunique().sort_values(ascending=False)
        tmp = []
        for cc, cnt in counts.items():
            tmp.append({
                "base_topic": int(topic),
                "cpc_class": cc,
                "count": int(cnt),
                "semantic_similarity": float(np.dot(cpc_vec[cc], centroid)),
            })
        t = pd.DataFrame(tmp)
        t["similarity_z"] = safe_z(t["semantic_similarity"]).values
        t["reference_field_score"] = np.log1p(t["count"]) * t["similarity_z"]
        t["cpc_title"] = t["cpc_class"].map(title_map).fillna(t["cpc_class"].map(lambda x: f"CPC class {x}"))
        score_rows.append(t)

    scores = pd.concat(score_rows, ignore_index=True).sort_values(
        ["base_topic","reference_field_score"], ascending=[True,False]
    )
    scores.to_parquet(REF_FIELD_SCORES, index=False)
    save_table(scores, "07_reference_field_scores", print_full=True)

    reps = scores.groupby("base_topic", as_index=False).first()
    reps["cluster_rank"] = reps["reference_field_score"].rank(ascending=False, method="dense").astype(int)
    reps.to_parquet(REP_FIELDS_PATH, index=False)
    save_table(reps, "07_representative_reference_fields_by_cluster", print_full=True)

    unique_fields = reps.sort_values("reference_field_score", ascending=False).drop_duplicates("cpc_class")
    if N_REP_REFERENCE_FIELDS:
        unique_fields = unique_fields.head(N_REP_REFERENCE_FIELDS)
    save_table(unique_fields, "07_unique_reference_fields_for_zeroshot", print_full=True)

    body = f"""
Each preliminary semantic cluster was characterized by a CPC **class-level** reference field. The score combines
within-cluster family frequency with semantic similarity between the official CPC title and the cluster centroid:
`log(1 + count) × z(cosine similarity)`. The procedure yielded **{len(reps)} cluster-level fields** and
**{unique_fields['cpc_class'].nunique()} unique reference fields** for zero-shot label construction.
"""
    R.update_section("07_reference_fields", "Representative reference fields", body)
    R.emit(body)
    mark_stage_done(stage, [CPC_TITLES_PATH, REF_FIELD_SCORES, REP_FIELDS_PATH])


# %% CELL: STAGES_08_09
# ============================================================
# MECHANISM-AWARE FUNCTION EXTRACTION + SEMANTIC NORMALIZATION
# ============================================================

RAW_FUNCTIONS = DATA_DIR / "08_raw_mechanism_functions.parquet"
FUNCTION_MAP = DATA_DIR / "09_function_normalization_map.parquet"
FUNCTION_SCORES = DATA_DIR / "09_representative_function_scores.parquet"
REP_FUNCTIONS = DATA_DIR / "09_representative_functions.parquet"

GENERIC_VERBS = {
    "be","have","include","comprise","contain","consist","provide","disclose",
    "describe","relate","present","show","indicate","use","form","make"
}
MECH_MARKERS = {"by","via","through","with","upon","under","using"}

def clean_phrase(s, max_chars=160):
    s = re.sub(r"\s+", " ", str(s or "")).strip(" ,;:.()[]")
    return s[:max_chars].strip()

def extract_np(token):
    toks = []
    for t in token.subtree:
        if t.is_punct or t.pos_ == "DET":
            continue
        toks.append(t.lemma_.lower() if t.lemma_ else t.text.lower())
    return clean_phrase(" ".join(toks[:14]), 120)

def extract_mechanism_from_sentence(sent_text, verb):
    for child in verb.children:
        low = child.text.lower()
        if (child.dep_ == "prep" and low in MECH_MARKERS) or low == "using":
            phrase = clean_phrase(" ".join(t.text for t in child.subtree), 140)
            if len(phrase) >= 4:
                return phrase
    m = re.search(
        r"\b(by|via|through|using|with|upon|under|in response to)\b\s+([^.;]{3,120})",
        sent_text, flags=re.I
    )
    if m:
        return clean_phrase(m.group(1).lower() + " " + m.group(2), 140)
    return ""

def extract_functions_doc(doc, family_id, doc_idx):
    out = []
    for sent in doc.sents:
        sent_text = clean_phrase(sent.text, 500)
        if len(sent_text) < 20:
            continue
        for tok in sent:
            if tok.pos_ != "VERB":
                continue
            action = (tok.lemma_ or tok.text).lower().strip()
            if action in GENERIC_VERBS or len(action) < 2:
                continue

            obj = next(
                (ch for ch in tok.children if ch.dep_ in {"dobj","obj","attr","oprd","dative","nsubjpass","nsubj:pass"}),
                None
            )
            if obj is None:
                for ch in tok.children:
                    if ch.dep_ == "prep":
                        pobj = next((x for x in ch.children if x.dep_ == "pobj"), None)
                        if pobj is not None:
                            obj = pobj
                            break
            if obj is None:
                continue

            obj_phrase = extract_np(obj)
            if not obj_phrase or obj_phrase in {"method","system","device","apparatus"}:
                continue

            subject = ""
            for ch in tok.children:
                if ch.dep_ in {"nsubj","nsubjpass","nsubj:pass"}:
                    subject = extract_np(ch)
                    break

            mechanism = extract_mechanism_from_sentence(sent_text, tok)
            action_object = clean_phrase(f"{action} {obj_phrase}", 160)
            function_phrase = clean_phrase(
                f"{action_object} via {mechanism}" if mechanism else action_object,
                260
            )
            out.append({
                "doc_idx": int(doc_idx),
                "family_id": family_id,
                "sentence": sent_text,
                "subject": subject,
                "action": action,
                "object": obj_phrase,
                "mechanism": mechanism,
                "action_object": action_object,
                "function_phrase": function_phrase,
            })
    return out

def stage_08_extract_mechanism_functions():
    stage = "08_mechanism_function_extraction"
    stage_banner(stage)
    if stage_done(stage, [RAW_FUNCTIONS]):
        R.emit("Checkpoint found — mechanism function extraction skipped.")
        return

    import spacy
    df = pd.read_parquet(FAMILY_PARQUET)
    nlp = spacy.load("en_core_web_sm", disable=["ner"])
    nlp.max_length = max(2_000_000, int(df["function_text"].str.len().max() + 1000))

    shard_dir = CHECKPOINT_DIR / "function_extraction_shards"
    shard_dir.mkdir(parents=True, exist_ok=True)

    for start in range(0, len(df), FUNCTION_EXTRACTION_SHARD_SIZE):
        end = min(start + FUNCTION_EXTRACTION_SHARD_SIZE, len(df))
        path = shard_dir / f"functions_{start:09d}_{end:09d}.parquet"
        if path.exists():
            R.emit(f"[function shard cached] {path.name}")
            continue

        sub = df.iloc[start:end]
        texts = sub["function_text"].fillna("").astype(str).tolist()
        rows = []
        R.emit(f"spaCy parsing documents {start:,}..{end-1:,}")
        docs = nlp.pipe(texts, batch_size=128, n_process=2)
        for (_, r), doc in zip(sub.iterrows(), docs):
            rows.extend(extract_functions_doc(doc, r["family_id"], int(r["doc_idx"])))
        shard_df = pd.DataFrame(rows)
        tmp = path.with_suffix(".tmp.parquet")
        shard_df.to_parquet(tmp, index=False)
        os.replace(tmp, path)
        R.emit(f"Saved {len(shard_df):,} raw function instances -> {path.name}")

    shards = sorted(shard_dir.glob("functions_*.parquet"))
    allf = pd.concat([pd.read_parquet(p) for p in shards], ignore_index=True) if shards else pd.DataFrame()
    if len(allf) == 0:
        raise RuntimeError("No mechanism-aware function instances were extracted.")
    allf.to_parquet(RAW_FUNCTIONS, index=False)

    mech_rate = (allf["mechanism"].fillna("").str.len() > 0).mean()
    save_table(allf["action"].value_counts().head(30).rename_axis("action").reset_index(name="count"), "08_top_actions", True)
    save_table(allf["object"].value_counts().head(30).rename_axis("object").reset_index(name="count"), "08_top_objects", True)
    save_table(allf, "08_raw_mechanism_functions", print_full=PRINT_FULL_RAW_EXTRACTIONS)

    body = f"""
Mechanism-aware dependency parsing extracted **{len(allf):,} raw Action–Object instances** from
**{allf['family_id'].nunique():,} patent families**. A Mechanism/Condition phrase was detected in
**{mech_rate:.1%}** of instances. The adapted representation preserves
**Action + Object + Mechanism/Condition** whenever the mechanism is recoverable from the patent sentence.
"""
    R.update_section("08_functions_raw", "Mechanism-aware function extraction", body)
    R.emit(body)
    mark_stage_done(stage, [RAW_FUNCTIONS])

def greedy_semantic_normalization(phrases, vectors, frequencies, threshold):
    """
    Greedy semantic normalization without FAISS.

    Uses PyTorch/CUDA when available and NumPy otherwise.
    Phrases are processed from highest to lowest frequency.
    """
    order = np.argsort(-np.asarray(frequencies))
    vectors = l2_normalize(vectors).astype("float32")

    prototype_indices = []
    mapping = {}
    use_cuda = torch.cuda.is_available()

    if use_cuda:
        device = torch.device("cuda")
        vec_t = torch.from_numpy(vectors).to(device)
        proto_t = torch.empty(
            (len(order), vectors.shape[1]),
            dtype=torch.float32,
            device=device
        )
    else:
        vec_t = None
        proto_np = np.empty(
            (len(order), vectors.shape[1]),
            dtype=np.float32
        )

    n_proto = 0

    for idx in tqdm(order, desc="Normalizing function phrases"):
        idx = int(idx)

        if n_proto == 0:
            prototype_indices.append(idx)
            mapping[idx] = idx
            if use_cuda:
                proto_t[0] = vec_t[idx]
            else:
                proto_np[0] = vectors[idx]
            n_proto = 1
            continue

        if use_cuda:
            sims = torch.mv(proto_t[:n_proto], vec_t[idx])
            best_sim_t, best_pos_t = torch.max(sims, dim=0)
            best_sim = float(best_sim_t.item())
            best_pos = int(best_pos_t.item())
        else:
            sims = proto_np[:n_proto] @ vectors[idx]
            best_pos = int(np.argmax(sims))
            best_sim = float(sims[best_pos])

        if best_sim >= threshold:
            mapping[idx] = int(prototype_indices[best_pos])
        else:
            prototype_indices.append(idx)
            mapping[idx] = idx
            if use_cuda:
                proto_t[n_proto] = vec_t[idx]
            else:
                proto_np[n_proto] = vectors[idx]
            n_proto += 1

    if use_cuda:
        del vec_t, proto_t
        torch.cuda.empty_cache()

    return mapping, prototype_indices

def stage_09_normalize_score_functions():
    stage = "09_normalize_score_functions"
    stage_banner(stage)
    if stage_done(stage, [FUNCTION_MAP, FUNCTION_SCORES, REP_FUNCTIONS]):
        R.emit("Checkpoint found — function normalization/scoring skipped.")
        return

    raw = pd.read_parquet(RAW_FUNCTIONS)
    freq = (
        raw.groupby("function_phrase")
           .agg(frequency=("family_id","size"), patent_families=("family_id","nunique"))
           .reset_index()
    )
    freq = freq[freq["frequency"] >= FUNCTION_MIN_FREQUENCY].sort_values(
        ["frequency","patent_families"], ascending=False
    )
    if len(freq) > MAX_FUNCTION_PHRASES_FOR_NORMALIZATION:
        R.emit(
            f"Candidate normalization set limited from {len(freq):,} to "
            f"{MAX_FUNCTION_PHRASES_FOR_NORMALIZATION:,}; raw extraction remains complete."
        )
        freq = freq.head(MAX_FUNCTION_PHRASES_FOR_NORMALIZATION).copy()

    phrases = freq["function_phrase"].tolist()
    femb_path = encode_texts_sharded(phrases, "function_phrase_embeddings", shard_size=3000)
    femb = np.asarray(np.load(femb_path), dtype=np.float32)
    mapping, _ = greedy_semantic_normalization(
        phrases, femb, freq["frequency"].values, FUNCTION_NORMALIZATION_SIM_THRESHOLD
    )

    map_rows = []
    for idx, proto in mapping.items():
        map_rows.append({
            "function_phrase": phrases[idx],
            "prototype_phrase": phrases[proto],
            "similarity_to_prototype": float(np.dot(femb[idx], femb[proto])),
            "raw_frequency": int(freq.iloc[idx]["frequency"]),
            "raw_patent_families": int(freq.iloc[idx]["patent_families"]),
        })
    fmap = pd.DataFrame(map_rows)
    fmap.to_parquet(FUNCTION_MAP, index=False)
    save_table(fmap, "09_function_normalization_map", print_full=True)

    prot = (
        fmap.groupby("prototype_phrase")
            .agg(
                normalized_frequency=("raw_frequency","sum"),
                normalized_patent_families=("raw_patent_families","sum"),
                merged_variants=("function_phrase","nunique")
            ).reset_index()
    )

    model = get_embedder()
    target_vec = model.encode([TARGET_TECHNOLOGY], convert_to_numpy=True, normalize_embeddings=True)[0].astype("float32")
    proto_vecs = model.encode(
        prot["prototype_phrase"].tolist(),
        batch_size=EMBED_BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    ).astype("float32")

    prot["target_similarity"] = proto_vecs @ target_vec
    prot["target_similarity_z"] = safe_z(prot["target_similarity"]).values
    prot["function_score"] = np.log1p(prot["normalized_frequency"]) * prot["target_similarity_z"]
    prot = prot.sort_values("function_score", ascending=False).reset_index(drop=True)
    prot.to_parquet(FUNCTION_SCORES, index=False)
    save_table(prot, "09_representative_function_scores", print_full=True)

    reps = prot.head(N_REP_FUNCTIONS).copy()
    reps.insert(0, "function_rank", np.arange(1, len(reps)+1))
    reps.to_parquet(REP_FUNCTIONS, index=False)
    save_table(reps, "09_selected_representative_functions", print_full=True)

    fig, ax = plt.subplots(figsize=(11, 6.5))
    plot = reps.sort_values("function_score")
    ax.barh(plot["prototype_phrase"], plot["function_score"])
    ax.set_title("Representative mechanism-aware functions")
    ax.set_xlabel("Function score")
    text = (
        f"{len(prot):,} normalized prototypes scored; top {len(reps)} selected. "
        f"Normalization cosine threshold={FUNCTION_NORMALIZATION_SIM_THRESHOLD:.2f}.\n"
        + reps[["function_rank","prototype_phrase","normalized_frequency","target_similarity","function_score"]].to_string(index=False)
    )
    save_figure(fig, "09_selected_functions", text, reps)

    body = f"""
Action–Object–Mechanism expressions were semantically normalized at cosine similarity
**≥ {FUNCTION_NORMALIZATION_SIM_THRESHOLD:.2f}**, prioritizing the more frequent expression as prototype.
Normalized functions were scored as `log(1 + frequency) × z(cosine similarity to target technology)`.
The top **{len(reps)} functions** were retained.
"""
    R.update_section("09_functions", "Function normalization and representative functions", body)
    R.emit(body)
    mark_stage_done(stage, [FUNCTION_MAP, FUNCTION_SCORES, REP_FUNCTIONS])


# %% CELL: STAGES_10_11
# ============================================================
# CIRCULARITY RELEVANCE + ZERO-SHOT FIELD×FUNCTION ASSIGNMENT
# ============================================================

CR_PATH = DATA_DIR / "10_patent_circularity_relevance.parquet"
ZS_LABELS = DATA_DIR / "11_zeroshot_labels.parquet"
ZS_ASSIGN = DATA_DIR / "11_zeroshot_assignments.parquet"
ZS_SIM_MATRIX = EMBED_DIR / "11_zeroshot_similarity_matrix.npy"

def stage_10_circularity_relevance():
    stage = "10_circularity_relevance"
    stage_banner(stage)
    if stage_done(stage, [CR_PATH]):
        R.emit("Checkpoint found — Circularity Relevance skipped.")
        return

    df = pd.read_parquet(FAMILY_PARQUET)
    emb = np.asarray(np.load(PATENT_EMB, mmap_mode="r"), dtype=np.float32)
    model = get_embedder()

    pos = model.encode(CIRCULARITY_ANCHORS, convert_to_numpy=True, normalize_embeddings=True)
    service = model.encode(SERVICE_ANCHORS, convert_to_numpy=True, normalize_embeddings=True)

    pos_sims = emb @ pos.T
    service_sims = emb @ service.T
    k = min(3, pos_sims.shape[1])
    circ = np.partition(pos_sims, -k, axis=1)[:, -k:].mean(axis=1)
    serv = service_sims.max(axis=1)
    cr = circ - serv

    out = df[["doc_idx","family_id","lens_id","title"]].copy()
    out["circularity_similarity"] = circ
    out["serviceability_similarity"] = serv
    out["circularity_relevance_raw"] = cr
    out["circularity_relevance_z"] = safe_z(cr).values
    out.to_parquet(CR_PATH, index=False)
    save_table(out.sort_values("circularity_relevance_raw", ascending=False), "10_patent_circularity_relevance", True)

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.scatter(out["serviceability_similarity"], out["circularity_similarity"], s=14, alpha=.35)
    lo = min(out["serviceability_similarity"].min(), out["circularity_similarity"].min())
    hi = max(out["serviceability_similarity"].max(), out["circularity_similarity"].max())
    ax.plot([lo,hi],[lo,hi], linestyle="--", alpha=.6)
    ax.set_xlabel("Service / repair semantic similarity")
    ax.set_ylabel("Circularity / recycling semantic similarity")
    ax.set_title("Circularity orientation of DfD/DfR patents")
    positive_share = (out["circularity_relevance_raw"] > 0).mean()
    text = (
        f"CR = mean(top-3 circularity-anchor similarities) - max(service/repair-anchor similarity). "
        f"{positive_share:.1%} of patent families have CR>0. "
        f"Mean={out['circularity_relevance_raw'].mean():.4f}; median={out['circularity_relevance_raw'].median():.4f}; "
        f"min={out['circularity_relevance_raw'].min():.4f}; max={out['circularity_relevance_raw'].max():.4f}."
    )
    save_figure(fig, "10_circularity_vs_serviceability", text, out[[
        "family_id","circularity_similarity","serviceability_similarity","circularity_relevance_raw"
    ]])

    body = f"""
A domain-specific **Circularity Relevance (CR)** signal distinguishes disassembly aligned with end-of-life
recovery from disassembly aligned mainly with repair/service. At patent level:
`CR = mean(top-3 circularity similarities) − max(service/repair similarity)`.
**{positive_share:.1%}** of patent families received CR > 0. CR remains independent of TV/TR/TN.
"""
    R.update_section("10_cr", "Circularity Relevance extension", body)
    R.emit(body)
    mark_stage_done(stage, [CR_PATH])

def stage_11_zeroshot_assignment():
    stage = "11_zeroshot_assignment"
    stage_banner(stage)
    if stage_done(stage, [ZS_LABELS, ZS_ASSIGN, ZS_SIM_MATRIX]):
        R.emit("Checkpoint found — zero-shot assignment skipped.")
        return

    df = pd.read_parquet(FAMILY_PARQUET)
    emb = np.asarray(np.load(PATENT_EMB, mmap_mode="r"), dtype=np.float32)
    reps_fields = pd.read_parquet(REP_FIELDS_PATH)
    unique_fields = reps_fields.sort_values("reference_field_score", ascending=False).drop_duplicates("cpc_class")
    if N_REP_REFERENCE_FIELDS:
        unique_fields = unique_fields.head(N_REP_REFERENCE_FIELDS)
    functions = pd.read_parquet(REP_FUNCTIONS)

    labels = []
    lid = 0
    for _, f in unique_fields.iterrows():
        for _, fn in functions.iterrows():
            labels.append({
                "label_id": lid,
                "cpc_class": f["cpc_class"],
                "reference_field": f["cpc_title"],
                "function_rank": int(fn["function_rank"]),
                "function": fn["prototype_phrase"],
                "label_text": f"{f['cpc_title']} that {fn['prototype_phrase']}",
            })
            lid += 1
    labels = pd.DataFrame(labels)
    labels.to_parquet(ZS_LABELS, index=False)
    save_table(labels, "11_zeroshot_labels", True)

    model = get_embedder()
    label_emb = model.encode(
        labels["label_text"].tolist(),
        batch_size=EMBED_BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    ).astype("float32")

    sims = (emb @ label_emb.T).astype("float32")
    np.save(ZS_SIM_MATRIX, sims)
    best_idx = sims.argmax(axis=1)
    best_sim = sims[np.arange(len(sims)), best_idx]

    tuning_rows = []
    for th in ZEROSHOT_THRESHOLD_GRID:
        mask = best_sim >= th
        counts = Counter(best_idx[mask])
        nonempty = len(counts)
        tuning_rows.append({
            "threshold": th,
            "assigned_documents": int(mask.sum()),
            "assignment_rate": float(mask.mean()),
            "nonempty_labels": nonempty,
            "label_coverage": nonempty / max(len(labels),1),
            "mean_assigned_similarity": float(best_sim[mask].mean()) if mask.any() else np.nan,
            "median_cluster_size": float(np.median(list(counts.values()))) if counts else 0,
            "min_cluster_size": int(min(counts.values())) if counts else 0,
        })
    tuning = pd.DataFrame(tuning_rows)
    tuning["selection_score"] = (
        -np.abs(tuning["assignment_rate"] - ZEROSHOT_TARGET_ASSIGNMENT_RATE)
        + 0.15*tuning["label_coverage"]
        + 0.05*tuning["mean_assigned_similarity"].fillna(0)
    )
    save_table(tuning, "11_zeroshot_threshold_tuning", True)

    if ZEROSHOT_THRESHOLD_OVERRIDE is not None:
        threshold = float(ZEROSHOT_THRESHOLD_OVERRIDE)
        selection_reason = "manual override"
    else:
        threshold = float(tuning.sort_values("selection_score", ascending=False).iloc[0]["threshold"])
        selection_reason = "automatic tuning"

    assigned = best_sim >= threshold
    out = df[["doc_idx","family_id","lens_id","title"]].copy()
    out["zeroshot_label_id"] = np.where(assigned, best_idx, -1)
    out["zeroshot_similarity"] = best_sim
    out = out.merge(
        labels[["label_id","label_text","cpc_class","reference_field","function"]],
        left_on="zeroshot_label_id", right_on="label_id", how="left"
    )
    out["assigned"] = assigned
    out.to_parquet(ZS_ASSIGN, index=False)
    save_table(out, "11_zeroshot_assignments", True)

    assigned_out = out[out["assigned"]].copy()
    heat = pd.crosstab(assigned_out["reference_field"], assigned_out["function"])
    heat.to_csv(TABLE_DIR / "11_field_function_heatmap_counts.csv")

    fig, ax = plt.subplots(figsize=(max(10, 1.3*heat.shape[1]), max(6, .65*heat.shape[0])))
    im = ax.imshow(heat.values, aspect="auto")
    ax.set_xticks(np.arange(heat.shape[1]))
    ax.set_xticklabels(heat.columns, rotation=55, ha="right", fontsize=8)
    ax.set_yticks(np.arange(heat.shape[0]))
    ax.set_yticklabels(heat.index, fontsize=8)
    ax.set_xlabel("Representative mechanism-aware function")
    ax.set_ylabel("Reference field")
    ax.set_title("Reference Field × Function patent-family counts")
    fig.colorbar(im, ax=ax, label="Patent families")

    cells = []
    for i, rr in enumerate(heat.index):
        for j, cc in enumerate(heat.columns):
            cells.append((int(heat.iloc[i,j]), rr, cc))
    cells.sort(reverse=True, key=lambda x: x[0])
    top_cells = "\n".join(f"{n:>6} | {rr} | {cc}" for n,rr,cc in cells[:20])
    summary = (
        f"Zero-shot threshold={threshold:.2f} ({selection_reason}); assigned={assigned.sum():,}/{len(out):,} "
        f"({assigned.mean():.1%}); nonempty labels={assigned_out['zeroshot_label_id'].nunique()}/{len(labels)}. "
        f"Heatmap dimensions={heat.shape[0]} fields × {heat.shape[1]} functions.\nTop cells:\n{top_cells}"
    )
    save_figure(fig, "11_field_function_heatmap", summary)

    body = f"""
Representative fields and the top **{len(functions)}** mechanism-aware functions generated
**{len(labels)} Reference Field × Function labels**. The selected cosine threshold was **{threshold:.2f}**
({selection_reason}), assigning **{assigned.sum():,}/{len(out):,} documents ({assigned.mean():.1%})** to
**{assigned_out['zeroshot_label_id'].nunique()} nonempty predefined clusters**. Under-threshold documents remain
saved for audit but are excluded from downstream opportunity scoring.
"""
    R.update_section("11_zeroshot", "Semi-supervised Reference Field × Function clustering", body)
    R.emit(body)
    mark_stage_done(stage, [ZS_LABELS, ZS_ASSIGN, ZS_SIM_MATRIX], extra={"threshold": threshold})


# %% CELL: STAGES_12_13
# ============================================================
# TN SENTENCE CORPUS + TV/TR/TN/CR CLUSTER EVALUATION
# ============================================================

FIELD_SENTENCES = DATA_DIR / "12_field_interest_sentences.parquet"
FIELD_SENT_EMB = EMBED_DIR / "field_sentence_embeddings.npy"
CLUSTER_METRICS = DATA_DIR / "13_cluster_metrics.parquet"

def stage_12_field_sentence_corpus():
    stage = "12_field_sentence_corpus"
    stage_banner(stage)
    if stage_done(stage, [FIELD_SENTENCES, FIELD_SENT_EMB]):
        R.emit("Checkpoint found — field sentence corpus skipped.")
        return

    df = pd.read_parquet(FAMILY_PARQUET)
    pattern = re.compile("|".join(re.escape(x) for x in FIELD_TERMS), flags=re.I)
    rows = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc="Field sentence extraction"):
        for sent in sentence_split_simple(r["document_text"]):
            if pattern.search(sent):
                rows.append({
                    "doc_idx": int(r["doc_idx"]),
                    "family_id": r["family_id"],
                    "sentence": sent,
                    "publication_date": r.get("family_earliest_publication_date", r.get("publication_date")),
                })
    sents = pd.DataFrame(rows).drop_duplicates(["family_id","sentence"]).reset_index(drop=True)
    if len(sents) == 0:
        raise RuntimeError("No field-of-interest sentences found; edit FIELD_TERMS.")
    sents.to_parquet(FIELD_SENTENCES, index=False)
    save_table(sents, "12_field_interest_sentences", True)

    path = encode_texts_sharded(sents["sentence"].tolist(), "field_sentence_embeddings", shard_size=5000)
    if path != FIELD_SENT_EMB:
        shutil.copy2(path, FIELD_SENT_EMB)

    body = f"""
For TN, **{len(sents):,} sentence-level field-of-interest observations** explicitly mentioning recycling/circularity
terms were extracted and embedded as the established target-field knowledge base.
"""
    R.update_section("12_tn_corpus", "Field-of-interest sentence corpus for novelty", body)
    R.emit(body)
    mark_stage_done(stage, [FIELD_SENTENCES, FIELD_SENT_EMB])

def stage_13_cluster_metrics():
    stage = "13_cluster_metrics"
    stage_banner(stage)
    if stage_done(stage, [CLUSTER_METRICS]):
        R.emit("Checkpoint found — cluster evaluation skipped.")
        return

    df = pd.read_parquet(FAMILY_PARQUET)
    zs = pd.read_parquet(ZS_ASSIGN)
    cr = pd.read_parquet(CR_PATH)
    emb = np.asarray(np.load(PATENT_EMB, mmap_mode="r"), dtype=np.float32)
    sents = pd.read_parquet(FIELD_SENTENCES)
    sent_emb = np.asarray(np.load(FIELD_SENT_EMB, mmap_mode="r"), dtype=np.float32)
    labels = pd.read_parquet(ZS_LABELS)

    work = (
        df.merge(zs[["doc_idx","zeroshot_label_id","zeroshot_similarity","assigned"]], on="doc_idx")
          .merge(cr[["doc_idx","circularity_relevance_raw"]], on="doc_idx")
    )
    work = work[work["assigned"]].copy()

    analysis_dt = pd.Timestamp(ANALYSIS_DATE)
    pub = pd.to_datetime(work["family_earliest_publication_date"], errors="coerce")
    work["citation_age_years"] = ((analysis_dt - pub).dt.days / 365.25).clip(lower=1.0).fillna(1.0)
    work["cited_by_patent_count"] = pd.to_numeric(work["cited_by_patent_count"], errors="coerce").fillna(0)
    work["simple_family_size"] = pd.to_numeric(work["simple_family_size"], errors="coerce").fillna(1).clip(lower=1)

    cluster_ids = sorted(work["zeroshot_label_id"].unique().astype(int))
    centroids = {}
    for cid in cluster_ids:
        ids = work.loc[work["zeroshot_label_id"]==cid, "doc_idx"].astype(int).values
        centroids[cid] = l2_normalize(emb[ids].mean(axis=0, keepdims=True))[0]

    model = get_embedder()
    target_vec = model.encode([TARGET_TECHNOLOGY], convert_to_numpy=True, normalize_embeddings=True)[0].astype("float32")

    rows = []
    for cid, grp in work.groupby("zeroshot_label_id"):
        cid = int(cid)
        P = len(grp)
        citation_rate = grp["cited_by_patent_count"] / grp["citation_age_years"]
        rows.append({
            "cluster_id": cid,
            "cluster_size": P,
            "citation_component_raw": float(np.median(citation_rate)) * (1 + math.log(max(P,1))),
            "family_component_raw": float(np.median(grp["simple_family_size"])) * (1 + math.log(max(P,1))),
            "TR_raw_similarity": float(np.dot(centroids[cid], target_vec)),
            "CR_raw_mean": float(grp["circularity_relevance_raw"].mean()),
        })
    metrics = pd.DataFrame(rows)
    metrics["citation_component_z"] = safe_z(metrics["citation_component_raw"]).values
    metrics["family_component_z"] = safe_z(metrics["family_component_raw"]).values
    metrics["TV"] = (metrics["citation_component_z"] + metrics["family_component_z"]) / 2
    metrics["TR"] = safe_z(metrics["TR_raw_similarity"]).values
    metrics["CR"] = safe_z(metrics["CR_raw_mean"]).values

    C = np.stack([centroids[cid] for cid in cluster_ids]).astype("float32")
    sim_matrix = C @ sent_emb.T
    np.save(EMBED_DIR / "13_tn_cluster_sentence_similarity.npy", sim_matrix.astype("float32"))
    q3 = float(np.quantile(sim_matrix, 0.75))

    sent_pub = pd.to_datetime(sents["publication_date"], errors="coerce")
    sent_age = ((analysis_dt - sent_pub).dt.days / 365.25).clip(lower=1.0).fillna(1.0).values
    weights = 1.0 / sent_age

    tn_rows = []
    for i, cid in enumerate(cluster_ids):
        mask = sim_matrix[i] >= q3
        tn_rows.append({
            "cluster_id": cid,
            "TN_overlap_raw": float(weights[mask].sum()),
            "TN_matching_sentences": int(mask.sum())
        })
    tn = pd.DataFrame(tn_rows)
    tn["TN"] = -safe_z(tn["TN_overlap_raw"]).values

    metrics = metrics.merge(tn, on="cluster_id", how="left")
    metrics = metrics.merge(
        labels[["label_id","label_text","cpc_class","reference_field","function"]],
        left_on="cluster_id", right_on="label_id", how="left"
    )
    metrics["key_cluster"] = (metrics["TV"] > 0) & (metrics["TR"] > 0)
    metrics["TV_plus_TR"] = metrics["TV"] + metrics["TR"]
    metrics["recycling_priority_supplement"] = metrics["TV"] + metrics["TR"] + metrics["CR"]
    metrics["novelty_orientation"] = np.where(metrics["TN"] >= 0, "exploration-oriented", "exploitation-oriented")
    metrics["circularity_orientation"] = np.where(metrics["CR"] >= 0, "circularity-oriented", "serviceability-leaning")
    metrics = metrics.sort_values(["key_cluster","TV_plus_TR"], ascending=[False,False]).reset_index(drop=True)
    metrics.to_parquet(CLUSTER_METRICS, index=False)
    save_table(metrics, "13_cluster_metrics_TV_TR_TN_CR", True)

    fig, ax = plt.subplots(figsize=(10, 8))
    sc = ax.scatter(
        metrics["TR"], metrics["TV"], c=metrics["TN"],
        s=40 + 4*np.sqrt(metrics["cluster_size"]), alpha=.75
    )
    ax.axhline(0, color="black", linewidth=1)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_xlabel("TR (z-score)")
    ax.set_ylabel("TV (z-score)")
    ax.set_title("Technology-opportunity clusters: TV × TR (color = TN)")
    fig.colorbar(sc, ax=ax, label="TN")
    keys = metrics[metrics["key_cluster"]]
    for _, r in keys.head(12).iterrows():
        ax.annotate(str(int(r["cluster_id"])), (r["TR"], r["TV"]), fontsize=8)

    q_counts = {
        "High TV / High TR": int(((metrics.TV>0)&(metrics.TR>0)).sum()),
        "High TV / Low TR": int(((metrics.TV>0)&(metrics.TR<=0)).sum()),
        "Low TV / High TR": int(((metrics.TV<=0)&(metrics.TR>0)).sum()),
        "Low TV / Low TR": int(((metrics.TV<=0)&(metrics.TR<=0)).sum()),
    }
    text = (
        f"Evaluated {len(metrics)} zero-shot clusters; TN global Q3={q3:.4f}; quadrant counts={q_counts}; "
        f"key clusters={len(keys)}.\nTop key clusters:\n"
        + keys[["cluster_id","label_text","cluster_size","TV","TR","TN","CR","TV_plus_TR"]].head(15).to_string(index=False)
    )
    save_figure(fig, "13_TV_TR_scatter", text, metrics)

    top = keys.head(5).copy()
    if len(top):
        dims = ["TV","TR","TN","CR"]
        norm = top[["cluster_id","label_text"] + dims].copy()
        for c in dims:
            lo, hi = metrics[c].min(), metrics[c].max()
            norm[c] = (top[c]-lo)/(hi-lo) if hi>lo else 0.5
        angles = np.linspace(0, 2*np.pi, len(dims), endpoint=False).tolist()
        angles += angles[:1]
        fig = plt.figure(figsize=(9, 8))
        ax = plt.subplot(111, polar=True)
        for _, r in norm.iterrows():
            vals = [r[c] for c in dims] + [r[dims[0]]]
            ax.plot(angles, vals, linewidth=2, label=f"Cluster {int(r['cluster_id'])}")
            ax.fill(angles, vals, alpha=.08)
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(dims)
        ax.set_ylim(0,1)
        ax.set_title("Top key clusters across TV, TR, TN and CR")
        ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1))
        text2 = (
            "Radar values are min-max normalized across all clusters. Raw plotted values:\n"
            + top[["cluster_id","label_text","TV","TR","TN","CR","cluster_size"]].to_string(index=False)
        )
        save_figure(fig, "13_key_cluster_radar", text2, top)

    body = f"""
Clusters were evaluated using **TV**, **TR**, **TN**, and the added independent **CR** dimension.
TV combines age-adjusted forward citations and simple-family breadth; TR is semantic proximity to the target;
TN is the negative z-score of recency-weighted overlap with field sentences above global Q3
(**Q3={q3:.4f}**); CR is the cluster mean circularity-orientation score, z-normalized across clusters.
Following the source screening logic, key clusters satisfy **TV > 0 and TR > 0**, yielding **{len(keys)} key clusters**.
"""
    R.update_section("13_metrics", "Technology-opportunity cluster evaluation", body)
    R.emit(body)
    mark_stage_done(stage, [CLUSTER_METRICS], extra={"tn_q3": q3, "key_clusters": len(keys)})


# %% CELL: STAGE_14
# ============================================================
# KEY CLUSTER NETWORKS, KEY PATENTS, KEY SENTENCES
# ============================================================

KEY_PATENTS_PATH = DATA_DIR / "14_key_patents_all_clusters.parquet"
KEY_SENTENCES_PATH = DATA_DIR / "14_key_sentences_all_clusters.parquet"
KEY_NODES_PATH = DATA_DIR / "14_key_nodes_all_clusters.parquet"

def eigen_or_frequency_centrality(G):
    if G.number_of_nodes() < 3 or G.number_of_edges() < 2:
        freq = dict(G.nodes(data="frequency", default=1))
        s = sum(freq.values()) or 1
        return {k: v/s for k,v in freq.items()}, "frequency_fallback"
    try:
        return nx.eigenvector_centrality(G, max_iter=2000, weight="weight"), "eigenvector"
    except Exception:
        freq = dict(G.degree(weight="weight"))
        s = sum(freq.values()) or 1
        return {k: v/s for k,v in freq.items()}, "weighted_degree_fallback"

def build_cooccurrence(items_by_doc):
    G = nx.Graph()
    edge_counter = Counter()
    node_freq = Counter()
    for items in items_by_doc:
        items = sorted(set(x for x in items if x))
        for x in items:
            node_freq[x] += 1
        for a,b in combinations(items,2):
            edge_counter[(a,b)] += 1
    for n,f in node_freq.items():
        G.add_node(n, frequency=f)
    for (a,b),w in edge_counter.items():
        G.add_edge(a,b,weight=w)
    return G

def network_tables(G, cluster_id, network_type):
    cent, method = eigen_or_frequency_centrality(G)
    nodes = pd.DataFrame([
        {
            "cluster_id": cluster_id,
            "network_type": network_type,
            "node": n,
            "frequency": G.nodes[n].get("frequency",0),
            "centrality": cent.get(n,0),
            "centrality_method": method,
        } for n in G.nodes()
    ])
    if len(nodes):
        nodes = nodes.sort_values("centrality", ascending=False)
    edges = pd.DataFrame([
        {
            "cluster_id": cluster_id,
            "network_type": network_type,
            "source": a, "target": b, "weight": d.get("weight",1)
        } for a,b,d in G.edges(data=True)
    ])
    if len(edges):
        edges = edges.sort_values("weight", ascending=False)
    return nodes, edges

def plot_network(G, node_df, name, title):
    if G.number_of_nodes() == 0 or len(node_df) == 0:
        return
    top_nodes = set(node_df.head(NETWORK_PLOT_TOP_NODES)["node"])
    H = G.subgraph(top_nodes).copy()
    if H.number_of_edges() > NETWORK_PLOT_TOP_EDGES:
        edges_sorted = sorted(H.edges(data=True), key=lambda x: x[2].get("weight",1), reverse=True)[:NETWORK_PLOT_TOP_EDGES]
        H2 = nx.Graph()
        H2.add_nodes_from(H.nodes(data=True))
        H2.add_edges_from(edges_sorted)
        H = H2

    pos = nx.spring_layout(H, seed=RANDOM_SEED, weight="weight")
    freq = np.array([H.nodes[n].get("frequency",1) for n in H.nodes()], dtype=float)
    sizes = 250 + 1200*(freq/max(freq.max(),1))**0.6
    widths = [0.5 + 2*np.log1p(H.edges[e].get("weight",1)) for e in H.edges()]

    fig, ax = plt.subplots(figsize=(12, 9))
    nx.draw_networkx_edges(H, pos, width=widths, alpha=.25, ax=ax)
    nx.draw_networkx_nodes(H, pos, node_size=sizes, alpha=.85, ax=ax)
    nx.draw_networkx_labels(H, pos, font_size=7, ax=ax)
    ax.set_title(title)
    ax.axis("off")
    summary = (
        f"Displayed {H.number_of_nodes()} nodes and {H.number_of_edges()} edges; "
        f"original network={G.number_of_nodes()} nodes/{G.number_of_edges()} edges.\nTop nodes:\n"
        + node_df.head(15)[["node","frequency","centrality"]].to_string(index=False)
    )
    save_figure(fig, name, summary)

def stage_14_key_cluster_analysis():
    stage = "14_key_cluster_analysis"
    stage_banner(stage)
    if stage_done(stage, [KEY_PATENTS_PATH, KEY_SENTENCES_PATH, KEY_NODES_PATH]):
        R.emit("Checkpoint found — key cluster analysis skipped.")
        return

    df = pd.read_parquet(FAMILY_PARQUET)
    zs = pd.read_parquet(ZS_ASSIGN)
    metrics = pd.read_parquet(CLUSTER_METRICS)
    rawf = pd.read_parquet(RAW_FUNCTIONS)
    fmap = pd.read_parquet(FUNCTION_MAP)

    rawf = rawf.merge(
        fmap[["function_phrase","prototype_phrase"]].drop_duplicates(),
        on="function_phrase", how="left"
    )
    rawf["prototype_phrase"] = rawf["prototype_phrase"].fillna(rawf["function_phrase"])

    work = df.merge(zs[["doc_idx","zeroshot_label_id","assigned"]], on="doc_idx")
    work = work[work["assigned"]].copy()
    keys = metrics[metrics["key_cluster"]].sort_values("TV_plus_TR", ascending=False)
    if not ANALYZE_ALL_KEY_CLUSTERS:
        keys = keys.head(10)

    all_nodes, all_key_patents, all_key_sentences = [], [], []

    for _, kr in keys.iterrows():
        cid = int(kr["cluster_id"])
        cdir = NETWORK_DIR / f"cluster_{cid:03d}"
        cdir.mkdir(parents=True, exist_ok=True)
        done_marker = cdir / "DONE.json"

        if done_marker.exists():
            R.emit(f"[cluster {cid}] cached.")
            for fn, collector in [
                ("nodes.parquet", all_nodes),
                ("key_patents.parquet", all_key_patents),
                ("key_sentences.parquet", all_key_sentences)
            ]:
                p = cdir / fn
                if p.exists():
                    collector.append(pd.read_parquet(p))
            continue

        cluster_docs = work[work["zeroshot_label_id"] == cid].copy()
        doc_ids = set(cluster_docs["doc_idx"].astype(int))

        cpc_by_doc = {}
        for _, r in cluster_docs.iterrows():
            subclasses = sorted(set(filter(None, (cpc_subclass(x) for x in parse_cpc_codes(r["cpc_classifications"])))))
            cpc_by_doc[int(r["doc_idx"])] = subclasses

        fsub = rawf[rawf["doc_idx"].isin(doc_ids)]
        func_by_doc = fsub.groupby("doc_idx")["prototype_phrase"].apply(lambda s: sorted(set(s))).to_dict()

        cpc_G = build_cooccurrence([cpc_by_doc.get(i,[]) for i in doc_ids])
        func_G = build_cooccurrence([func_by_doc.get(i,[]) for i in doc_ids])
        cpc_nodes, cpc_edges = network_tables(cpc_G, cid, "CPC_subclass")
        fn_nodes, fn_edges = network_tables(func_G, cid, "Function")

        nodes = pd.concat([cpc_nodes, fn_nodes], ignore_index=True)
        edges = pd.concat([cpc_edges, fn_edges], ignore_index=True)
        nodes.to_parquet(cdir / "nodes.parquet", index=False)
        edges.to_parquet(cdir / "edges.parquet", index=False)
        all_nodes.append(nodes)

        cpc_cent = cpc_nodes.set_index("node")["centrality"] if len(cpc_nodes) else pd.Series(dtype=float)
        fn_cent = fn_nodes.set_index("node")["centrality"] if len(fn_nodes) else pd.Series(dtype=float)
        cpc_z = dict(zip(cpc_cent.index, safe_z(cpc_cent.values)))
        fn_z = dict(zip(fn_cent.index, safe_z(fn_cent.values)))

        p_rows = []
        for _, r in cluster_docs.iterrows():
            did = int(r["doc_idx"])
            cpcs = cpc_by_doc.get(did, [])
            funcs = func_by_doc.get(did, [])
            score = sum(cpc_z.get(x,0) for x in cpcs) + sum(fn_z.get(x,0) for x in funcs)
            p_rows.append({
                "cluster_id": cid, "doc_idx": did, "family_id": r["family_id"],
                "lens_id": r.get("lens_id"), "title": r.get("title"),
                "applicants": r.get("applicants"), "owners": r.get("owners"),
                "publication_date": r.get("publication_date"),
                "patent_centrality_score": float(score),
                "cpc_subclasses": " ; ".join(cpcs),
                "normalized_functions": " ; ".join(funcs),
            })
        ptab = pd.DataFrame(p_rows).sort_values("patent_centrality_score", ascending=False)
        keyp = ptab.head(KEY_PATENTS_PER_CLUSTER).copy()
        keyp.to_parquet(cdir / "key_patents.parquet", index=False)
        all_key_patents.append(keyp)

        top_funcs = set(fn_nodes.head(5)["node"]) if len(fn_nodes) else set()
        ks_rows = []
        for _, p in keyp.iterrows():
            cand = fsub[
                (fsub["doc_idx"] == int(p["doc_idx"])) &
                (fsub["prototype_phrase"].isin(top_funcs))
            ].drop_duplicates("sentence")
            if len(cand) == 0:
                cand = fsub[fsub["doc_idx"] == int(p["doc_idx"])].drop_duplicates("sentence")
            for _, rr in cand.head(KEY_SENTENCES_PER_PATENT).iterrows():
                ks_rows.append({
                    "cluster_id": cid, "doc_idx": int(p["doc_idx"]), "family_id": p["family_id"],
                    "lens_id": p["lens_id"], "title": p["title"],
                    "key_function": rr["prototype_phrase"], "mechanism": rr.get("mechanism",""),
                    "sentence": rr["sentence"], "patent_centrality_score": p["patent_centrality_score"],
                })
        ksent = pd.DataFrame(ks_rows)
        ksent.to_parquet(cdir / "key_sentences.parquet", index=False)
        all_key_sentences.append(ksent)

        plot_network(cpc_G, cpc_nodes, f"14_cluster_{cid:03d}_cpc_network", f"Cluster {cid}: CPC-subclass co-occurrence")
        plot_network(func_G, fn_nodes, f"14_cluster_{cid:03d}_function_network", f"Cluster {cid}: function/mechanism co-occurrence")

        summary = {
            "cluster_id": cid,
            "label": kr["label_text"],
            "n_patent_families": len(cluster_docs),
            "cpc_nodes": cpc_G.number_of_nodes(),
            "cpc_edges": cpc_G.number_of_edges(),
            "function_nodes": func_G.number_of_nodes(),
            "function_edges": func_G.number_of_edges(),
            "top_cpc_nodes": cpc_nodes.head(5).to_dict("records"),
            "top_function_nodes": fn_nodes.head(5).to_dict("records"),
            "key_patents": keyp.head(5)[["lens_id","title","patent_centrality_score"]].to_dict("records"),
        }
        atomic_write_json(done_marker, summary)
        R.emit(json.dumps(summary, indent=2, ensure_ascii=False, default=str))

    nodes_all = pd.concat(all_nodes, ignore_index=True) if all_nodes else pd.DataFrame()
    kp_all = pd.concat(all_key_patents, ignore_index=True) if all_key_patents else pd.DataFrame()
    ks_all = pd.concat(all_key_sentences, ignore_index=True) if all_key_sentences else pd.DataFrame()

    nodes_all.to_parquet(KEY_NODES_PATH, index=False)
    kp_all.to_parquet(KEY_PATENTS_PATH, index=False)
    ks_all.to_parquet(KEY_SENTENCES_PATH, index=False)

    save_table(nodes_all, "14_all_key_cluster_nodes", True)
    save_table(kp_all, "14_all_key_patents", True)
    save_table(ks_all, "14_all_key_sentences", True)

    body = f"""
For every key cluster, a CPC-subclass co-occurrence network and a normalized mechanism-function network were built.
Edges count patent-family co-occurrence. Weighted eigenvector centrality identifies key nodes, with small-network
fallbacks. Patent importance is the sum of z-normalized centralities of its CPC and function nodes.
The stage produced **{len(kp_all):,} key-patent records** and **{len(ks_all):,} traceable key sentences**
across **{len(keys)} key clusters**.
"""
    R.update_section("14_networks", "Key-cluster network analysis and traceable evidence", body)
    R.emit(body)
    mark_stage_done(stage, [KEY_PATENTS_PATH, KEY_SENTENCES_PATH, KEY_NODES_PATH])


# %% CELL: STAGES_15_16
# ============================================================
# VALIDATION READINESS, LLM PROMPTS, FINAL ARTIFACT INDEX
# ============================================================

LLM_PROMPT_INDEX = DATA_DIR / "15_llm_prompt_index.parquet"

def stage_15_validation_and_llm_prompts():
    stage = "15_validation_llm_prompts"
    stage_banner(stage)
    marker = CHECKPOINT_DIR / "15_validation_llm_prompts.done"
    if stage_done(stage, [marker, LLM_PROMPT_INDEX]):
        R.emit("Checkpoint found — validation/LLM prompt package skipped.")
        return

    metrics = pd.read_parquet(CLUSTER_METRICS)
    keysent = pd.read_parquet(KEY_SENTENCES_PATH)
    nodes = pd.read_parquet(KEY_NODES_PATH)

    validation = pd.DataFrame([
        {
            "validation_component": "Historical data-based validation",
            "status": "Not executed by default",
            "reason": (
                "The export contains cumulative Cited by Patent Count but no citation-event dates. "
                "Leakage-free historical TV reconstruction requires dated citation events or citing-patent dates."
            )
        },
        {
            "validation_component": "Expert validation",
            "status": "Requires human experts",
            "reason": (
                "Experts should assess relevance, applicability, novelty, and usefulness of fields, mechanisms, "
                "key patents, and key sentences."
            )
        },
        {
            "validation_component": "Internal semantic diagnostics",
            "status": "Executed",
            "reason": (
                "HDBSCAN persistence/relative validity, zero-shot threshold diagnostics, semantic similarities, "
                "coverage, and traceable evidence are saved."
            )
        },
    ])
    save_table(validation, "15_validation_readiness", True)

    keys = metrics[metrics["key_cluster"]].sort_values("TV_plus_TR", ascending=False).head(N_LLM_CLUSTERS)
    prompt_rows = []
    for _, r in keys.iterrows():
        cid = int(r["cluster_id"])
        ref_nodes = nodes[(nodes["cluster_id"]==cid)&(nodes["network_type"]=="CPC_subclass")].head(5)
        fn_nodes = nodes[(nodes["cluster_id"]==cid)&(nodes["network_type"]=="Function")].head(5)
        sent = keysent[keysent["cluster_id"]==cid].head(15)

        prompt = f"""
You are a technology innovation strategist supporting circular product design R&D.

TASK
Generate exactly {N_SOLUTIONS_PER_CLUSTER} technically distinct solution concepts that:
1. Address the target technology challenge.
2. Transfer and recombine principles from the identified cross-field patent cluster.
3. Use the extracted key functions/mechanisms.
4. Improve recycling-oriented disassembly or recyclability rather than merely copying prior art.
5. Preserve traceability to the supplied patent evidence.

TARGET TECHNOLOGY
{TARGET_TECHNOLOGY}

FIELD OF INTEREST
{FIELD_OF_INTEREST}

CLUSTER
ID: {cid}
Label: {r['label_text']}
TV={r['TV']:.4f}; TR={r['TR']:.4f}; TN={r['TN']:.4f}; CR={r['CR']:.4f}

KEY CPC SUBCLASSES
{ref_nodes[['node','frequency','centrality']].to_string(index=False) if len(ref_nodes) else 'None'}

KEY FUNCTIONS / MECHANISMS
{fn_nodes[['node','frequency','centrality']].to_string(index=False) if len(fn_nodes) else 'None'}

TRACEABLE KEY PATENT SENTENCES
{sent[['lens_id','key_function','sentence']].to_string(index=False) if len(sent) else 'None'}

OUTPUT FOR EACH SOLUTION
- Solution number
- Title
- Core innovation
- Cross-field integration
- Key functions/mechanisms applied
- Technical feasibility (High/Medium/Low + justification)
- Circularity contribution
- Competitive differentiation from supplied prior art
- Implementation pathway (3-4 steps)
- Potential technical challenges
- Evidence references (Lens IDs / sentences used)

Do not claim freedom-to-operate or legal clearance.
""".strip()

        p = LLM_DIR / f"cluster_{cid:03d}_prompt.txt"
        atomic_write_text(p, prompt)
        R.emit(f"\n===== LLM PROMPT FOR CLUSTER {cid} =====\n{prompt}\n")
        prompt_rows.append({
            "cluster_id": cid, "label_text": r["label_text"], "prompt_path": str(p),
            "TV": r["TV"], "TR": r["TR"], "TN": r["TN"], "CR": r["CR"],
        })

    prompt_index = pd.DataFrame(prompt_rows)
    prompt_index.to_parquet(LLM_PROMPT_INDEX, index=False)
    save_table(prompt_index, "15_llm_prompt_index", True)

    if ENABLE_LOCAL_LLM_IDEATION:
        R.emit("Local LLM ideation ENABLED. Installing/loading 4-bit model if supported...")
        try:
            import bitsandbytes  # noqa
        except Exception:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes>=0.45"])
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        qconf = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
        tok = AutoTokenizer.from_pretrained(LOCAL_LLM_MODEL)
        llm = AutoModelForCausalLM.from_pretrained(
            LOCAL_LLM_MODEL, device_map="auto",
            quantization_config=qconf, torch_dtype=torch.float16
        )
        for _, pr in prompt_index.iterrows():
            cid = int(pr["cluster_id"])
            outp = LLM_DIR / f"cluster_{cid:03d}_local_llm_output.txt"
            if outp.exists():
                R.emit(f"[LLM cached] {outp.name}")
                R.emit(outp.read_text(encoding="utf-8"))
                continue
            prompt = Path(pr["prompt_path"]).read_text(encoding="utf-8")
            text = tok.apply_chat_template([{"role":"user","content":prompt}], tokenize=False, add_generation_prompt=True)
            inputs = tok(text, return_tensors="pt").to(llm.device)
            with torch.no_grad():
                gen = llm.generate(**inputs, max_new_tokens=2200, do_sample=False)
            result = tok.decode(gen[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            atomic_write_text(outp, result)
            R.emit(result)
        del llm
        torch.cuda.empty_cache()
        llm_status = f"Local ideation executed with {LOCAL_LLM_MODEL}."
    else:
        llm_status = "Prompts saved; local inference not executed. Set ENABLE_LOCAL_LLM_IDEATION=True if desired."

    body = f"""
A leakage-free historical validation is not automatically claimed because current forward-citation counts are
cumulative and citation-event dates are absent. Expert validation remains a human step. Structured evidence-grounded
LLM prompts were prepared for **{len(prompt_index)} key clusters**. {llm_status}
"""
    R.update_section("15_validation_llm", "Validation readiness and solution-concept prompts", body)
    R.emit(body)

    marker.write_text("done", encoding="utf-8")
    mark_stage_done(stage, [marker, LLM_PROMPT_INDEX])

def stage_16_finalize():
    stage = "16_finalize"
    stage_banner(stage)
    package_environment_snapshot()

    rows = []
    for p in sorted(OUTPUT_ROOT.rglob("*")):
        if p.is_file() and p.name != "artifact_index.csv":
            rows.append({
                "relative_path": str(p.relative_to(OUTPUT_ROOT)),
                "size_bytes": p.stat().st_size,
                "modified": datetime.fromtimestamp(p.stat().st_mtime).isoformat(),
            })
    idx = pd.DataFrame(rows)
    idx.to_csv(OUTPUT_ROOT / "artifact_index.csv", index=False)
    save_table(idx, "16_artifact_index", True)

    bundle = OUTPUT_ROOT / "article_ready_bundle.zip"
    with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for root in [REPORT_DIR, FIG_DIR, TABLE_DIR, TABLE_TEXT_DIR, LLM_DIR, ENV_DIR]:
            for p in root.rglob("*"):
                if p.is_file():
                    zf.write(p, p.relative_to(OUTPUT_ROOT))
        for p in [CONFIG_PATH, MANIFEST_PATH, OUTPUT_ROOT / "artifact_index.csv"]:
            if p.exists():
                zf.write(p, p.relative_to(OUTPUT_ROOT))

    body = f"""
All artifacts are persisted under `{OUTPUT_ROOT}`.
A compact article-writing bundle is saved at `{bundle}`.
The checkpoint manifest is `{MANIFEST_PATH}`.
"""
    R.update_section("16_final", "Reproducibility and saved outputs", body)
    R.emit(body)
    mark_stage_done(stage, [OUTPUT_ROOT / "artifact_index.csv", bundle])

    if REPLAY_ALL_OUTPUTS_AT_END:
        R.replay_all_saved()


# %% CELL: CORRECTION_V2_OVERRIDES
# ============================================================
# CORRECTED METHODOLOGY V2
#
# Corrections:
#   1) TV follows the source-paper order exactly:
#      z-score(cluster median) FIRST, cluster-size factor SECOND.
#   2) Reference-field score uses log(N), not log(1+N), and requires
#      minimum empirical support. Y02 is excluded as a cross-sectional tag.
#   3) Function and mechanism are separated:
#      zero-shot labels use clean normalized Action–Object functions;
#      mechanisms are ranked and reported independently.
#   4) Zero-shot threshold is selected from the empirical assignment curve
#      (elbow), NOT by matching the source paper's 73.4% assignment rate.
#   5) "Circularity Relevance" is renamed Recycling Orientation Score (ROS).
#   6) Small clusters are retained but flagged; article-primary clusters
#      require >= 10 patent families.
#
# This cell intentionally overrides selected V1 functions. Earlier expensive
# outputs (clean data, patent embeddings, UMAP, base BERTopic, raw spaCy
# extraction) are reused from Drive when available.
# ============================================================

PIPELINE_REVISION = "V2_1_NO_FAISS_2026_09_14"

# Reference-field robustness
REF_MIN_COUNT = 3
REF_MIN_SHARE = 0.05
EXCLUDED_REFERENCE_CPC_CLASSES = {"Y02"}

# Function-quality filtering
BAD_FUNCTION_OBJECTS_EXACT = {
    "efficiency", "operation efficiency", "tool", "operation",
    "method", "process", "system", "device", "apparatus",
    "information", "thing"
}
MIN_FUNCTION_PATENT_FAMILIES = 2
TOP_MECHANISMS_PER_FUNCTION = 10

# Zero-shot selection
ZEROSHOT_MIN_ACCEPTABLE_ASSIGNMENT = 0.50
ZEROSHOT_MAX_ACCEPTABLE_ASSIGNMENT = 0.95
ZEROSHOT_BORDERLINE_BAND = 0.03
ZEROSHOT_AUDIT_SAMPLE_SIZE = 30

# Primary article ranking
PRIMARY_MIN_CLUSTER_SIZE = 10
ANALYZE_PRIMARY_KEY_CLUSTERS_ONLY = True

# Revised output paths
FUNCTION_MECHANISMS = DATA_DIR / "09_selected_function_mechanisms.parquet"
ROS_PATH = DATA_DIR / "10_patent_recycling_orientation.parquet"
CR_PATH = ROS_PATH  # backward-compatible path alias only; V2 uses ROS columns
ZS_SIM_MATRIX = EMBED_DIR / "11_zeroshot_similarity_matrix_v2.npy"
FIELD_SENT_EMB = EMBED_DIR / "field_sentence_embeddings_v2.npy"
CLUSTER_METRICS = DATA_DIR / "13_cluster_metrics_corrected.parquet"
KEY_PATENTS_PATH = DATA_DIR / "14_key_patents_all_clusters_corrected.parquet"
KEY_SENTENCES_PATH = DATA_DIR / "14_key_sentences_all_clusters_corrected.parquet"
KEY_NODES_PATH = DATA_DIR / "14_key_nodes_all_clusters_corrected.parquet"
NETWORK_DIR_V2 = NETWORK_DIR / "corrected_v2"
NETWORK_DIR_V2.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Archive superseded downstream V1 outputs once, while keeping expensive
# stable checkpoints and raw function extraction.
# ---------------------------------------------------------------------
V2_MIGRATION_MARKER = CHECKPOINT_DIR / "V2_CORRECTION_MIGRATED.json"

def _archive_v1_downstream_once():
    if V2_MIGRATION_MARKER.exists():
        return

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    archive = ARCHIVE_DIR / f"pre_v2_downstream_{stamp}"
    archive.mkdir(parents=True, exist_ok=True)

    prefixes = ("07_", "09_", "10_", "11_", "12_", "13_", "14_", "15_", "16_")

    for root in [DATA_DIR, TABLE_DIR, TABLE_TEXT_DIR, FIG_DIR]:
        rel_root = archive / root.name
        rel_root.mkdir(parents=True, exist_ok=True)
        for p in list(root.glob("*")):
            if not p.is_file():
                continue
            # Keep official CPC title cache; it is source data, not an analytical result.
            if root == DATA_DIR and p.name == "07_cpc_titles.parquet":
                continue
            if p.name.startswith(prefixes):
                shutil.move(str(p), str(rel_root / p.name))

    # Preserve old LLM prompt package.
    if LLM_DIR.exists() and any(LLM_DIR.iterdir()):
        llm_archive = archive / "llm"
        llm_archive.mkdir(parents=True, exist_ok=True)
        for p in list(LLM_DIR.iterdir()):
            shutil.move(str(p), str(llm_archive / p.name))

    # Preserve old network directory content, but leave the root available.
    if NETWORK_DIR.exists():
        old_items = [p for p in NETWORK_DIR.iterdir() if p.name != "corrected_v2"]
        if old_items:
            net_archive = archive / "networks"
            net_archive.mkdir(parents=True, exist_ok=True)
            for p in old_items:
                shutil.move(str(p), str(net_archive / p.name))

    atomic_write_json(V2_MIGRATION_MARKER, {
        "pipeline_revision": PIPELINE_REVISION,
        "archived_at": datetime.now(timezone.utc).isoformat(),
        "archive_path": str(archive),
        "note": "Superseded V1 downstream analytical outputs archived; stable preprocessing/embedding checkpoints retained."
    })
    R.emit(f"[V2 migration] Superseded downstream V1 outputs archived to: {archive}")

_archive_v1_downstream_once()

R.update_section(
    "v2_correction",
    "Corrected analysis revision",
    f"""
This report was regenerated with **{PIPELINE_REVISION}**. The corrected revision:
(1) applies z-score normalization before cluster-size weighting in TV;
(2) uses the source-paper `log(N)` CPC score plus minimum support filtering;
(3) separates normalized Action–Object functions from implementation mechanisms;
(4) selects the zero-shot cutoff from the empirical assignment-rate elbow rather than reproducing another study's assignment rate;
(5) interprets the recycling-vs-service semantic contrast as **Recycling Orientation Score (ROS)**; and
(6) distinguishes all high-TV/high-TR clusters from the article-primary subset with at least **{PRIMARY_MIN_CLUSTER_SIZE}** patent families.
"""
)

# ---------------------------------------------------------------------
# V2 Stage 07 — robust CPC reference fields
# ---------------------------------------------------------------------
def stage_07_reference_fields():
    stage = "07_reference_fields_v2"
    stage_banner(stage)
    required = [CPC_TITLES_PATH, REF_FIELD_SCORES, REP_FIELDS_PATH]
    if stage_done(stage, required):
        R.emit("V2 checkpoint found — reference-field extraction skipped.")
        return

    df = pd.read_parquet(FAMILY_PARQUET)
    assign = pd.read_parquet(BASE_ASSIGN)
    emb = np.asarray(np.load(PATENT_EMB, mmap_mode="r"), dtype=np.float32)

    cpc_titles = pd.read_parquet(CPC_TITLES_PATH) if CPC_TITLES_PATH.exists() else download_and_parse_cpc_titles()
    class_titles = cpc_titles[cpc_titles["cpc_symbol"].str.match(r"^[A-HY]\d{2}$", na=False)].copy()
    title_map = dict(zip(class_titles["cpc_symbol"], class_titles["cpc_title"]))

    work = df[["doc_idx","family_id","cpc_classifications"]].merge(
        assign[["doc_idx","base_topic"]], on="doc_idx", how="left"
    )

    topic_sizes = (
        work[work["base_topic"] >= 0]
        .groupby("base_topic")["family_id"]
        .nunique()
        .to_dict()
    )

    rows = []
    for _, r in work.iterrows():
        if r["base_topic"] < 0:
            continue
        classes = sorted(set(filter(None, (cpc_class(x) for x in parse_cpc_codes(r["cpc_classifications"])))))
        for cc in classes:
            rows.append((int(r["doc_idx"]), r["family_id"], int(r["base_topic"]), cc))

    long = pd.DataFrame(rows, columns=["doc_idx","family_id","base_topic","cpc_class"]).drop_duplicates()
    save_table(long, "07_doc_cpc_class_long", print_full=True)

    candidate_classes = sorted(
        cc for cc in long["cpc_class"].unique()
        if cc not in EXCLUDED_REFERENCE_CPC_CLASSES
    )
    if not candidate_classes:
        raise RuntimeError("No CPC classes remain after V2 exclusions.")

    cpc_texts = [f"{cc}: {title_map.get(cc, 'CPC class '+cc)}" for cc in candidate_classes]
    cpc_emb_path = encode_texts_sharded(
        cpc_texts,
        "cpc_class_title_embeddings_v2",
        shard_size=1000
    )
    cpc_emb = np.asarray(np.load(cpc_emb_path), dtype=np.float32)
    cpc_vec = {cc: cpc_emb[i] for i, cc in enumerate(candidate_classes)}

    score_rows = []
    for topic, grp in long.groupby("base_topic"):
        doc_ids = grp["doc_idx"].unique().astype(int)
        centroid = l2_normalize(emb[doc_ids].mean(axis=0, keepdims=True))[0]
        counts = grp.groupby("cpc_class")["family_id"].nunique().sort_values(ascending=False)
        cluster_size = int(topic_sizes.get(topic, len(doc_ids)))

        tmp = []
        for cc, cnt in counts.items():
            if cc in EXCLUDED_REFERENCE_CPC_CLASSES:
                sim = np.nan
            else:
                sim = float(np.dot(cpc_vec[cc], centroid))
            tmp.append({
                "base_topic": int(topic),
                "cluster_size": cluster_size,
                "cpc_class": cc,
                "count": int(cnt),
                "support_share": float(cnt / max(cluster_size, 1)),
                "semantic_similarity": sim,
                "excluded_cross_sectional_tag": cc in EXCLUDED_REFERENCE_CPC_CLASSES,
            })

        t = pd.DataFrame(tmp)
        valid_sim = t["semantic_similarity"].notna()
        t["similarity_z"] = np.nan
        if valid_sim.any():
            t.loc[valid_sim, "similarity_z"] = safe_z(t.loc[valid_sim, "semantic_similarity"]).values

        # Exact source-paper count transform: log(N), not log(1+N).
        t["log_count"] = np.log(t["count"].clip(lower=1))
        t["reference_field_score"] = t["log_count"] * t["similarity_z"]

        # Robustness requirement for calling a field "representative".
        t["eligible_support"] = (
            (t["count"] >= REF_MIN_COUNT)
            & (t["support_share"] >= REF_MIN_SHARE)
            & (~t["excluded_cross_sectional_tag"])
        )
        t["cpc_title"] = t["cpc_class"].map(title_map).fillna(
            t["cpc_class"].map(lambda x: f"CPC class {x}")
        )
        score_rows.append(t)

    scores = pd.concat(score_rows, ignore_index=True).sort_values(
        ["base_topic","eligible_support","reference_field_score"],
        ascending=[True,False,False]
    )
    scores.to_parquet(REF_FIELD_SCORES, index=False)
    save_table(scores, "07_reference_field_scores", print_full=True)

    eligible = scores[scores["eligible_support"]].copy()
    if len(eligible) == 0:
        raise RuntimeError(
            "No reference fields satisfy V2 support criteria. "
            "Inspect 07_reference_field_scores.csv and relax REF_MIN_COUNT/REF_MIN_SHARE only if substantively justified."
        )

    reps = (
        eligible.sort_values(["base_topic","reference_field_score"], ascending=[True,False])
        .groupby("base_topic", as_index=False)
        .first()
    )
    reps["cluster_rank"] = reps["reference_field_score"].rank(
        ascending=False, method="dense"
    ).astype(int)
    reps.to_parquet(REP_FIELDS_PATH, index=False)
    save_table(reps, "07_representative_reference_fields_by_cluster", print_full=True)

    excluded_topics = sorted(set(topic_sizes) - set(reps["base_topic"]))
    excluded_df = pd.DataFrame({
        "base_topic": excluded_topics,
        "reason": "No CPC class satisfied minimum count/share support after cross-sectional-tag exclusion"
    })
    save_table(excluded_df, "07_base_topics_without_supported_reference_field", print_full=True)

    unique_fields = (
        reps.sort_values("reference_field_score", ascending=False)
        .drop_duplicates("cpc_class")
    )
    if N_REP_REFERENCE_FIELDS:
        unique_fields = unique_fields.head(N_REP_REFERENCE_FIELDS)
    save_table(unique_fields, "07_unique_reference_fields_for_zeroshot", print_full=True)

    support_diag = reps[[
        "base_topic","cluster_size","cpc_class","cpc_title",
        "count","support_share","semantic_similarity","reference_field_score"
    ]].copy()
    save_table(support_diag, "07_selected_reference_field_support_diagnostics", print_full=True)

    body = f"""
The corrected reference-field stage follows the source-paper count transform **log(N)** rather than `log(1+N)`.
A CPC class is eligible to represent a semantic cluster only when it is observed in at least
**{REF_MIN_COUNT} patent families** and at least **{REF_MIN_SHARE:.0%} of the cluster**.
The cross-sectional tag **Y02** is excluded from reference-field selection.

This produced **{len(reps)} supported cluster-level reference fields** and
**{unique_fields['cpc_class'].nunique()} unique fields** for zero-shot construction.
**{len(excluded_topics)} base clusters** were not assigned a representative field because no CPC class passed
the support criteria.
"""
    R.update_section("07_reference_fields", "Representative reference fields — corrected", body)
    R.emit(body)
    mark_stage_done(stage, required, extra={
        "min_count": REF_MIN_COUNT,
        "min_share": REF_MIN_SHARE,
        "excluded_cpc_classes": sorted(EXCLUDED_REFERENCE_CPC_CLASSES),
        "excluded_base_topics": excluded_topics,
    })


# ---------------------------------------------------------------------
# V2 Stage 09 — normalize clean Action–Object functions separately
# from mechanisms.
# ---------------------------------------------------------------------
def _clean_mechanism_text(x):
    x = clean_phrase(x, 180).lower()
    x = re.sub(r"^(by|via|through|using|with|upon|under)\s+", "", x).strip()
    x = re.sub(r"\s+", " ", x)
    if len(x) < 5:
        return ""
    if x in {"a method", "the method", "a system", "the system", "the device"}:
        return ""
    return x

def _function_quality_mask(raw):
    obj = raw["object"].fillna("").astype(str).str.lower().str.strip()
    ao = raw["action_object"].fillna("").astype(str).str.lower().str.strip()
    action = raw["action"].fillna("").astype(str).str.lower().str.strip()
    mask = (
        (ao.str.len() >= 5)
        & (~action.isin(GENERIC_VERBS))
        & (~obj.isin(BAD_FUNCTION_OBJECTS_EXACT))
        & (~ao.str.contains(r"\b(?:is|are|was|were) a\b", regex=True))
    )
    return mask

def stage_09_normalize_score_functions():
    stage = "09_normalize_score_functions_v2"
    stage_banner(stage)
    required = [FUNCTION_MAP, FUNCTION_SCORES, REP_FUNCTIONS, FUNCTION_MECHANISMS]
    if stage_done(stage, required):
        R.emit("V2 checkpoint found — function/mechanism normalization skipped.")
        return

    raw = pd.read_parquet(RAW_FUNCTIONS).copy()
    raw["quality_pass"] = _function_quality_mask(raw)
    clean_raw = raw[raw["quality_pass"]].copy()

    freq = (
        clean_raw.groupby("action_object")
        .agg(
            frequency=("family_id","size"),
            patent_families=("family_id","nunique")
        )
        .reset_index()
    )
    freq = freq[
        (freq["frequency"] >= FUNCTION_MIN_FREQUENCY)
        & (freq["patent_families"] >= MIN_FUNCTION_PATENT_FAMILIES)
    ].sort_values(["frequency","patent_families"], ascending=False)

    if len(freq) == 0:
        raise RuntimeError("No clean Action–Object functions remain after V2 filtering.")

    if len(freq) > MAX_FUNCTION_PHRASES_FOR_NORMALIZATION:
        R.emit(
            f"Function normalization candidate set limited from {len(freq):,} to "
            f"{MAX_FUNCTION_PHRASES_FOR_NORMALIZATION:,}; all raw extraction remains saved."
        )
        freq = freq.head(MAX_FUNCTION_PHRASES_FOR_NORMALIZATION).copy()

    phrases = freq["action_object"].tolist()
    femb_path = encode_texts_sharded(
        phrases,
        "function_action_object_embeddings_v2",
        shard_size=3000
    )
    femb = np.asarray(np.load(femb_path), dtype=np.float32)

    mapping, _ = greedy_semantic_normalization(
        phrases,
        femb,
        freq["frequency"].values,
        FUNCTION_NORMALIZATION_SIM_THRESHOLD
    )

    map_rows = []
    for idx, proto in mapping.items():
        map_rows.append({
            "action_object": phrases[idx],
            "prototype_function": phrases[proto],
            "similarity_to_prototype": float(np.dot(femb[idx], femb[proto])),
            "raw_frequency": int(freq.iloc[idx]["frequency"]),
            "raw_patent_families": int(freq.iloc[idx]["patent_families"]),
        })
    fmap = pd.DataFrame(map_rows)
    fmap.to_parquet(FUNCTION_MAP, index=False)
    save_table(fmap, "09_function_normalization_map", print_full=True)

    prot = (
        fmap.groupby("prototype_function")
        .agg(
            normalized_frequency=("raw_frequency","sum"),
            normalized_patent_families=("raw_patent_families","sum"),
            merged_variants=("action_object","nunique")
        )
        .reset_index()
    )

    model = get_embedder()
    target_vec = model.encode(
        [TARGET_TECHNOLOGY],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0].astype("float32")

    proto_vecs = model.encode(
        prot["prototype_function"].tolist(),
        batch_size=EMBED_BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    ).astype("float32")

    prot["target_similarity"] = proto_vecs @ target_vec
    prot["target_similarity_z"] = safe_z(prot["target_similarity"]).values
    prot["function_score"] = (
        np.log1p(prot["normalized_frequency"])
        * prot["target_similarity_z"]
    )
    prot = prot.sort_values("function_score", ascending=False).reset_index(drop=True)
    prot.to_parquet(FUNCTION_SCORES, index=False)
    save_table(prot, "09_representative_function_scores", print_full=True)

    reps = prot.head(N_REP_FUNCTIONS).copy()
    reps.insert(0, "function_rank", np.arange(1, len(reps)+1))
    reps.to_parquet(REP_FUNCTIONS, index=False)
    save_table(reps, "09_selected_representative_functions", print_full=True)

    # Attach normalized function to raw evidence and rank mechanisms separately.
    evidence = clean_raw.merge(
        fmap[["action_object","prototype_function"]].drop_duplicates(),
        on="action_object",
        how="inner"
    )
    evidence["mechanism_clean"] = evidence["mechanism"].map(_clean_mechanism_text)
    evidence = evidence[
        evidence["prototype_function"].isin(reps["prototype_function"])
    ].copy()

    mech = (
        evidence[evidence["mechanism_clean"].str.len() > 0]
        .groupby(["prototype_function","mechanism_clean"])
        .agg(
            frequency=("family_id","size"),
            patent_families=("family_id","nunique"),
            example_sentence=("sentence","first")
        )
        .reset_index()
    )
    if len(mech):
        mech["mechanism_score"] = (
            np.log1p(mech["frequency"])
            + np.log1p(mech["patent_families"])
        )
        mech = mech.sort_values(
            ["prototype_function","mechanism_score","patent_families"],
            ascending=[True,False,False]
        )
        mech["mechanism_rank"] = (
            mech.groupby("prototype_function").cumcount() + 1
        )
        mech = mech[mech["mechanism_rank"] <= TOP_MECHANISMS_PER_FUNCTION].copy()
    else:
        mech = pd.DataFrame(columns=[
            "prototype_function","mechanism_clean","frequency",
            "patent_families","example_sentence","mechanism_score","mechanism_rank"
        ])

    mech.to_parquet(FUNCTION_MECHANISMS, index=False)
    save_table(mech, "09_selected_function_mechanisms", print_full=True)

    fig, ax = plt.subplots(figsize=(11, 6.5))
    plot = reps.sort_values("function_score")
    ax.barh(plot["prototype_function"], plot["function_score"])
    ax.set_title("Representative normalized disassembly functions")
    ax.set_xlabel("Function score")
    text = (
        f"{len(prot):,} normalized Action–Object function prototypes were scored; "
        f"top {len(reps)} selected. Mechanisms are no longer embedded inside the function label.\n\n"
        "Selected functions:\n"
        + reps[[
            "function_rank","prototype_function",
            "normalized_patent_families","target_similarity","function_score"
        ]].to_string(index=False)
        + "\n\nTop mechanisms by selected function:\n"
        + (mech[[
            "prototype_function","mechanism_rank","mechanism_clean",
            "patent_families","frequency"
        ]].to_string(index=False) if len(mech) else "No mechanisms retained.")
    )
    save_figure(fig, "09_selected_functions", text, reps)

    # Compact mechanism-support figure.
    if len(mech):
        topm = mech[mech["mechanism_rank"] <= 3].copy()
        topm["display"] = (
            topm["prototype_function"].str.slice(0, 38)
            + " → "
            + topm["mechanism_clean"].str.slice(0, 55)
        )
        topm = topm.sort_values("patent_families")
        fig, ax = plt.subplots(figsize=(12, max(7, 0.34*len(topm))))
        ax.barh(topm["display"], topm["patent_families"])
        ax.set_xlabel("Supporting patent families")
        ax.set_title("Top implementation mechanisms for selected functions")
        save_figure(
            fig,
            "09_function_mechanism_support",
            "Top three mechanisms per selected function, ranked by repeated family-level support.\n"
            + topm[[
                "prototype_function","mechanism_rank","mechanism_clean",
                "patent_families","frequency","example_sentence"
            ]].to_string(index=False),
            topm
        )

    body = f"""
The corrected functional stage separates **what the invention does** from **how it does it**.
Action–Object expressions are first quality-filtered and semantically normalized; these normalized functions
form the zero-shot labels. Mechanism/condition phrases are then ranked independently from the traceable source
sentences.

The top **{len(reps)} normalized functions** were retained, with up to
**{TOP_MECHANISMS_PER_FUNCTION} recurrent mechanisms per function** reported separately.
This prevents malformed expressions such as generic “efficiency/tool/operation” objects from becoming
representative functional labels while preserving mechanism-level technical detail for interpretation.
"""
    R.update_section("09_functions", "Functions and mechanisms — corrected separation", body)
    R.emit(body)
    mark_stage_done(stage, required)


# ---------------------------------------------------------------------
# V2 Stage 10 — Recycling Orientation Score (ROS)
# ---------------------------------------------------------------------
def stage_10_circularity_relevance():
    stage = "10_recycling_orientation_v2"
    stage_banner(stage)
    if stage_done(stage, [ROS_PATH]):
        R.emit("V2 checkpoint found — Recycling Orientation Score skipped.")
        return

    df = pd.read_parquet(FAMILY_PARQUET)
    emb = np.asarray(np.load(PATENT_EMB, mmap_mode="r"), dtype=np.float32)
    model = get_embedder()

    recycling_anchors = [
        "end-of-life material recovery for recycling",
        "product recyclability",
        "component separation for recycling",
        "closed-loop material recycling",
        "recovery of recyclable materials from products",
        "end-of-life disassembly for material recovery",
        "sorting and separation of recyclable product materials",
    ]
    service_anchors = SERVICE_ANCHORS

    pos = model.encode(
        recycling_anchors,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    service = model.encode(
        service_anchors,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    pos_sims = emb @ pos.T
    service_sims = emb @ service.T
    k = min(3, pos_sims.shape[1])
    recycling_sim = np.partition(pos_sims, -k, axis=1)[:, -k:].mean(axis=1)
    service_sim = service_sims.max(axis=1)
    ros = recycling_sim - service_sim

    out = df[["doc_idx","family_id","lens_id","title"]].copy()
    out["recycling_similarity"] = recycling_sim
    out["service_repair_similarity"] = service_sim
    out["recycling_orientation_raw"] = ros
    out["recycling_orientation_z"] = safe_z(ros).values
    out.to_parquet(ROS_PATH, index=False)
    save_table(
        out.sort_values("recycling_orientation_raw", ascending=False),
        "10_patent_recycling_orientation",
        print_full=True
    )

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.scatter(
        out["service_repair_similarity"],
        out["recycling_similarity"],
        s=14, alpha=.35
    )
    lo = min(out["service_repair_similarity"].min(), out["recycling_similarity"].min())
    hi = max(out["service_repair_similarity"].max(), out["recycling_similarity"].max())
    ax.plot([lo,hi],[lo,hi], linestyle="--", alpha=.6)
    ax.set_xlabel("Repair / service semantic similarity")
    ax.set_ylabel("End-of-life recycling semantic similarity")
    ax.set_title("Recycling orientation versus service-oriented disassembly")

    positive_share = (out["recycling_orientation_raw"] > 0).mean()
    text = (
        "ROS = mean(top-3 end-of-life recycling anchor similarities) "
        "- max(repair/service anchor similarity). "
        f"{positive_share:.1%} of patent families have ROS>0. "
        f"Mean={out['recycling_orientation_raw'].mean():.4f}; "
        f"median={out['recycling_orientation_raw'].median():.4f}; "
        f"min={out['recycling_orientation_raw'].min():.4f}; "
        f"max={out['recycling_orientation_raw'].max():.4f}."
    )
    save_figure(
        fig,
        "10_recycling_vs_service_orientation",
        text,
        out[[
            "family_id","recycling_similarity",
            "service_repair_similarity","recycling_orientation_raw"
        ]]
    )

    body = f"""
The earlier “Circularity Relevance” label is replaced by the more precise
**Recycling Orientation Score (ROS)** because repair and maintenance are themselves circular strategies.
ROS specifically contrasts semantic alignment to **end-of-life recycling/material recovery** with alignment
to **repair/service disassembly**:

`ROS = mean(top-3 recycling-anchor similarities) − max(service/repair similarity)`.

**{positive_share:.1%}** of patent families have ROS > 0.
ROS remains a descriptive dimension and does not replace TV/TR/TN.
"""
    R.update_section("10_cr", "Recycling Orientation Score (ROS)", body)
    R.emit(body)
    mark_stage_done(stage, [ROS_PATH])


# ---------------------------------------------------------------------
# V2 Stage 11 — zero-shot threshold selected by empirical elbow
# ---------------------------------------------------------------------
def _select_threshold_by_elbow(tuning):
    d = tuning.sort_values("threshold").reset_index(drop=True).copy()
    if len(d) < 3:
        return float(d.iloc[len(d)//2]["threshold"]), d.assign(elbow_distance=0.0)

    x = d["threshold"].to_numpy(dtype=float)
    y = d["assignment_rate"].to_numpy(dtype=float)

    xn = (x - x.min()) / max(x.max() - x.min(), 1e-12)
    yn = (y - y.min()) / max(y.max() - y.min(), 1e-12)

    x1, y1 = xn[0], yn[0]
    x2, y2 = xn[-1], yn[-1]
    denom = max(np.hypot(y2-y1, x2-x1), 1e-12)
    dist = np.abs((y2-y1)*xn - (x2-x1)*yn + x2*y1 - y2*x1) / denom
    dist[0] = 0
    dist[-1] = 0
    d["elbow_distance"] = dist

    admissible = d[
        (d["assignment_rate"] >= ZEROSHOT_MIN_ACCEPTABLE_ASSIGNMENT)
        & (d["assignment_rate"] <= ZEROSHOT_MAX_ACCEPTABLE_ASSIGNMENT)
    ]
    if len(admissible) == 0:
        admissible = d.iloc[1:-1]
    chosen = admissible.sort_values(
        ["elbow_distance","mean_assigned_similarity"],
        ascending=[False,False]
    ).iloc[0]
    return float(chosen["threshold"]), d

def stage_11_zeroshot_assignment():
    stage = "11_zeroshot_assignment_v2"
    stage_banner(stage)
    required = [ZS_LABELS, ZS_ASSIGN, ZS_SIM_MATRIX]
    if stage_done(stage, required):
        R.emit("V2 checkpoint found — zero-shot assignment skipped.")
        return

    df = pd.read_parquet(FAMILY_PARQUET)
    emb = np.asarray(np.load(PATENT_EMB, mmap_mode="r"), dtype=np.float32)
    reps_fields = pd.read_parquet(REP_FIELDS_PATH)
    unique_fields = (
        reps_fields.sort_values("reference_field_score", ascending=False)
        .drop_duplicates("cpc_class")
    )
    if N_REP_REFERENCE_FIELDS:
        unique_fields = unique_fields.head(N_REP_REFERENCE_FIELDS)

    functions = pd.read_parquet(REP_FUNCTIONS)

    labels = []
    lid = 0
    for _, f in unique_fields.iterrows():
        for _, fn in functions.iterrows():
            function_text = fn["prototype_function"]
            labels.append({
                "label_id": lid,
                "cpc_class": f["cpc_class"],
                "reference_field": f["cpc_title"],
                "function_rank": int(fn["function_rank"]),
                "function": function_text,
                "label_text": f"{f['cpc_title']} that {function_text}",
            })
            lid += 1

    labels = pd.DataFrame(labels)
    labels.to_parquet(ZS_LABELS, index=False)
    save_table(labels, "11_zeroshot_labels", print_full=True)

    model = get_embedder()
    label_emb = model.encode(
        labels["label_text"].tolist(),
        batch_size=EMBED_BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    ).astype("float32")

    sims = (emb @ label_emb.T).astype("float32")
    np.save(ZS_SIM_MATRIX, sims)
    best_idx = sims.argmax(axis=1)
    best_sim = sims[np.arange(len(sims)), best_idx]

    tuning_rows = []
    for th in ZEROSHOT_THRESHOLD_GRID:
        mask = best_sim >= th
        counts = Counter(best_idx[mask])
        nonempty = len(counts)
        sizes = np.array(list(counts.values()), dtype=float) if counts else np.array([])
        tuning_rows.append({
            "threshold": th,
            "assigned_documents": int(mask.sum()),
            "assignment_rate": float(mask.mean()),
            "nonempty_labels": nonempty,
            "label_coverage": nonempty / max(len(labels),1),
            "mean_assigned_similarity": float(best_sim[mask].mean()) if mask.any() else np.nan,
            "median_cluster_size": float(np.median(sizes)) if len(sizes) else 0,
            "small_cluster_share_lt10": float((sizes < PRIMARY_MIN_CLUSTER_SIZE).mean()) if len(sizes) else np.nan,
        })

    tuning = pd.DataFrame(tuning_rows)

    if ZEROSHOT_THRESHOLD_OVERRIDE is not None:
        threshold = float(ZEROSHOT_THRESHOLD_OVERRIDE)
        tuning["elbow_distance"] = np.nan
        selection_reason = "manual override"
    else:
        threshold, tuning = _select_threshold_by_elbow(tuning)
        selection_reason = "empirical elbow of assignment-rate curve"

    save_table(tuning, "11_zeroshot_threshold_tuning", print_full=True)

    assigned = best_sim >= threshold
    out = df[["doc_idx","family_id","lens_id","title"]].copy()
    out["zeroshot_label_id"] = np.where(assigned, best_idx, -1)
    out["zeroshot_similarity"] = best_sim
    out = out.merge(
        labels[["label_id","label_text","cpc_class","reference_field","function"]],
        left_on="zeroshot_label_id",
        right_on="label_id",
        how="left"
    )
    out["assigned"] = assigned
    out.to_parquet(ZS_ASSIGN, index=False)
    save_table(out, "11_zeroshot_assignments", print_full=True)

    # Borderline sample to support manual threshold review.
    border = out[
        np.abs(out["zeroshot_similarity"] - threshold) <= ZEROSHOT_BORDERLINE_BAND
    ].copy()
    if len(border) > ZEROSHOT_AUDIT_SAMPLE_SIZE:
        border = border.sample(
            ZEROSHOT_AUDIT_SAMPLE_SIZE,
            random_state=RANDOM_SEED
        )
    save_table(
        border[[
            "doc_idx","lens_id","title","label_text",
            "zeroshot_similarity","assigned"
        ]],
        "11_zeroshot_borderline_audit_sample",
        print_full=True
    )

    assigned_out = out[out["assigned"]].copy()
    heat = pd.crosstab(
        assigned_out["reference_field"],
        assigned_out["function"]
    )
    heat.to_csv(TABLE_DIR / "11_field_function_heatmap_counts.csv")

    fig, ax = plt.subplots(
        figsize=(max(10, 1.3*heat.shape[1]), max(6, .65*heat.shape[0]))
    )
    im = ax.imshow(heat.values, aspect="auto")
    ax.set_xticks(np.arange(heat.shape[1]))
    ax.set_xticklabels(heat.columns, rotation=55, ha="right", fontsize=8)
    ax.set_yticks(np.arange(heat.shape[0]))
    ax.set_yticklabels(heat.index, fontsize=8)
    ax.set_xlabel("Normalized function")
    ax.set_ylabel("Reference field")
    ax.set_title("Reference Field × Function patent-family counts")
    fig.colorbar(im, ax=ax, label="Patent families")

    cells = []
    for i, rr in enumerate(heat.index):
        for j, cc in enumerate(heat.columns):
            cells.append((int(heat.iloc[i,j]), rr, cc))
    cells.sort(reverse=True, key=lambda x: x[0])
    top_cells = "\n".join(
        f"{n:>6} | {rr} | {cc}" for n, rr, cc in cells[:20]
    )

    summary = (
        f"Zero-shot threshold={threshold:.2f} ({selection_reason}); "
        f"assigned={assigned.sum():,}/{len(out):,} ({assigned.mean():.1%}); "
        f"nonempty labels={assigned_out['zeroshot_label_id'].nunique()}/{len(labels)}.\n"
        f"Top field-function cells:\n{top_cells}"
    )
    save_figure(fig, "11_field_function_heatmap", summary)

    # Explicit threshold sensitivity figure.
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(
        tuning["threshold"], tuning["assignment_rate"],
        marker="o", label="Assignment rate"
    )
    ax.plot(
        tuning["threshold"], tuning["label_coverage"],
        marker="s", label="Label coverage"
    )
    ax.plot(
        tuning["threshold"], tuning["mean_assigned_similarity"],
        marker="^", label="Mean assigned similarity"
    )
    ax.axvline(threshold, linestyle="--", label=f"Selected = {threshold:.2f}")
    ax.set_xlabel("Zero-shot similarity threshold")
    ax.set_ylabel("Rate / similarity")
    ax.set_title("Zero-shot threshold sensitivity")
    ax.legend()
    ax.grid(alpha=.25)

    save_figure(
        fig,
        "11_zeroshot_threshold_sensitivity",
        f"Threshold selected by {selection_reason}, not by matching another study's assignment percentage.\n"
        + tuning.to_string(index=False),
        tuning
    )

    body = f"""
The corrected zero-shot cutoff is **not calibrated to reproduce the source case study's 73.4% assignment rate**.
Instead, the threshold is selected from the **empirical elbow of the assignment-rate curve** subject to a broad
acceptable-retention window ({ZEROSHOT_MIN_ACCEPTABLE_ASSIGNMENT:.0%}–{ZEROSHOT_MAX_ACCEPTABLE_ASSIGNMENT:.0%}).
The selected threshold is **{threshold:.2f}**, assigning
**{assigned.sum():,}/{len(out):,} documents ({assigned.mean():.1%})** to
**{assigned_out['zeroshot_label_id'].nunique()} nonempty labels**.

A borderline audit sample within ±{ZEROSHOT_BORDERLINE_BAND:.2f} similarity of the cutoff is saved for
manual review and reporting.
"""
    R.update_section("11_zeroshot", "Semi-supervised zero-shot clustering — corrected threshold", body)
    R.emit(body)
    mark_stage_done(stage, required, extra={
        "threshold": threshold,
        "selection_reason": selection_reason,
    })


# ---------------------------------------------------------------------
# V2 Stage 12 — TN corpus focused on end-of-life recycling
# ---------------------------------------------------------------------
def stage_12_field_sentence_corpus():
    stage = "12_field_sentence_corpus_v2"
    stage_banner(stage)
    required = [FIELD_SENTENCES, FIELD_SENT_EMB]
    if stage_done(stage, required):
        R.emit("V2 checkpoint found — field sentence corpus skipped.")
        return

    df = pd.read_parquet(FAMILY_PARQUET)
    pattern = re.compile(
        "|".join(re.escape(x) for x in FIELD_TERMS),
        flags=re.I
    )

    rows = []
    for _, r in tqdm(
        df.iterrows(),
        total=len(df),
        desc="End-of-life recycling sentence extraction"
    ):
        for sent in sentence_split_simple(r["document_text"]):
            if pattern.search(sent):
                rows.append({
                    "doc_idx": int(r["doc_idx"]),
                    "family_id": r["family_id"],
                    "sentence": sent,
                    "publication_date": r.get(
                        "family_earliest_publication_date",
                        r.get("publication_date")
                    ),
                })

    sents = pd.DataFrame(rows).drop_duplicates(
        ["family_id","sentence"]
    ).reset_index(drop=True)

    if len(sents) == 0:
        raise RuntimeError(
            "No end-of-life recycling field sentences found; inspect FIELD_TERMS."
        )

    sents.to_parquet(FIELD_SENTENCES, index=False)
    save_table(sents, "12_field_interest_sentences", print_full=True)

    path = encode_texts_sharded(
        sents["sentence"].tolist(),
        "field_sentence_embeddings_v2",
        shard_size=5000
    )
    if path != FIELD_SENT_EMB:
        shutil.copy2(path, FIELD_SENT_EMB)

    body = f"""
For TN, **{len(sents):,} sentence-level observations** explicitly associated with the
**end-of-life recycling/product-recyclability field** were extracted and embedded.
This keeps the field-of-interest evidence distinct from the target technology
(**design for disassembly / component separation**).
"""
    R.update_section("12_tn_corpus", "Field-of-interest sentence corpus — corrected separation", body)
    R.emit(body)
    mark_stage_done(stage, required)


# ---------------------------------------------------------------------
# V2 Stage 13 — corrected TV formula + ROS + cluster-size sensitivity
# ---------------------------------------------------------------------
def stage_13_cluster_metrics():
    stage = "13_cluster_metrics_v2"
    stage_banner(stage)
    if stage_done(stage, [CLUSTER_METRICS]):
        R.emit("V2 checkpoint found — cluster evaluation skipped.")
        return

    df = pd.read_parquet(FAMILY_PARQUET)
    zs = pd.read_parquet(ZS_ASSIGN)
    ros = pd.read_parquet(ROS_PATH)
    emb = np.asarray(np.load(PATENT_EMB, mmap_mode="r"), dtype=np.float32)
    sents = pd.read_parquet(FIELD_SENTENCES)
    sent_emb = np.asarray(np.load(FIELD_SENT_EMB, mmap_mode="r"), dtype=np.float32)
    labels = pd.read_parquet(ZS_LABELS)

    work = (
        df.merge(
            zs[[
                "doc_idx","zeroshot_label_id",
                "zeroshot_similarity","assigned"
            ]],
            on="doc_idx"
        )
        .merge(
            ros[["doc_idx","recycling_orientation_raw"]],
            on="doc_idx"
        )
    )
    work = work[work["assigned"]].copy()

    analysis_dt = pd.Timestamp(ANALYSIS_DATE)
    pub = pd.to_datetime(
        work["family_earliest_publication_date"],
        errors="coerce"
    )
    work["citation_age_years"] = (
        (analysis_dt - pub).dt.days / 365.25
    ).clip(lower=1.0).fillna(1.0)

    work["cited_by_patent_count"] = pd.to_numeric(
        work["cited_by_patent_count"],
        errors="coerce"
    ).fillna(0)
    work["simple_family_size"] = pd.to_numeric(
        work["simple_family_size"],
        errors="coerce"
    ).fillna(1).clip(lower=1)

    cluster_ids = sorted(
        work["zeroshot_label_id"].unique().astype(int)
    )

    centroids = {}
    for cid in cluster_ids:
        ids = work.loc[
            work["zeroshot_label_id"] == cid,
            "doc_idx"
        ].astype(int).values
        centroids[cid] = l2_normalize(
            emb[ids].mean(axis=0, keepdims=True)
        )[0]

    model = get_embedder()
    target_vec = model.encode(
        [TARGET_TECHNOLOGY],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0].astype("float32")

    # IMPORTANT CORRECTION:
    # First compute cluster medians. Then z-score across clusters.
    # Only after z-scoring multiply each component by (1 + log(P)).
    rows = []
    for cid, grp in work.groupby("zeroshot_label_id"):
        cid = int(cid)
        P = len(grp)
        citation_rate = (
            grp["cited_by_patent_count"]
            / grp["citation_age_years"]
        )
        rows.append({
            "cluster_id": cid,
            "cluster_size": int(P),
            "size_factor": float(1 + math.log(max(P, 1))),
            "median_age_normalized_forward_citations": float(np.median(citation_rate)),
            "median_simple_family_size": float(np.median(grp["simple_family_size"])),
            "TR_raw_similarity": float(np.dot(centroids[cid], target_vec)),
            "ROS_raw_mean": float(grp["recycling_orientation_raw"].mean()),
        })

    metrics = pd.DataFrame(rows)

    metrics["citation_median_z"] = safe_z(
        metrics["median_age_normalized_forward_citations"]
    ).values
    metrics["family_median_z"] = safe_z(
        metrics["median_simple_family_size"]
    ).values

    metrics["citation_component"] = (
        metrics["citation_median_z"]
        * metrics["size_factor"]
    )
    metrics["family_component"] = (
        metrics["family_median_z"]
        * metrics["size_factor"]
    )
    metrics["TV"] = (
        metrics["citation_component"]
        + metrics["family_component"]
    ) / 2

    metrics["TR"] = safe_z(metrics["TR_raw_similarity"]).values
    metrics["ROS"] = safe_z(metrics["ROS_raw_mean"]).values

    # TN
    C = np.stack([
        centroids[cid] for cid in cluster_ids
    ]).astype("float32")
    sim_matrix = C @ sent_emb.T
    np.save(
        EMBED_DIR / "13_tn_cluster_sentence_similarity_v2.npy",
        sim_matrix.astype("float32")
    )
    q3 = float(np.quantile(sim_matrix, 0.75))

    sent_pub = pd.to_datetime(
        sents["publication_date"],
        errors="coerce"
    )
    sent_age = (
        (analysis_dt - sent_pub).dt.days / 365.25
    ).clip(lower=1.0).fillna(1.0).values
    weights = 1.0 / sent_age

    tn_rows = []
    for i, cid in enumerate(cluster_ids):
        mask = sim_matrix[i] >= q3
        tn_rows.append({
            "cluster_id": cid,
            "TN_overlap_raw": float(weights[mask].sum()),
            "TN_matching_sentences": int(mask.sum())
        })
    tn = pd.DataFrame(tn_rows)
    tn["TN"] = -safe_z(tn["TN_overlap_raw"]).values

    metrics = metrics.merge(tn, on="cluster_id", how="left")
    metrics = metrics.merge(
        labels[[
            "label_id","label_text","cpc_class",
            "reference_field","function"
        ]],
        left_on="cluster_id",
        right_on="label_id",
        how="left"
    )

    metrics["key_cluster_all"] = (
        (metrics["TV"] > 0)
        & (metrics["TR"] > 0)
    )
    metrics["primary_key_cluster"] = (
        metrics["key_cluster_all"]
        & (metrics["cluster_size"] >= PRIMARY_MIN_CLUSTER_SIZE)
    )
    # Compatibility alias: downstream "key_cluster" now means primary article set.
    metrics["key_cluster"] = metrics["primary_key_cluster"]

    metrics["TV_plus_TR"] = metrics["TV"] + metrics["TR"]
    metrics["recycling_priority_supplement"] = (
        metrics["TV"] + metrics["TR"] + metrics["ROS"]
    )
    metrics["novelty_orientation"] = np.where(
        metrics["TN"] >= 0,
        "exploration-oriented",
        "exploitation-oriented"
    )
    metrics["recycling_orientation"] = np.where(
        metrics["ROS"] >= 0,
        "end-of-life-recycling-oriented",
        "repair/service-leaning"
    )
    metrics["cluster_support_class"] = np.where(
        metrics["cluster_size"] >= PRIMARY_MIN_CLUSTER_SIZE,
        "primary-support",
        "low-support exploratory"
    )

    metrics = metrics.sort_values(
        ["primary_key_cluster","key_cluster_all","TV_plus_TR"],
        ascending=[False,False,False]
    ).reset_index(drop=True)

    metrics.to_parquet(CLUSTER_METRICS, index=False)
    save_table(
        metrics,
        "13_cluster_metrics_TV_TR_TN_ROS_corrected",
        print_full=True
    )

    sensitivity = pd.DataFrame([
        {
            "screen": "TV>0 and TR>0, all cluster sizes",
            "clusters": int(metrics["key_cluster_all"].sum())
        },
        {
            "screen": f"TV>0 and TR>0 and n>={PRIMARY_MIN_CLUSTER_SIZE}",
            "clusters": int(metrics["primary_key_cluster"].sum())
        },
        {
            "screen": f"All evaluated clusters with n<{PRIMARY_MIN_CLUSTER_SIZE}",
            "clusters": int((metrics["cluster_size"] < PRIMARY_MIN_CLUSTER_SIZE).sum())
        }
    ])
    save_table(
        sensitivity,
        "13_cluster_size_sensitivity",
        print_full=True
    )

    # Main scatter
    fig, ax = plt.subplots(figsize=(10, 8))
    sc = ax.scatter(
        metrics["TR"],
        metrics["TV"],
        c=metrics["TN"],
        s=40 + 4*np.sqrt(metrics["cluster_size"]),
        alpha=.72
    )
    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)
    ax.set_xlabel("TR (z-score)")
    ax.set_ylabel("TV (corrected score)")
    ax.set_title("Technology-opportunity clusters: corrected TV × TR (color = TN)")
    fig.colorbar(sc, ax=ax, label="TN")

    primary = metrics[metrics["primary_key_cluster"]]
    for _, r in primary.head(15).iterrows():
        ax.annotate(
            str(int(r["cluster_id"])),
            (r["TR"], r["TV"]),
            fontsize=8
        )

    q_counts = {
        "High TV / High TR": int(((metrics.TV>0)&(metrics.TR>0)).sum()),
        "High TV / Low TR": int(((metrics.TV>0)&(metrics.TR<=0)).sum()),
        "Low TV / High TR": int(((metrics.TV<=0)&(metrics.TR>0)).sum()),
        "Low TV / Low TR": int(((metrics.TV<=0)&(metrics.TR<=0)).sum()),
    }

    text = (
        f"Evaluated {len(metrics)} zero-shot clusters. TN global Q3={q3:.4f}. "
        f"Quadrant counts={q_counts}. High-TV/high-TR clusters regardless of size="
        f"{int(metrics['key_cluster_all'].sum())}; article-primary clusters with "
        f"n>={PRIMARY_MIN_CLUSTER_SIZE}={len(primary)}.\n\n"
        "Primary clusters ranked by TV+TR:\n"
        + primary[[
            "cluster_id","label_text","cluster_size",
            "TV","TR","TN","ROS","TV_plus_TR"
        ]].head(20).to_string(index=False)
    )
    save_figure(
        fig,
        "13_TV_TR_scatter_corrected",
        text,
        metrics
    )

    # Primary-cluster radar
    top = primary.head(5).copy()
    if len(top):
        dims = ["TV","TR","TN","ROS"]
        norm = top[["cluster_id","label_text"] + dims].copy()
        for c in dims:
            lo, hi = metrics[c].min(), metrics[c].max()
            norm[c] = (
                (top[c]-lo)/(hi-lo)
                if hi > lo else 0.5
            )

        angles = np.linspace(
            0, 2*np.pi, len(dims), endpoint=False
        ).tolist()
        angles += angles[:1]

        fig = plt.figure(figsize=(9, 8))
        ax = plt.subplot(111, polar=True)
        for _, r in norm.iterrows():
            vals = [r[c] for c in dims] + [r[dims[0]]]
            ax.plot(
                angles, vals,
                linewidth=2,
                label=f"Cluster {int(r['cluster_id'])}"
            )
            ax.fill(angles, vals, alpha=.08)
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(dims)
        ax.set_ylim(0,1)
        ax.set_title("Top primary clusters across TV, TR, TN and ROS")
        ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1))

        save_figure(
            fig,
            "13_primary_key_cluster_radar_corrected",
            "Radar values are min-max normalized across all evaluated clusters. Raw values:\n"
            + top[[
                "cluster_id","label_text","TV","TR","TN","ROS","cluster_size"
            ]].to_string(index=False),
            top
        )

    body = f"""
**TV was corrected to match the source-paper operation order.**
For each cluster, median age-normalized forward citations and median simple-family size are calculated first.
Each median is then **z-score normalized across clusters**, and only then multiplied by
`(1 + log(cluster size))`; TV is the mean of those two weighted standardized components.

TR uses the disassembly-specific target vector. TN compares cluster centroids with the end-of-life recycling
sentence corpus; its global Q3 threshold is **{q3:.4f}**. ROS is reported independently.

The source-style high-TV/high-TR screen identifies
**{int(metrics['key_cluster_all'].sum())} clusters**. For manuscript-primary ranking, clusters with fewer than
**{PRIMARY_MIN_CLUSTER_SIZE} patent families** are retained as exploratory evidence but not treated as primary;
this yields **{len(primary)} primary key clusters**.
"""
    R.update_section("13_metrics", "Technology-opportunity cluster evaluation — corrected", body)
    R.emit(body)
    mark_stage_done(stage, [CLUSTER_METRICS], extra={
        "tn_q3": q3,
        "key_clusters_all_sizes": int(metrics["key_cluster_all"].sum()),
        "primary_key_clusters": len(primary),
        "primary_min_cluster_size": PRIMARY_MIN_CLUSTER_SIZE,
    })


# ---------------------------------------------------------------------
# V2 Stage 14 — key CPCs, functions, mechanisms, patents, sentences
# ---------------------------------------------------------------------
def stage_14_key_cluster_analysis():
    stage = "14_key_cluster_analysis_v2"
    stage_banner(stage)
    required = [KEY_PATENTS_PATH, KEY_SENTENCES_PATH, KEY_NODES_PATH]
    if stage_done(stage, required):
        R.emit("V2 checkpoint found — key-cluster analysis skipped.")
        return

    df = pd.read_parquet(FAMILY_PARQUET)
    zs = pd.read_parquet(ZS_ASSIGN)
    metrics = pd.read_parquet(CLUSTER_METRICS)
    rawf = pd.read_parquet(RAW_FUNCTIONS)
    fmap = pd.read_parquet(FUNCTION_MAP)

    rawf = rawf.merge(
        fmap[["action_object","prototype_function"]].drop_duplicates(),
        on="action_object",
        how="left"
    )
    rawf["normalized_function"] = rawf["prototype_function"]
    rawf["mechanism_clean"] = rawf["mechanism"].map(_clean_mechanism_text)

    work = df.merge(
        zs[["doc_idx","zeroshot_label_id","assigned"]],
        on="doc_idx"
    )
    work = work[work["assigned"]].copy()

    if ANALYZE_PRIMARY_KEY_CLUSTERS_ONLY:
        keys = metrics[metrics["primary_key_cluster"]].sort_values(
            "TV_plus_TR", ascending=False
        )
    else:
        keys = metrics[metrics["key_cluster_all"]].sort_values(
            "TV_plus_TR", ascending=False
        )

    all_nodes = []
    all_key_patents = []
    all_key_sentences = []

    for _, kr in keys.iterrows():
        cid = int(kr["cluster_id"])
        cdir = NETWORK_DIR_V2 / f"cluster_{cid:03d}"
        cdir.mkdir(parents=True, exist_ok=True)
        done_marker = cdir / "DONE.json"

        if done_marker.exists():
            R.emit(f"[V2 cluster {cid}] cached.")
            for fn, collector in [
                ("nodes.parquet", all_nodes),
                ("key_patents.parquet", all_key_patents),
                ("key_sentences.parquet", all_key_sentences)
            ]:
                p = cdir / fn
                if p.exists():
                    collector.append(pd.read_parquet(p))
            continue

        cluster_docs = work[
            work["zeroshot_label_id"] == cid
        ].copy()
        doc_ids = set(cluster_docs["doc_idx"].astype(int))

        # CPC subclasses
        cpc_by_doc = {}
        for _, r in cluster_docs.iterrows():
            subclasses = sorted(set(filter(
                None,
                (cpc_subclass(x) for x in parse_cpc_codes(r["cpc_classifications"]))
            )))
            cpc_by_doc[int(r["doc_idx"])] = subclasses

        # Normalized functions and mechanisms
        fsub = rawf[
            rawf["doc_idx"].isin(doc_ids)
            & rawf["normalized_function"].notna()
        ].copy()

        func_by_doc = (
            fsub.groupby("doc_idx")["normalized_function"]
            .apply(lambda s: sorted(set(s)))
            .to_dict()
        )
        mech_by_doc = (
            fsub[fsub["mechanism_clean"].str.len() > 0]
            .groupby("doc_idx")["mechanism_clean"]
            .apply(lambda s: sorted(set(s)))
            .to_dict()
        )

        cpc_G = build_cooccurrence([
            cpc_by_doc.get(i,[]) for i in doc_ids
        ])
        func_G = build_cooccurrence([
            func_by_doc.get(i,[]) for i in doc_ids
        ])
        mech_G = build_cooccurrence([
            mech_by_doc.get(i,[]) for i in doc_ids
        ])

        cpc_nodes, cpc_edges = network_tables(
            cpc_G, cid, "CPC_subclass"
        )
        fn_nodes, fn_edges = network_tables(
            func_G, cid, "Function"
        )
        mech_nodes, mech_edges = network_tables(
            mech_G, cid, "Mechanism"
        )

        nodes = pd.concat(
            [cpc_nodes, fn_nodes, mech_nodes],
            ignore_index=True
        )
        edges = pd.concat(
            [cpc_edges, fn_edges, mech_edges],
            ignore_index=True
        )
        nodes.to_parquet(cdir / "nodes.parquet", index=False)
        edges.to_parquet(cdir / "edges.parquet", index=False)
        all_nodes.append(nodes)

        # Key-patent score remains based on CPC + FUNCTION centralities
        # to preserve comparability with the source logic.
        cpc_cent = (
            cpc_nodes.set_index("node")["centrality"]
            if len(cpc_nodes) else pd.Series(dtype=float)
        )
        fn_cent = (
            fn_nodes.set_index("node")["centrality"]
            if len(fn_nodes) else pd.Series(dtype=float)
        )
        cpc_z = dict(zip(
            cpc_cent.index,
            safe_z(cpc_cent.values)
        ))
        fn_z = dict(zip(
            fn_cent.index,
            safe_z(fn_cent.values)
        ))

        p_rows = []
        for _, r in cluster_docs.iterrows():
            did = int(r["doc_idx"])
            cpcs = cpc_by_doc.get(did, [])
            funcs = func_by_doc.get(did, [])
            mechs = mech_by_doc.get(did, [])

            score = (
                sum(cpc_z.get(x,0) for x in cpcs)
                + sum(fn_z.get(x,0) for x in funcs)
            )

            p_rows.append({
                "cluster_id": cid,
                "doc_idx": did,
                "family_id": r["family_id"],
                "lens_id": r.get("lens_id"),
                "title": r.get("title"),
                "applicants": r.get("applicants"),
                "owners": r.get("owners"),
                "publication_date": r.get("publication_date"),
                "patent_centrality_score": float(score),
                "cpc_subclasses": " ; ".join(cpcs),
                "normalized_functions": " ; ".join(funcs),
                "mechanisms": " ; ".join(mechs),
            })

        ptab = pd.DataFrame(p_rows).sort_values(
            "patent_centrality_score",
            ascending=False
        )
        keyp = ptab.head(KEY_PATENTS_PER_CLUSTER).copy()
        keyp.to_parquet(cdir / "key_patents.parquet", index=False)
        all_key_patents.append(keyp)

        top_funcs = (
            set(fn_nodes.head(5)["node"])
            if len(fn_nodes) else set()
        )

        ks_rows = []
        for _, p in keyp.iterrows():
            cand = fsub[
                (fsub["doc_idx"] == int(p["doc_idx"]))
                & (fsub["normalized_function"].isin(top_funcs))
            ].drop_duplicates("sentence")

            if len(cand) == 0:
                cand = fsub[
                    fsub["doc_idx"] == int(p["doc_idx"])
                ].drop_duplicates("sentence")

            for _, rr in cand.head(
                KEY_SENTENCES_PER_PATENT
            ).iterrows():
                ks_rows.append({
                    "cluster_id": cid,
                    "doc_idx": int(p["doc_idx"]),
                    "family_id": p["family_id"],
                    "lens_id": p["lens_id"],
                    "title": p["title"],
                    "key_function": rr["normalized_function"],
                    "mechanism": rr["mechanism_clean"],
                    "sentence": rr["sentence"],
                    "patent_centrality_score": p["patent_centrality_score"],
                })

        ksent = pd.DataFrame(ks_rows)
        ksent.to_parquet(cdir / "key_sentences.parquet", index=False)
        all_key_sentences.append(ksent)

        plot_network(
            cpc_G, cpc_nodes,
            f"14_cluster_{cid:03d}_cpc_network_corrected",
            f"Cluster {cid}: CPC-subclass co-occurrence"
        )
        plot_network(
            func_G, fn_nodes,
            f"14_cluster_{cid:03d}_function_network_corrected",
            f"Cluster {cid}: normalized function co-occurrence"
        )

        # Mechanisms are easier to read as ranked bars than long-label networks.
        if len(mech_nodes):
            topm = mech_nodes.head(12).sort_values("centrality")
            fig, ax = plt.subplots(
                figsize=(11, max(6, .42*len(topm)))
            )
            ax.barh(topm["node"], topm["centrality"])
            ax.set_xlabel("Mechanism centrality")
            ax.set_title(f"Cluster {cid}: key implementation mechanisms")
            save_figure(
                fig,
                f"14_cluster_{cid:03d}_mechanisms_corrected",
                topm[[
                    "node","frequency","centrality",
                    "centrality_method"
                ]].to_string(index=False),
                topm
            )

        summary = {
            "cluster_id": cid,
            "label": kr["label_text"],
            "n_patent_families": len(cluster_docs),
            "TV": kr["TV"],
            "TR": kr["TR"],
            "TN": kr["TN"],
            "ROS": kr["ROS"],
            "top_cpc_nodes": cpc_nodes.head(5).to_dict("records"),
            "top_function_nodes": fn_nodes.head(5).to_dict("records"),
            "top_mechanism_nodes": mech_nodes.head(8).to_dict("records"),
            "key_patents": keyp.head(5)[[
                "lens_id","title","patent_centrality_score"
            ]].to_dict("records"),
        }
        atomic_write_json(done_marker, summary)
        R.emit(json.dumps(
            summary,
            indent=2,
            ensure_ascii=False,
            default=str
        ))

    nodes_all = (
        pd.concat(all_nodes, ignore_index=True)
        if all_nodes else pd.DataFrame()
    )
    kp_all = (
        pd.concat(all_key_patents, ignore_index=True)
        if all_key_patents else pd.DataFrame()
    )
    ks_all = (
        pd.concat(all_key_sentences, ignore_index=True)
        if all_key_sentences else pd.DataFrame()
    )

    nodes_all.to_parquet(KEY_NODES_PATH, index=False)
    kp_all.to_parquet(KEY_PATENTS_PATH, index=False)
    ks_all.to_parquet(KEY_SENTENCES_PATH, index=False)

    save_table(
        nodes_all,
        "14_all_key_cluster_nodes_corrected",
        print_full=True
    )
    save_table(
        kp_all,
        "14_all_key_patents_corrected",
        print_full=True
    )
    save_table(
        ks_all,
        "14_all_key_sentences_corrected",
        print_full=True
    )

    body = f"""
The corrected key-cluster analysis explicitly distinguishes three evidence layers:
**CPC subclasses, normalized functions, and implementation mechanisms**.
CPC/function eigenvector centralities retain the source-style key-patent scoring logic, while mechanisms are
reported independently for technical interpretation.

The primary analysis produced **{len(kp_all):,} key-patent records** and
**{len(ks_all):,} traceable sentence-level evidence records** across
**{len(keys)} primary key clusters**.
"""
    R.update_section("14_networks", "Key-cluster evidence — corrected function/mechanism separation", body)
    R.emit(body)
    mark_stage_done(stage, required)


# ---------------------------------------------------------------------
# V2 Stage 15 — prompts and validation references use corrected metrics
# ---------------------------------------------------------------------
def stage_15_validation_and_llm_prompts():
    stage = "15_validation_llm_prompts_v2"
    stage_banner(stage)
    marker = CHECKPOINT_DIR / "15_validation_llm_prompts_v2.done"
    if stage_done(stage, [marker, LLM_PROMPT_INDEX]):
        R.emit("V2 checkpoint found — validation/LLM prompt package skipped.")
        return

    metrics = pd.read_parquet(CLUSTER_METRICS)
    keysent = pd.read_parquet(KEY_SENTENCES_PATH)
    nodes = pd.read_parquet(KEY_NODES_PATH)

    validation = pd.DataFrame([
        {
            "validation_component": "Historical data-based validation",
            "status": "Not executed automatically",
            "reason": (
                "The export contains cumulative forward-citation counts but no citation-event dates. "
                "A leakage-free historical TV reconstruction requires dated citation events or citing-patent dates."
            )
        },
        {
            "validation_component": "Zero-shot threshold review",
            "status": "Prepared",
            "reason": (
                "The threshold is selected from the empirical assignment elbow, and a borderline patent sample "
                "is saved for manual relevance review."
            )
        },
        {
            "validation_component": "Cluster-size sensitivity",
            "status": "Executed",
            "reason": (
                f"All high-TV/high-TR clusters are retained, while the article-primary ranking requires "
                f"at least {PRIMARY_MIN_CLUSTER_SIZE} patent families."
            )
        },
        {
            "validation_component": "Expert validation",
            "status": "Requires human experts",
            "reason": (
                "Experts should assess technical transferability, novelty, applicability, and the usefulness "
                "of key functions, mechanisms, patents, and sentences."
            )
        },
    ])
    save_table(validation, "15_validation_readiness", print_full=True)

    keys = (
        metrics[metrics["primary_key_cluster"]]
        .sort_values("TV_plus_TR", ascending=False)
        .head(N_LLM_CLUSTERS)
    )

    prompt_rows = []
    for _, r in keys.iterrows():
        cid = int(r["cluster_id"])
        ref_nodes = nodes[
            (nodes["cluster_id"] == cid)
            & (nodes["network_type"] == "CPC_subclass")
        ].head(5)
        fn_nodes = nodes[
            (nodes["cluster_id"] == cid)
            & (nodes["network_type"] == "Function")
        ].head(5)
        mech_nodes = nodes[
            (nodes["cluster_id"] == cid)
            & (nodes["network_type"] == "Mechanism")
        ].head(8)
        sent = keysent[
            keysent["cluster_id"] == cid
        ].head(15)

        prompt = f"""
You are a technology innovation strategist supporting end-of-life recycling and product-recyclability R&D.

TASK
Generate exactly {N_SOLUTIONS_PER_CLUSTER} technically distinct solution concepts that:
1. Address the target technology challenge: {TARGET_TECHNOLOGY}
2. Transfer and recombine principles from the identified cross-field patent cluster.
3. Use the extracted normalized functions and implementation mechanisms.
4. Improve end-of-life recycling/recyclability rather than merely copying prior art.
5. Preserve traceability to the supplied patent evidence.

FIELD OF INTEREST
{FIELD_OF_INTEREST}

CLUSTER
ID: {cid}
Label: {r['label_text']}
TV={r['TV']:.4f}; TR={r['TR']:.4f}; TN={r['TN']:.4f}; ROS={r['ROS']:.4f}

KEY CPC SUBCLASSES
{ref_nodes[['node','frequency','centrality']].to_string(index=False) if len(ref_nodes) else 'None'}

KEY FUNCTIONS
{fn_nodes[['node','frequency','centrality']].to_string(index=False) if len(fn_nodes) else 'None'}

KEY IMPLEMENTATION MECHANISMS
{mech_nodes[['node','frequency','centrality']].to_string(index=False) if len(mech_nodes) else 'None'}

TRACEABLE KEY PATENT SENTENCES
{sent[['lens_id','key_function','mechanism','sentence']].to_string(index=False) if len(sent) else 'None'}

OUTPUT FOR EACH SOLUTION
- Solution number
- Title
- Core innovation
- Cross-field integration
- Key functions applied
- Mechanisms adapted
- Technical feasibility (High/Medium/Low + justification)
- End-of-life recycling/recyclability contribution
- Competitive differentiation from supplied prior art
- Implementation pathway (3–4 steps)
- Potential technical challenges
- Evidence references (Lens IDs / sentences used)

Do not claim freedom-to-operate or legal clearance.
""".strip()

        p = LLM_DIR / f"cluster_{cid:03d}_prompt_v2.txt"
        atomic_write_text(p, prompt)
        R.emit(
            f"\n===== CORRECTED LLM PROMPT FOR CLUSTER {cid} =====\n{prompt}\n"
        )
        prompt_rows.append({
            "cluster_id": cid,
            "label_text": r["label_text"],
            "prompt_path": str(p),
            "TV": r["TV"],
            "TR": r["TR"],
            "TN": r["TN"],
            "ROS": r["ROS"],
        })

    prompt_index = pd.DataFrame(prompt_rows)
    prompt_index.to_parquet(LLM_PROMPT_INDEX, index=False)
    save_table(
        prompt_index,
        "15_llm_prompt_index",
        print_full=True
    )

    body = f"""
The corrected validation package records the empirical zero-shot threshold audit and the cluster-size sensitivity
rule. Historical predictive validation is still not claimed without dated citation events. Evidence-grounded
solution-concept prompts were prepared for the top **{len(prompt_index)} primary key clusters**, now separating
functions from mechanisms and reporting ROS rather than the broader “circularity relevance” label.
"""
    R.update_section("15_validation_llm", "Validation readiness and solution-concept prompts — corrected", body)
    R.emit(body)

    marker.write_text("done", encoding="utf-8")
    mark_stage_done(stage, [marker, LLM_PROMPT_INDEX])


# %% CELL: FINAL_V3_MANUSCRIPT_READY
# ============================================================
# FINAL V3 — MANUSCRIPT-READY ANALYSIS
#
# Final methodological decisions:
#   A) Zero-shot threshold fixed at 0.45 after the completed sensitivity run:
#      0.40 retained too many weak assignments, while 0.50 was overly restrictive.
#      V3 still recomputes and prints the full threshold-sensitivity diagnostics,
#      including top-1/top-2 semantic margins and a borderline audit sample.
#
#   B) Key patents MUST contain the cluster's normalized representative function.
#      A patent can no longer become "key" solely because it carries many central CPCs.
#
#   C) Key-patent score uses three balanced components:
#      mean CPC centrality + mean function centrality + zero-shot label fit.
#      Each component is z-normalized within the cluster before averaging.
#
#   D) Mechanisms are split into:
#      - recurrent/supported mechanisms: >= 2 patent families;
#      - singleton case evidence: retained separately, not called dominant.
#
#   E) Primary manuscript opportunity clusters retain the prior rule:
#      TV > 0, TR > 0, and cluster size >= 10.
#
# Every V3 table/figure has a text equivalent and all intermediate artifacts
# remain persisted to Drive.
# ============================================================

PIPELINE_REVISION = "FINAL_V3_MANUSCRIPT_READY_2026_09_14"

FINAL_ZEROSHOT_THRESHOLD = 0.45
FINAL_THRESHOLD_RATIONALE = (
    "Pre-specified after empirical sensitivity inspection: 0.40 yielded excessive assignment "
    "coverage with weak borderline relevance; 0.50 was overly restrictive. 0.45 provides a "
    "midpoint trade-off and is retained transparently rather than optimized to reproduce another study."
)

KEY_PATENT_REQUIRE_CLUSTER_FUNCTION = True
KEY_PATENT_TOP_CPC_N = 3
KEY_PATENT_TOP_FUNCTION_N = 2

MECHANISM_MIN_FAMILY_SUPPORT = 2
GENERIC_MECHANISM_PATTERNS = [
    r"\bmeans? of (the )?design\b",
    r"\baccording to\b",
    r"\bthe present invention\b",
    r"\bthe invention\b",
    r"\bthe device\b",
    r"\bthe system\b",
    r"\bthe method\b",
    r"\bas described\b",
]

# V3-specific paths: no ambiguity with earlier runs.
ZS_LABELS = DATA_DIR / "11_zeroshot_labels_FINAL_V3.parquet"
ZS_ASSIGN = DATA_DIR / "11_zeroshot_assignments_FINAL_V3.parquet"
ZS_SIM_MATRIX = EMBED_DIR / "11_zeroshot_similarity_matrix_FINAL_V3.npy"

CLUSTER_METRICS = DATA_DIR / "13_cluster_metrics_FINAL_V3.parquet"

KEY_PATENTS_PATH = DATA_DIR / "14_key_patents_FINAL_V3.parquet"
KEY_SENTENCES_PATH = DATA_DIR / "14_key_sentences_FINAL_V3.parquet"
KEY_NODES_PATH = DATA_DIR / "14_key_nodes_FINAL_V3.parquet"
KEY_MECHANISMS_SUPPORTED_PATH = DATA_DIR / "14_supported_mechanisms_FINAL_V3.parquet"
KEY_MECHANISMS_SINGLETON_PATH = DATA_DIR / "14_singleton_mechanisms_FINAL_V3.parquet"

LLM_PROMPT_INDEX = DATA_DIR / "15_llm_prompt_index_FINAL_V3.parquet"
FINAL_OPPORTUNITY_TABLE = DATA_DIR / "15_final_opportunity_table_FINAL_V3.parquet"
FINAL_QA_PATH = DATA_DIR / "15_final_QA_FINAL_V3.parquet"

NETWORK_DIR_V3 = NETWORK_DIR / "FINAL_V3"
NETWORK_DIR_V3.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# One-time archive of superseded downstream outputs so final replay is clean.
# ---------------------------------------------------------------------
V3_MIGRATION_MARKER = CHECKPOINT_DIR / "FINAL_V3_MIGRATED.json"

def _archive_superseded_v2_outputs_once():
    if V3_MIGRATION_MARKER.exists():
        return

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    archive = ARCHIVE_DIR / f"pre_FINAL_V3_{stamp}"
    archive.mkdir(parents=True, exist_ok=True)

    # Archive only downstream outputs affected by the final decisions.
    prefixes = ("11_", "13_", "14_", "15_", "16_")

    for root in [TABLE_DIR, TABLE_TEXT_DIR, FIG_DIR]:
        dst = archive / root.name
        dst.mkdir(parents=True, exist_ok=True)
        for p in list(root.glob("*")):
            if p.is_file() and p.name.startswith(prefixes):
                shutil.move(str(p), str(dst / p.name))

    # Archive older downstream data except the expensive/stable sources.
    dst = archive / "data"
    dst.mkdir(parents=True, exist_ok=True)
    for p in list(DATA_DIR.glob("*")):
        if p.is_file() and p.name.startswith(("11_", "13_", "14_", "15_", "16_")):
            shutil.move(str(p), str(dst / p.name))

    # Archive old downstream network directories except FINAL_V3.
    if NETWORK_DIR.exists():
        ndst = archive / "networks"
        ndst.mkdir(parents=True, exist_ok=True)
        for p in list(NETWORK_DIR.iterdir()):
            if p.name == "FINAL_V3":
                continue
            shutil.move(str(p), str(ndst / p.name))

    # Archive old LLM prompt package.
    if LLM_DIR.exists():
        ldst = archive / "llm"
        ldst.mkdir(parents=True, exist_ok=True)
        for p in list(LLM_DIR.iterdir()):
            shutil.move(str(p), str(ldst / p.name))

    atomic_write_json(V3_MIGRATION_MARKER, {
        "revision": PIPELINE_REVISION,
        "migrated_at": datetime.now(timezone.utc).isoformat(),
        "archive": str(archive),
        "note": "Superseded V2/V2.1 downstream outputs archived. Stable preprocessing, embeddings, raw function extraction, CPC fields, functions, ROS, and TN corpus retained."
    })
    R.emit(f"[FINAL V3 migration] Archived superseded downstream outputs -> {archive}")

_archive_superseded_v2_outputs_once()

R.update_section(
    "v3_final_decisions",
    "Final manuscript-ready analytical decisions",
    f"""
The final analysis uses a **pre-specified zero-shot threshold of {FINAL_ZEROSHOT_THRESHOLD:.2f}** following the
completed threshold-sensitivity run. The threshold is not tuned to reproduce the assignment percentage of the
source study. The final key-patent procedure also requires direct evidence of the cluster's normalized function,
preventing CPC-rich but functionally irrelevant patents from being promoted as representative evidence.

Mechanisms supported by at least **{MECHANISM_MIN_FAMILY_SUPPORT} patent families** are reported as recurrent
mechanisms; one-family mechanisms are retained separately as case-specific examples.
"""
)

# ---------------------------------------------------------------------
# FINAL Stage 11 — fixed, transparent threshold + margin diagnostics
# ---------------------------------------------------------------------
def stage_11_zeroshot_assignment():
    stage = "11_zeroshot_assignment_FINAL_V3"
    stage_banner(stage)
    required = [ZS_LABELS, ZS_ASSIGN, ZS_SIM_MATRIX]
    if stage_done(stage, required):
        R.emit("FINAL V3 checkpoint found — zero-shot assignment skipped.")
        return

    df = pd.read_parquet(FAMILY_PARQUET)
    emb = np.asarray(np.load(PATENT_EMB, mmap_mode="r"), dtype=np.float32)

    reps_fields = pd.read_parquet(REP_FIELDS_PATH)
    unique_fields = (
        reps_fields.sort_values("reference_field_score", ascending=False)
        .drop_duplicates("cpc_class")
    )
    if N_REP_REFERENCE_FIELDS:
        unique_fields = unique_fields.head(N_REP_REFERENCE_FIELDS)

    functions = pd.read_parquet(REP_FUNCTIONS)

    labels = []
    lid = 0
    for _, f in unique_fields.iterrows():
        for _, fn in functions.iterrows():
            function_text = fn["prototype_function"]
            labels.append({
                "label_id": lid,
                "cpc_class": f["cpc_class"],
                "reference_field": f["cpc_title"],
                "function_rank": int(fn["function_rank"]),
                "function": function_text,
                "label_text": f"{f['cpc_title']} that {function_text}",
            })
            lid += 1

    labels = pd.DataFrame(labels)
    labels.to_parquet(ZS_LABELS, index=False)
    save_table(labels, "11_zeroshot_labels_FINAL_V3", print_full=True)

    model = get_embedder()
    label_emb = model.encode(
        labels["label_text"].tolist(),
        batch_size=EMBED_BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    ).astype("float32")

    sims = (emb @ label_emb.T).astype("float32")
    np.save(ZS_SIM_MATRIX, sims)

    # Top-1 and top-2 similarities for semantic-margin diagnostics.
    top2_idx_unsorted = np.argpartition(sims, -2, axis=1)[:, -2:]
    top2_vals_unsorted = np.take_along_axis(sims, top2_idx_unsorted, axis=1)
    order = np.argsort(top2_vals_unsorted, axis=1)
    second_pos = order[:, 0]
    best_pos = order[:, 1]

    row_ix = np.arange(len(sims))
    best_idx = top2_idx_unsorted[row_ix, best_pos]
    second_idx = top2_idx_unsorted[row_ix, second_pos]
    best_sim = sims[row_ix, best_idx]
    second_sim = sims[row_ix, second_idx]
    margin = best_sim - second_sim

    tuning_rows = []
    for th in ZEROSHOT_THRESHOLD_GRID:
        mask = best_sim >= th
        counts = Counter(best_idx[mask])
        sizes = np.array(list(counts.values()), dtype=float) if counts else np.array([])
        tuning_rows.append({
            "threshold": float(th),
            "assigned_documents": int(mask.sum()),
            "assignment_rate": float(mask.mean()),
            "nonempty_labels": int(len(counts)),
            "label_coverage": float(len(counts) / max(len(labels), 1)),
            "mean_assigned_similarity": float(best_sim[mask].mean()) if mask.any() else np.nan,
            "median_assigned_similarity": float(np.median(best_sim[mask])) if mask.any() else np.nan,
            "mean_top1_top2_margin": float(margin[mask].mean()) if mask.any() else np.nan,
            "p10_top1_top2_margin": float(np.quantile(margin[mask], .10)) if mask.any() else np.nan,
            "median_cluster_size": float(np.median(sizes)) if len(sizes) else 0,
            "small_cluster_share_lt10": float((sizes < PRIMARY_MIN_CLUSTER_SIZE).mean()) if len(sizes) else np.nan,
        })

    tuning = pd.DataFrame(tuning_rows)
    tuning["selected_final_threshold"] = np.isclose(
        tuning["threshold"], FINAL_ZEROSHOT_THRESHOLD
    )
    save_table(
        tuning,
        "11_zeroshot_threshold_sensitivity_FINAL_V3",
        print_full=True
    )

    threshold = float(FINAL_ZEROSHOT_THRESHOLD)
    assigned = best_sim >= threshold

    out = df[["doc_idx","family_id","lens_id","title"]].copy()
    out["zeroshot_label_id"] = np.where(assigned, best_idx, -1)
    out["zeroshot_similarity"] = best_sim
    out["second_best_similarity"] = second_sim
    out["semantic_margin"] = margin
    out["second_best_label_id"] = second_idx

    out = out.merge(
        labels[[
            "label_id","label_text","cpc_class",
            "reference_field","function"
        ]],
        left_on="zeroshot_label_id",
        right_on="label_id",
        how="left"
    )

    second_label_map = dict(zip(labels["label_id"], labels["label_text"]))
    out["second_best_label_text"] = [
        second_label_map.get(int(x), "")
        for x in second_idx
    ]
    out["assigned"] = assigned
    out.to_parquet(ZS_ASSIGN, index=False)
    save_table(
        out,
        "11_zeroshot_assignments_FINAL_V3",
        print_full=True
    )

    # Borderline audit sample around 0.45: deterministic and balanced above/below threshold.
    band = 0.025
    below = out[
        (out["zeroshot_similarity"] < threshold)
        & (out["zeroshot_similarity"] >= threshold-band)
    ].copy()
    above = out[
        (out["zeroshot_similarity"] >= threshold)
        & (out["zeroshot_similarity"] <= threshold+band)
    ].copy()

    n_half = max(1, ZEROSHOT_AUDIT_SAMPLE_SIZE // 2)
    if len(below) > n_half:
        below = below.sample(n_half, random_state=RANDOM_SEED)
    if len(above) > n_half:
        above = above.sample(n_half, random_state=RANDOM_SEED)

    audit = pd.concat([below, above], ignore_index=True)
    audit["audit_side"] = np.where(
        audit["zeroshot_similarity"] >= threshold,
        "accepted_borderline",
        "rejected_borderline"
    )
    audit = audit.sort_values("zeroshot_similarity")
    save_table(
        audit[[
            "audit_side","doc_idx","lens_id","title","label_text",
            "zeroshot_similarity","second_best_label_text",
            "second_best_similarity","semantic_margin","assigned"
        ]],
        "11_zeroshot_borderline_audit_FINAL_V3",
        print_full=True
    )

    assigned_out = out[out["assigned"]].copy()
    heat = pd.crosstab(
        assigned_out["reference_field"],
        assigned_out["function"]
    )
    heat.to_csv(
        TABLE_DIR / "11_field_function_heatmap_counts_FINAL_V3.csv"
    )

    fig, ax = plt.subplots(
        figsize=(max(10, 1.3*heat.shape[1]), max(6, .65*heat.shape[0]))
    )
    im = ax.imshow(heat.values, aspect="auto")
    ax.set_xticks(np.arange(heat.shape[1]))
    ax.set_xticklabels(
        heat.columns, rotation=55, ha="right", fontsize=8
    )
    ax.set_yticks(np.arange(heat.shape[0]))
    ax.set_yticklabels(heat.index, fontsize=8)
    ax.set_xlabel("Normalized function")
    ax.set_ylabel("Reference field")
    ax.set_title(
        f"Reference Field × Function counts (threshold={threshold:.2f})"
    )
    fig.colorbar(im, ax=ax, label="Patent families")

    cells = []
    for i, rr in enumerate(heat.index):
        for j, cc in enumerate(heat.columns):
            cells.append((int(heat.iloc[i,j]), rr, cc))
    cells.sort(reverse=True, key=lambda x: x[0])

    summary = (
        f"FINAL threshold={threshold:.2f}. {FINAL_THRESHOLD_RATIONALE}\n"
        f"Assigned={assigned.sum():,}/{len(out):,} ({assigned.mean():.1%}); "
        f"nonempty labels={assigned_out['zeroshot_label_id'].nunique()}/{len(labels)}; "
        f"mean accepted top1-top2 margin={margin[assigned].mean():.4f}.\n\n"
        "Top field-function cells:\n"
        + "\n".join(
            f"{n:>6} | {rr} | {cc}"
            for n, rr, cc in cells[:20]
        )
    )
    save_figure(
        fig,
        "11_field_function_heatmap_FINAL_V3",
        summary
    )

    # Threshold sensitivity figure
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(
        tuning["threshold"], tuning["assignment_rate"],
        marker="o", label="Assignment rate"
    )
    ax.plot(
        tuning["threshold"], tuning["label_coverage"],
        marker="s", label="Label coverage"
    )
    ax.plot(
        tuning["threshold"], tuning["mean_assigned_similarity"],
        marker="^", label="Mean assigned similarity"
    )
    ax.plot(
        tuning["threshold"], tuning["mean_top1_top2_margin"],
        marker="d", label="Mean semantic margin"
    )
    ax.axvline(
        threshold,
        linestyle="--",
        label=f"Final threshold={threshold:.2f}"
    )
    ax.set_xlabel("Similarity threshold")
    ax.set_ylabel("Diagnostic value")
    ax.set_title("Final zero-shot threshold sensitivity")
    ax.legend()
    ax.grid(alpha=.25)

    save_figure(
        fig,
        "11_zeroshot_threshold_sensitivity_FINAL_V3",
        "Full final threshold diagnostics:\n" + tuning.to_string(index=False),
        tuning
    )

    body = f"""
The final zero-shot similarity threshold is **{threshold:.2f}**. It was fixed after the completed
sensitivity analysis rather than selected to reproduce another study's assignment rate.
At this threshold, **{assigned.sum():,} of {len(out):,} patent families
({assigned.mean():.1%})** were assigned to **{assigned_out['zeroshot_label_id'].nunique()} nonempty labels**.

In addition to top-1 similarity, the final run records the **top-1 minus top-2 semantic margin** for every patent.
A deterministic sample immediately above and below the cutoff is saved for transparent manual audit.
"""
    R.update_section(
        "11_zeroshot",
        "Final semi-supervised zero-shot clustering",
        body
    )
    R.emit(body)

    mark_stage_done(
        stage,
        required,
        extra={
            "threshold": threshold,
            "rationale": FINAL_THRESHOLD_RATIONALE,
            "assignment_rate": float(assigned.mean()),
            "mean_margin_assigned": float(margin[assigned].mean()),
        }
    )


# ---------------------------------------------------------------------
# FINAL Stage 13 — corrected TV/TR/TN/ROS, using final assignments
# ---------------------------------------------------------------------
def stage_13_cluster_metrics():
    stage = "13_cluster_metrics_FINAL_V3"
    stage_banner(stage)
    if stage_done(stage, [CLUSTER_METRICS]):
        R.emit("FINAL V3 checkpoint found — cluster metrics skipped.")
        return

    df = pd.read_parquet(FAMILY_PARQUET)
    zs = pd.read_parquet(ZS_ASSIGN)
    ros = pd.read_parquet(ROS_PATH)
    emb = np.asarray(
        np.load(PATENT_EMB, mmap_mode="r"),
        dtype=np.float32
    )
    sents = pd.read_parquet(FIELD_SENTENCES)
    sent_emb = np.asarray(
        np.load(FIELD_SENT_EMB, mmap_mode="r"),
        dtype=np.float32
    )
    labels = pd.read_parquet(ZS_LABELS)

    work = (
        df.merge(
            zs[[
                "doc_idx","zeroshot_label_id",
                "zeroshot_similarity","semantic_margin","assigned"
            ]],
            on="doc_idx"
        )
        .merge(
            ros[["doc_idx","recycling_orientation_raw"]],
            on="doc_idx"
        )
    )
    work = work[work["assigned"]].copy()

    analysis_dt = pd.Timestamp(ANALYSIS_DATE)
    pub = pd.to_datetime(
        work["family_earliest_publication_date"],
        errors="coerce"
    )
    work["citation_age_years"] = (
        (analysis_dt - pub).dt.days / 365.25
    ).clip(lower=1.0).fillna(1.0)

    work["cited_by_patent_count"] = pd.to_numeric(
        work["cited_by_patent_count"],
        errors="coerce"
    ).fillna(0)

    work["simple_family_size"] = pd.to_numeric(
        work["simple_family_size"],
        errors="coerce"
    ).fillna(1).clip(lower=1)

    cluster_ids = sorted(
        work["zeroshot_label_id"].unique().astype(int)
    )

    centroids = {}
    for cid in cluster_ids:
        ids = work.loc[
            work["zeroshot_label_id"] == cid,
            "doc_idx"
        ].astype(int).values
        centroids[cid] = l2_normalize(
            emb[ids].mean(axis=0, keepdims=True)
        )[0]

    model = get_embedder()
    target_vec = model.encode(
        [TARGET_TECHNOLOGY],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0].astype("float32")

    rows = []
    for cid, grp in work.groupby("zeroshot_label_id"):
        cid = int(cid)
        P = len(grp)
        citation_rate = (
            grp["cited_by_patent_count"]
            / grp["citation_age_years"]
        )
        rows.append({
            "cluster_id": cid,
            "cluster_size": int(P),
            "size_factor": float(1 + math.log(max(P,1))),
            "median_age_normalized_forward_citations": float(np.median(citation_rate)),
            "median_simple_family_size": float(np.median(grp["simple_family_size"])),
            "mean_zeroshot_similarity": float(grp["zeroshot_similarity"].mean()),
            "mean_semantic_margin": float(grp["semantic_margin"].mean()),
            "TR_raw_similarity": float(np.dot(centroids[cid], target_vec)),
            "ROS_raw_mean": float(grp["recycling_orientation_raw"].mean()),
        })

    metrics = pd.DataFrame(rows)

    # Correct TV: z-score cluster medians FIRST, size factor SECOND.
    metrics["citation_median_z"] = safe_z(
        metrics["median_age_normalized_forward_citations"]
    ).values
    metrics["family_median_z"] = safe_z(
        metrics["median_simple_family_size"]
    ).values
    metrics["citation_component"] = (
        metrics["citation_median_z"]
        * metrics["size_factor"]
    )
    metrics["family_component"] = (
        metrics["family_median_z"]
        * metrics["size_factor"]
    )
    metrics["TV"] = (
        metrics["citation_component"]
        + metrics["family_component"]
    ) / 2

    metrics["TR"] = safe_z(
        metrics["TR_raw_similarity"]
    ).values
    metrics["ROS"] = safe_z(
        metrics["ROS_raw_mean"]
    ).values

    # TN unchanged in concept, recomputed against final cluster centroids.
    C = np.stack([
        centroids[cid] for cid in cluster_ids
    ]).astype("float32")
    sim_matrix = C @ sent_emb.T
    np.save(
        EMBED_DIR / "13_tn_cluster_sentence_similarity_FINAL_V3.npy",
        sim_matrix.astype("float32")
    )
    q3 = float(np.quantile(sim_matrix, 0.75))

    sent_pub = pd.to_datetime(
        sents["publication_date"],
        errors="coerce"
    )
    sent_age = (
        (analysis_dt - sent_pub).dt.days / 365.25
    ).clip(lower=1.0).fillna(1.0).values
    weights = 1.0 / sent_age

    tn_rows = []
    for i, cid in enumerate(cluster_ids):
        mask = sim_matrix[i] >= q3
        tn_rows.append({
            "cluster_id": cid,
            "TN_overlap_raw": float(weights[mask].sum()),
            "TN_matching_sentences": int(mask.sum())
        })

    tn = pd.DataFrame(tn_rows)
    tn["TN"] = -safe_z(
        tn["TN_overlap_raw"]
    ).values

    metrics = metrics.merge(
        tn, on="cluster_id", how="left"
    )
    metrics = metrics.merge(
        labels[[
            "label_id","label_text","cpc_class",
            "reference_field","function"
        ]],
        left_on="cluster_id",
        right_on="label_id",
        how="left"
    )

    metrics["key_cluster_all"] = (
        (metrics["TV"] > 0)
        & (metrics["TR"] > 0)
    )
    metrics["primary_key_cluster"] = (
        metrics["key_cluster_all"]
        & (metrics["cluster_size"] >= PRIMARY_MIN_CLUSTER_SIZE)
    )
    metrics["key_cluster"] = metrics["primary_key_cluster"]

    metrics["TV_plus_TR"] = (
        metrics["TV"] + metrics["TR"]
    )
    metrics["novelty_orientation"] = np.where(
        metrics["TN"] >= 0,
        "exploration-oriented",
        "exploitation-oriented"
    )
    metrics["recycling_orientation"] = np.where(
        metrics["ROS"] >= 0,
        "end-of-life-recycling-oriented",
        "repair/service-leaning"
    )
    metrics["cluster_support_class"] = np.where(
        metrics["cluster_size"] >= PRIMARY_MIN_CLUSTER_SIZE,
        "primary-support",
        "low-support exploratory"
    )

    metrics = metrics.sort_values(
        ["primary_key_cluster","key_cluster_all","TV_plus_TR"],
        ascending=[False,False,False]
    ).reset_index(drop=True)

    metrics.to_parquet(
        CLUSTER_METRICS,
        index=False
    )
    save_table(
        metrics,
        "13_cluster_metrics_FINAL_V3",
        print_full=True
    )

    primary = metrics[
        metrics["primary_key_cluster"]
    ].copy()
    emerging = metrics[
        metrics["key_cluster_all"]
        & (~metrics["primary_key_cluster"])
    ].copy()

    sensitivity = pd.DataFrame([
        {
            "category": "High TV / High TR, all sizes",
            "clusters": int(metrics["key_cluster_all"].sum())
        },
        {
            "category": f"Primary: High TV / High TR and n>={PRIMARY_MIN_CLUSTER_SIZE}",
            "clusters": int(primary.shape[0])
        },
        {
            "category": f"Emerging exploratory: High TV / High TR but n<{PRIMARY_MIN_CLUSTER_SIZE}",
            "clusters": int(emerging.shape[0])
        },
    ])
    save_table(
        sensitivity,
        "13_cluster_support_sensitivity_FINAL_V3",
        print_full=True
    )

    fig, ax = plt.subplots(figsize=(10, 8))
    sc = ax.scatter(
        metrics["TR"],
        metrics["TV"],
        c=metrics["TN"],
        s=45 + 4*np.sqrt(metrics["cluster_size"]),
        alpha=.74
    )
    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)
    ax.set_xlabel("TR (z-score)")
    ax.set_ylabel("TV (corrected)")
    ax.set_title("Final technology-opportunity map: TV × TR (color = TN)")
    fig.colorbar(sc, ax=ax, label="TN")

    for _, r in primary.head(15).iterrows():
        ax.annotate(
            f"P{int(r['cluster_id'])}",
            (r["TR"], r["TV"]),
            fontsize=8
        )
    for _, r in emerging.head(10).iterrows():
        ax.annotate(
            f"E{int(r['cluster_id'])}",
            (r["TR"], r["TV"]),
            fontsize=8
        )

    text = (
        f"Final threshold={FINAL_ZEROSHOT_THRESHOLD:.2f}. "
        f"Evaluated clusters={len(metrics)}; TN Q3={q3:.4f}; "
        f"High-TV/high-TR clusters={int(metrics['key_cluster_all'].sum())}; "
        f"primary n>={PRIMARY_MIN_CLUSTER_SIZE}: {len(primary)}; "
        f"emerging low-support high-TV/high-TR: {len(emerging)}.\n\n"
        "PRIMARY OPPORTUNITIES:\n"
        + primary[[
            "cluster_id","label_text","cluster_size",
            "TV","TR","TN","ROS",
            "mean_zeroshot_similarity","mean_semantic_margin"
        ]].to_string(index=False)
        + "\n\nEMERGING HIGH-TV/HIGH-TR OPPORTUNITIES:\n"
        + (
            emerging[[
                "cluster_id","label_text","cluster_size",
                "TV","TR","TN","ROS",
                "mean_zeroshot_similarity","mean_semantic_margin"
            ]].to_string(index=False)
            if len(emerging) else "None"
        )
    )
    save_figure(
        fig,
        "13_TV_TR_scatter_FINAL_V3",
        text,
        metrics
    )

    body = f"""
Using the final zero-shot assignments, TV/TR/TN/ROS were recomputed from scratch.
TV retains the corrected operation order: **cluster medians → z-score across clusters → cluster-size weighting**.
The final screen identified **{int(metrics['key_cluster_all'].sum())} high-TV/high-TR clusters**.
Of these, **{len(primary)} clusters** meet the manuscript-primary support criterion
(**n ≥ {PRIMARY_MIN_CLUSTER_SIZE} patent families**) and **{len(emerging)} clusters** are retained as smaller
exploratory/emerging opportunities.

TN remains an interpretive exploration-versus-exploitation dimension, and ROS remains an independent
end-of-life-recycling-versus-service orientation dimension.
"""
    R.update_section(
        "13_metrics",
        "Final technology-opportunity evaluation",
        body
    )
    R.emit(body)

    mark_stage_done(
        stage,
        [CLUSTER_METRICS],
        extra={
            "tn_q3": q3,
            "high_tv_tr_all": int(metrics["key_cluster_all"].sum()),
            "primary": len(primary),
            "emerging": len(emerging),
        }
    )


# ---------------------------------------------------------------------
# FINAL Stage 14 — defensible key patents and mechanism evidence
# ---------------------------------------------------------------------
def _mechanism_is_generic(x):
    x = clean_phrase(x, 220).lower()
    if len(x) < 8:
        return True
    for pat in GENERIC_MECHANISM_PATTERNS:
        if re.search(pat, x, flags=re.I):
            return True
    return False

def _top_mean(values, n):
    vals = sorted(
        [float(v) for v in values if pd.notna(v)],
        reverse=True
    )
    if not vals:
        return 0.0
    return float(np.mean(vals[:n]))

def stage_14_key_cluster_analysis():
    stage = "14_key_cluster_analysis_FINAL_V3"
    stage_banner(stage)
    required = [
        KEY_PATENTS_PATH,
        KEY_SENTENCES_PATH,
        KEY_NODES_PATH,
        KEY_MECHANISMS_SUPPORTED_PATH,
        KEY_MECHANISMS_SINGLETON_PATH
    ]
    if stage_done(stage, required):
        R.emit("FINAL V3 checkpoint found — key-cluster analysis skipped.")
        return

    df = pd.read_parquet(FAMILY_PARQUET)
    zs = pd.read_parquet(ZS_ASSIGN)
    metrics = pd.read_parquet(CLUSTER_METRICS)
    rawf = pd.read_parquet(RAW_FUNCTIONS)
    fmap = pd.read_parquet(FUNCTION_MAP)

    rawf = rawf.merge(
        fmap[[
            "action_object",
            "prototype_function"
        ]].drop_duplicates(),
        on="action_object",
        how="left"
    )
    rawf["normalized_function"] = rawf["prototype_function"]
    rawf["mechanism_clean"] = rawf["mechanism"].map(
        _clean_mechanism_text
    )
    rawf.loc[
        rawf["mechanism_clean"].map(_mechanism_is_generic),
        "mechanism_clean"
    ] = ""

    work = df.merge(
        zs[[
            "doc_idx","zeroshot_label_id","zeroshot_similarity",
            "semantic_margin","assigned"
        ]],
        on="doc_idx"
    )
    work = work[work["assigned"]].copy()

    # Analyze both primary and smaller emerging high-TV/high-TR clusters;
    # primary/emerging status remains explicit in outputs.
    keys = metrics[
        metrics["key_cluster_all"]
    ].sort_values(
        ["primary_key_cluster","TV_plus_TR"],
        ascending=[False,False]
    )

    all_nodes = []
    all_key_patents = []
    all_key_sentences = []
    all_supported_mechanisms = []
    all_singleton_mechanisms = []
    cluster_evidence_rows = []

    for _, kr in keys.iterrows():
        cid = int(kr["cluster_id"])
        target_function = str(kr["function"])

        cdir = NETWORK_DIR_V3 / f"cluster_{cid:03d}"
        cdir.mkdir(parents=True, exist_ok=True)
        done_marker = cdir / "DONE.json"

        if done_marker.exists():
            R.emit(f"[FINAL V3 cluster {cid}] cached.")
            for fn, collector in [
                ("nodes.parquet", all_nodes),
                ("key_patents.parquet", all_key_patents),
                ("key_sentences.parquet", all_key_sentences),
                ("supported_mechanisms.parquet", all_supported_mechanisms),
                ("singleton_mechanisms.parquet", all_singleton_mechanisms),
            ]:
                p = cdir / fn
                if p.exists():
                    collector.append(pd.read_parquet(p))
            meta = json.loads(done_marker.read_text(encoding="utf-8"))
            cluster_evidence_rows.append(meta["evidence_summary"])
            continue

        cluster_docs = work[
            work["zeroshot_label_id"] == cid
        ].copy()
        doc_ids = set(
            cluster_docs["doc_idx"].astype(int)
        )

        # CPC subclass per document
        cpc_by_doc = {}
        for _, r in cluster_docs.iterrows():
            subclasses = sorted(set(filter(
                None,
                (
                    cpc_subclass(x)
                    for x in parse_cpc_codes(
                        r["cpc_classifications"]
                    )
                )
            )))
            cpc_by_doc[int(r["doc_idx"])] = subclasses

        # Function/mechanism evidence
        fsub = rawf[
            rawf["doc_idx"].isin(doc_ids)
            & rawf["normalized_function"].notna()
        ].copy()

        func_by_doc = (
            fsub.groupby("doc_idx")["normalized_function"]
            .apply(lambda s: sorted(set(s)))
            .to_dict()
        )

        mech_by_doc = (
            fsub[fsub["mechanism_clean"].str.len() > 0]
            .groupby("doc_idx")["mechanism_clean"]
            .apply(lambda s: sorted(set(s)))
            .to_dict()
        )

        cpc_G = build_cooccurrence([
            cpc_by_doc.get(i, [])
            for i in doc_ids
        ])
        func_G = build_cooccurrence([
            func_by_doc.get(i, [])
            for i in doc_ids
        ])
        mech_G = build_cooccurrence([
            mech_by_doc.get(i, [])
            for i in doc_ids
        ])

        cpc_nodes, cpc_edges = network_tables(
            cpc_G, cid, "CPC_subclass"
        )
        fn_nodes, fn_edges = network_tables(
            func_G, cid, "Function"
        )
        mech_nodes, mech_edges = network_tables(
            mech_G, cid, "Mechanism"
        )

        nodes = pd.concat(
            [cpc_nodes, fn_nodes, mech_nodes],
            ignore_index=True
        )
        edges = pd.concat(
            [cpc_edges, fn_edges, mech_edges],
            ignore_index=True
        )
        nodes.to_parquet(
            cdir / "nodes.parquet",
            index=False
        )
        edges.to_parquet(
            cdir / "edges.parquet",
            index=False
        )
        all_nodes.append(nodes)

        cpc_cent = (
            cpc_nodes.set_index("node")["centrality"].to_dict()
            if len(cpc_nodes) else {}
        )
        fn_cent = (
            fn_nodes.set_index("node")["centrality"].to_dict()
            if len(fn_nodes) else {}
        )

        # Build patent evidence table.
        p_rows = []
        for _, r in cluster_docs.iterrows():
            did = int(r["doc_idx"])
            cpcs = cpc_by_doc.get(did, [])
            funcs = func_by_doc.get(did, [])
            mechs = mech_by_doc.get(did, [])

            contains_cluster_function = (
                target_function in funcs
            )

            cpc_component_raw = _top_mean(
                [cpc_cent.get(x, 0.0) for x in cpcs],
                KEY_PATENT_TOP_CPC_N
            )
            function_component_raw = _top_mean(
                [fn_cent.get(x, 0.0) for x in funcs],
                KEY_PATENT_TOP_FUNCTION_N
            )

            p_rows.append({
                "cluster_id": cid,
                "cluster_type": (
                    "primary"
                    if bool(kr["primary_key_cluster"])
                    else "emerging"
                ),
                "cluster_label": kr["label_text"],
                "cluster_target_function": target_function,
                "doc_idx": did,
                "family_id": r["family_id"],
                "lens_id": r.get("lens_id"),
                "title": r.get("title"),
                "applicants": r.get("applicants"),
                "owners": r.get("owners"),
                "publication_date": r.get("publication_date"),
                "contains_cluster_function": contains_cluster_function,
                "cpc_component_raw": cpc_component_raw,
                "function_component_raw": function_component_raw,
                "label_fit_raw": float(r["zeroshot_similarity"]),
                "semantic_margin_raw": float(r["semantic_margin"]),
                "cpc_subclasses": " ; ".join(cpcs),
                "normalized_functions": " ; ".join(funcs),
                "mechanisms": " ; ".join(mechs),
            })

        ptab = pd.DataFrame(p_rows)

        # Function evidence is mandatory for "key patent" status.
        eligible = ptab[
            ptab["contains_cluster_function"]
        ].copy()

        # Balanced cluster-internal standardized score.
        if len(eligible):
            eligible["cpc_component_z"] = safe_z(
                eligible["cpc_component_raw"]
            ).values
            eligible["function_component_z"] = safe_z(
                eligible["function_component_raw"]
            ).values
            eligible["label_fit_z"] = safe_z(
                eligible["label_fit_raw"]
            ).values

            eligible["key_patent_score"] = (
                eligible["cpc_component_z"]
                + eligible["function_component_z"]
                + eligible["label_fit_z"]
            ) / 3

            eligible = eligible.sort_values(
                ["key_patent_score","label_fit_raw"],
                ascending=[False,False]
            )
        else:
            eligible["cpc_component_z"] = []
            eligible["function_component_z"] = []
            eligible["label_fit_z"] = []
            eligible["key_patent_score"] = []

        keyp = eligible.head(
            KEY_PATENTS_PER_CLUSTER
        ).copy()
        keyp.to_parquet(
            cdir / "key_patents.parquet",
            index=False
        )
        all_key_patents.append(keyp)

        # Mechanism support is measured at family level.
        mech_support_rows = []
        for mech, grp in (
            fsub[fsub["mechanism_clean"].str.len() > 0]
            .groupby("mechanism_clean")
        ):
            fams = grp["family_id"].nunique()
            mech_support_rows.append({
                "cluster_id": cid,
                "cluster_type": (
                    "primary"
                    if bool(kr["primary_key_cluster"])
                    else "emerging"
                ),
                "cluster_label": kr["label_text"],
                "cluster_target_function": target_function,
                "mechanism": mech,
                "patent_families": int(fams),
                "instances": int(len(grp)),
                "example_sentence": grp.iloc[0]["sentence"],
            })

        mech_support = pd.DataFrame(mech_support_rows)
        if len(mech_support):
            mech_support = mech_support.sort_values(
                ["patent_families","instances"],
                ascending=[False,False]
            )

            supported = mech_support[
                mech_support["patent_families"]
                >= MECHANISM_MIN_FAMILY_SUPPORT
            ].copy()
            singleton = mech_support[
                mech_support["patent_families"]
                < MECHANISM_MIN_FAMILY_SUPPORT
            ].copy()
        else:
            supported = pd.DataFrame(columns=[
                "cluster_id","cluster_type","cluster_label",
                "cluster_target_function","mechanism",
                "patent_families","instances","example_sentence"
            ])
            singleton = supported.copy()

        supported.to_parquet(
            cdir / "supported_mechanisms.parquet",
            index=False
        )
        singleton.to_parquet(
            cdir / "singleton_mechanisms.parquet",
            index=False
        )
        all_supported_mechanisms.append(supported)
        all_singleton_mechanisms.append(singleton)

        # Key sentences MUST support the assigned cluster function.
        ks_rows = []
        for _, p in keyp.iterrows():
            cand = fsub[
                (fsub["doc_idx"] == int(p["doc_idx"]))
                & (
                    fsub["normalized_function"]
                    == target_function
                )
            ].drop_duplicates("sentence")

            # No fallback to unrelated functions.
            for _, rr in cand.head(
                KEY_SENTENCES_PER_PATENT
            ).iterrows():
                ks_rows.append({
                    "cluster_id": cid,
                    "cluster_type": p["cluster_type"],
                    "doc_idx": int(p["doc_idx"]),
                    "family_id": p["family_id"],
                    "lens_id": p["lens_id"],
                    "title": p["title"],
                    "key_function": target_function,
                    "mechanism": rr["mechanism_clean"],
                    "sentence": rr["sentence"],
                    "key_patent_score": p["key_patent_score"],
                    "zeroshot_similarity": p["label_fit_raw"],
                })

        ksent = pd.DataFrame(ks_rows)
        ksent.to_parquet(
            cdir / "key_sentences.parquet",
            index=False
        )
        all_key_sentences.append(ksent)

        # Figures
        plot_network(
            cpc_G,
            cpc_nodes,
            f"14_cluster_{cid:03d}_cpc_network_FINAL_V3",
            f"Cluster {cid}: CPC-subclass co-occurrence"
        )
        plot_network(
            func_G,
            fn_nodes,
            f"14_cluster_{cid:03d}_function_network_FINAL_V3",
            f"Cluster {cid}: normalized-function co-occurrence"
        )

        if len(supported):
            plotm = supported.head(15).sort_values(
                "patent_families"
            )
            fig, ax = plt.subplots(
                figsize=(11, max(6, .42*len(plotm)))
            )
            ax.barh(
                plotm["mechanism"],
                plotm["patent_families"]
            )
            ax.set_xlabel("Supporting patent families")
            ax.set_title(
                f"Cluster {cid}: recurrent implementation mechanisms"
            )
            save_figure(
                fig,
                f"14_cluster_{cid:03d}_supported_mechanisms_FINAL_V3",
                "Only recurrent mechanisms with family support >= "
                f"{MECHANISM_MIN_FAMILY_SUPPORT} are plotted.\n"
                + supported.head(20).to_string(index=False),
                supported
            )

        evidence_summary = {
            "cluster_id": cid,
            "cluster_type": (
                "primary"
                if bool(kr["primary_key_cluster"])
                else "emerging"
            ),
            "cluster_label": kr["label_text"],
            "cluster_size": int(len(cluster_docs)),
            "target_function": target_function,
            "function_matched_patents": int(len(eligible)),
            "function_evidence_share": float(
                len(eligible) / max(len(cluster_docs), 1)
            ),
            "key_patents_returned": int(len(keyp)),
            "supported_mechanisms": int(len(supported)),
            "singleton_mechanisms": int(len(singleton)),
            "TV": float(kr["TV"]),
            "TR": float(kr["TR"]),
            "TN": float(kr["TN"]),
            "ROS": float(kr["ROS"]),
        }
        cluster_evidence_rows.append(
            evidence_summary
        )

        atomic_write_json(
            done_marker,
            {
                "evidence_summary": evidence_summary,
                "top_key_patents": keyp.head(5)[[
                    "lens_id","title",
                    "key_patent_score","label_fit_raw"
                ]].to_dict("records") if len(keyp) else [],
                "top_supported_mechanisms": supported.head(8).to_dict("records") if len(supported) else [],
            }
        )

        R.emit(json.dumps(
            evidence_summary,
            indent=2,
            ensure_ascii=False
        ))

    nodes_all = (
        pd.concat(all_nodes, ignore_index=True)
        if all_nodes else pd.DataFrame()
    )
    kp_all = (
        pd.concat(all_key_patents, ignore_index=True)
        if all_key_patents else pd.DataFrame()
    )
    ks_all = (
        pd.concat(all_key_sentences, ignore_index=True)
        if all_key_sentences else pd.DataFrame()
    )
    supported_all = (
        pd.concat(all_supported_mechanisms, ignore_index=True)
        if all_supported_mechanisms else pd.DataFrame()
    )
    singleton_all = (
        pd.concat(all_singleton_mechanisms, ignore_index=True)
        if all_singleton_mechanisms else pd.DataFrame()
    )

    nodes_all.to_parquet(
        KEY_NODES_PATH,
        index=False
    )
    kp_all.to_parquet(
        KEY_PATENTS_PATH,
        index=False
    )
    ks_all.to_parquet(
        KEY_SENTENCES_PATH,
        index=False
    )
    supported_all.to_parquet(
        KEY_MECHANISMS_SUPPORTED_PATH,
        index=False
    )
    singleton_all.to_parquet(
        KEY_MECHANISMS_SINGLETON_PATH,
        index=False
    )

    save_table(
        nodes_all,
        "14_all_key_cluster_nodes_FINAL_V3",
        print_full=True
    )
    save_table(
        kp_all,
        "14_key_patents_FINAL_V3",
        print_full=True
    )
    save_table(
        ks_all,
        "14_key_sentences_FINAL_V3",
        print_full=True
    )
    save_table(
        supported_all,
        "14_supported_mechanisms_FINAL_V3",
        print_full=True
    )
    save_table(
        singleton_all,
        "14_singleton_mechanism_examples_FINAL_V3",
        print_full=True
    )

    evidence_df = pd.DataFrame(
        cluster_evidence_rows
    ).sort_values(
        ["cluster_type","function_evidence_share"],
        ascending=[True,False]
    )
    save_table(
        evidence_df,
        "14_cluster_evidence_support_FINAL_V3",
        print_full=True
    )

    # Merge evidence diagnostics into the final metric file.
    m = pd.read_parquet(CLUSTER_METRICS)
    m = m.merge(
        evidence_df[[
            "cluster_id","function_matched_patents",
            "function_evidence_share","key_patents_returned",
            "supported_mechanisms","singleton_mechanisms"
        ]],
        on="cluster_id",
        how="left"
    )
    m.to_parquet(
        CLUSTER_METRICS,
        index=False
    )
    save_table(
        m,
        "13_cluster_metrics_WITH_EVIDENCE_FINAL_V3",
        print_full=True
    )

    body = f"""
The final key-patent stage prevents functionally irrelevant, CPC-rich patents from becoming representative evidence.
A patent is eligible for key-patent ranking **only if its extracted normalized functions contain the cluster's
representative function**. Eligible patents are then ranked by the mean of three within-cluster z-normalized
components: **CPC-network centrality, function-network centrality, and zero-shot label similarity**.

Mechanisms supported by at least **{MECHANISM_MIN_FAMILY_SUPPORT} patent families** are reported as recurrent
mechanisms. One-family mechanisms remain available as case-level examples but are not described as dominant.
Key sentences are drawn only from sentences linked to the cluster's representative function.
"""
    R.update_section(
        "14_networks",
        "Final key-patent and mechanism evidence",
        body
    )
    R.emit(body)

    mark_stage_done(
        stage,
        required,
        extra={
            "clusters_analyzed": int(len(keys)),
            "key_patents": int(len(kp_all)),
            "key_sentences": int(len(ks_all)),
            "supported_mechanisms": int(len(supported_all)),
            "singleton_mechanisms": int(len(singleton_all)),
        }
    )


# ---------------------------------------------------------------------
# FINAL Stage 15 — final opportunity table, QA, prompts
# ---------------------------------------------------------------------
def stage_15_validation_and_llm_prompts():
    stage = "15_validation_llm_prompts_FINAL_V3"
    stage_banner(stage)
    marker = CHECKPOINT_DIR / "15_validation_llm_prompts_FINAL_V3.done"
    required = [
        marker,
        LLM_PROMPT_INDEX,
        FINAL_OPPORTUNITY_TABLE,
        FINAL_QA_PATH
    ]
    if stage_done(stage, required):
        R.emit("FINAL V3 checkpoint found — final validation package skipped.")
        return

    metrics = pd.read_parquet(CLUSTER_METRICS)
    keyp = pd.read_parquet(KEY_PATENTS_PATH)
    keysent = pd.read_parquet(KEY_SENTENCES_PATH)
    nodes = pd.read_parquet(KEY_NODES_PATH)
    supported_mech = pd.read_parquet(
        KEY_MECHANISMS_SUPPORTED_PATH
    )

    # Final opportunity table: primary first, then emerging.
    final_opp = metrics[
        metrics["key_cluster_all"]
    ].copy()
    final_opp["opportunity_status"] = np.where(
        final_opp["primary_key_cluster"],
        "Primary",
        "Emerging / exploratory"
    )
    final_opp["interpretation"] = np.select(
        [
            (final_opp["TN"] >= 0) & (final_opp["ROS"] >= 0),
            (final_opp["TN"] >= 0) & (final_opp["ROS"] < 0),
            (final_opp["TN"] < 0) & (final_opp["ROS"] >= 0),
        ],
        [
            "Exploration-oriented and end-of-life-recycling-oriented",
            "Exploration-oriented but repair/service-leaning",
            "Exploitation-oriented and end-of-life-recycling-oriented",
        ],
        default="Exploitation-oriented and repair/service-leaning"
    )

    final_cols = [
        "opportunity_status","cluster_id","label_text",
        "reference_field","function","cluster_size",
        "TV","TR","TN","ROS","TV_plus_TR",
        "mean_zeroshot_similarity","mean_semantic_margin",
        "function_matched_patents","function_evidence_share",
        "key_patents_returned","supported_mechanisms",
        "interpretation"
    ]
    final_opp = final_opp[
        [c for c in final_cols if c in final_opp.columns]
    ].sort_values(
        ["opportunity_status","TV_plus_TR"],
        ascending=[True,False]
    )

    final_opp.to_parquet(
        FINAL_OPPORTUNITY_TABLE,
        index=False
    )
    save_table(
        final_opp,
        "15_FINAL_OPPORTUNITY_TABLE",
        print_full=True
    )

    # QA gates: warnings are saved; structural failures stop the run.
    qa_rows = []

    assignment = pd.read_parquet(ZS_ASSIGN)
    assignment_rate = float(
        assignment["assigned"].mean()
    )
    qa_rows.append({
        "check": "Zero-shot assignment rate",
        "value": assignment_rate,
        "status": (
            "PASS"
            if 0.40 <= assignment_rate <= 0.85
            else "WARN"
        ),
        "criterion": "40%–85% retained as a broad final plausibility range"
    })

    qa_rows.append({
        "check": "Primary opportunity count",
        "value": int(metrics["primary_key_cluster"].sum()),
        "status": (
            "PASS"
            if int(metrics["primary_key_cluster"].sum()) >= 1
            else "FAIL"
        ),
        "criterion": "At least one primary high-TV/high-TR cluster with n>=10"
    })

    if len(keyp):
        key_function_match = float(
            keyp["contains_cluster_function"].mean()
        )
    else:
        key_function_match = 0.0

    qa_rows.append({
        "check": "Key patents contain cluster function",
        "value": key_function_match,
        "status": (
            "PASS"
            if np.isclose(key_function_match, 1.0)
            else "FAIL"
        ),
        "criterion": "100% of key patents must contain the normalized cluster function"
    })

    evidence_share = (
        final_opp["function_evidence_share"].dropna()
        if "function_evidence_share" in final_opp.columns
        else pd.Series(dtype=float)
    )
    qa_rows.append({
        "check": "Median function-evidence share among reported opportunities",
        "value": float(evidence_share.median()) if len(evidence_share) else np.nan,
        "status": (
            "PASS"
            if len(evidence_share) and evidence_share.median() > 0
            else "WARN"
        ),
        "criterion": "Reported opportunities should have traceable function-level patent evidence"
    })

    qa_rows.append({
        "check": "Supported mechanisms separated from singletons",
        "value": int(len(supported_mech)),
        "status": "PASS",
        "criterion": f"Only mechanisms with >= {MECHANISM_MIN_FAMILY_SUPPORT} patent families are called recurrent"
    })

    qa = pd.DataFrame(qa_rows)
    qa.to_parquet(
        FINAL_QA_PATH,
        index=False
    )
    save_table(
        qa,
        "15_FINAL_QA",
        print_full=True
    )

    # Hard-stop only on genuine methodological integrity failures.
    failures = qa[
        qa["status"] == "FAIL"
    ]
    if len(failures):
        raise RuntimeError(
            "FINAL QA FAILED:\n"
            + failures.to_string(index=False)
        )

    validation = pd.DataFrame([
        {
            "validation_component": "Threshold sensitivity",
            "status": "Completed",
            "basis": (
                f"Final threshold fixed at {FINAL_ZEROSHOT_THRESHOLD:.2f}; full assignment, coverage, "
                "similarity, semantic-margin, and borderline audit outputs saved."
            )
        },
        {
            "validation_component": "Key-patent functional traceability",
            "status": "Completed",
            "basis": (
                "Every reported key patent must contain the normalized representative cluster function."
            )
        },
        {
            "validation_component": "Mechanism support",
            "status": "Completed",
            "basis": (
                f"Recurrent mechanisms require support from >= {MECHANISM_MIN_FAMILY_SUPPORT} patent families; "
                "singletons are retained separately as examples."
            )
        },
        {
            "validation_component": "Historical predictive validation",
            "status": "Not claimed",
            "basis": (
                "The Lens export provides cumulative forward-citation counts but not historical citation-event dates; "
                "using current counts at an earlier cutoff would introduce future-information leakage."
            )
        },
        {
            "validation_component": "Expert validation",
            "status": "External human step",
            "basis": (
                "Experts can assess transferability, feasibility, novelty, and practical usefulness using the saved "
                "patent/sentence/mechanism evidence package."
            )
        },
    ])
    save_table(
        validation,
        "15_validation_readiness_FINAL_V3",
        print_full=True
    )

    # Final LLM prompts only for Primary opportunities.
    primary = metrics[
        metrics["primary_key_cluster"]
    ].sort_values(
        "TV_plus_TR",
        ascending=False
    ).head(N_LLM_CLUSTERS)

    prompt_rows = []
    for _, r in primary.iterrows():
        cid = int(r["cluster_id"])

        ref_nodes = nodes[
            (nodes["cluster_id"] == cid)
            & (nodes["network_type"] == "CPC_subclass")
        ].head(5)

        fn_nodes = nodes[
            (nodes["cluster_id"] == cid)
            & (nodes["network_type"] == "Function")
        ].head(5)

        mechs = supported_mech[
            supported_mech["cluster_id"] == cid
        ].head(8)

        kp = keyp[
            keyp["cluster_id"] == cid
        ].head(8)

        sent = keysent[
            keysent["cluster_id"] == cid
        ].head(15)

        prompt = f"""
You are a technology innovation strategist supporting end-of-life recycling and product-recyclability R&D.

Use only the supplied cluster evidence to generate exactly {N_SOLUTIONS_PER_CLUSTER} technically distinct
cross-field solution concepts.

TARGET TECHNOLOGY
{TARGET_TECHNOLOGY}

FIELD OF INTEREST
{FIELD_OF_INTEREST}

FINAL PRIMARY OPPORTUNITY
Cluster ID: {cid}
Label: {r['label_text']}
TV={r['TV']:.4f}; TR={r['TR']:.4f}; TN={r['TN']:.4f}; ROS={r['ROS']:.4f}
Cluster size={int(r['cluster_size'])}

KEY CPC SUBCLASSES
{ref_nodes[['node','frequency','centrality']].to_string(index=False) if len(ref_nodes) else 'None'}

KEY FUNCTIONS
{fn_nodes[['node','frequency','centrality']].to_string(index=False) if len(fn_nodes) else 'None'}

RECURRENT IMPLEMENTATION MECHANISMS
{mechs[['mechanism','patent_families','instances']].to_string(index=False) if len(mechs) else 'No recurrent mechanism met the >=2-family support rule.'}

FUNCTION-MATCHED KEY PATENTS
{kp[['lens_id','title','key_patent_score','label_fit_raw']].to_string(index=False) if len(kp) else 'None'}

TRACEABLE FUNCTION-MATCHED SENTENCES
{sent[['lens_id','key_function','mechanism','sentence']].to_string(index=False) if len(sent) else 'None'}

OUTPUT FOR EACH SOLUTION
- Solution number
- Title
- Core innovation
- Cross-field integration
- Function transferred
- Mechanism adapted
- Technical feasibility (High/Medium/Low + justification)
- End-of-life recycling/recyclability contribution
- Differentiation from supplied patent evidence
- Implementation pathway
- Technical challenges
- Evidence references (Lens IDs and sentences)

Do not claim freedom-to-operate, patentability, or legal clearance.
""".strip()

        p = LLM_DIR / f"cluster_{cid:03d}_FINAL_V3_prompt.txt"
        atomic_write_text(p, prompt)
        R.emit(
            f"\n===== FINAL V3 LLM PROMPT — CLUSTER {cid} =====\n{prompt}\n"
        )

        prompt_rows.append({
            "cluster_id": cid,
            "label_text": r["label_text"],
            "prompt_path": str(p),
            "TV": r["TV"],
            "TR": r["TR"],
            "TN": r["TN"],
            "ROS": r["ROS"],
        })

    prompt_index = pd.DataFrame(
        prompt_rows
    )
    prompt_index.to_parquet(
        LLM_PROMPT_INDEX,
        index=False
    )
    save_table(
        prompt_index,
        "15_llm_prompt_index_FINAL_V3",
        print_full=True
    )

    primary_count = int(
        metrics["primary_key_cluster"].sum()
    )
    emerging_count = int(
        (
            metrics["key_cluster_all"]
            & ~metrics["primary_key_cluster"]
        ).sum()
    )

    body = f"""
The final QA package passed all hard methodological integrity checks.
The manuscript-ready opportunity set contains **{primary_count} primary opportunities** and
**{emerging_count} smaller emerging/exploratory high-TV/high-TR opportunities**.
Every final key patent is function-matched to its cluster, and recurrent mechanisms are distinguished from
single-family examples.

The final threshold is **{FINAL_ZEROSHOT_THRESHOLD:.2f}**, with complete threshold-sensitivity and semantic-margin
diagnostics retained. Historical predictive validation is not claimed because time-stamped citation events are
not available in the export.
"""
    R.update_section(
        "15_validation_llm",
        "Final validation, QA, and manuscript-ready opportunity set",
        body
    )
    R.emit(body)

    marker.write_text("done", encoding="utf-8")
    mark_stage_done(
        stage,
        required,
        extra={
            "primary_opportunities": primary_count,
            "emerging_opportunities": emerging_count,
            "qa_failures": int((qa["status"] == "FAIL").sum()),
            "qa_warnings": int((qa["status"] == "WARN").sum()),
        }
    )


# ---------------------------------------------------------------------
# FINAL run order
# ---------------------------------------------------------------------
def run_pipeline():
    start = time.time()
    R.emit("\nBEGINNING / RESUMING FINAL V3 MANUSCRIPT-READY RECYCLE TOA PIPELINE")
    R.emit(f"Revision: {PIPELINE_REVISION}")
    R.emit(f"Start time: {datetime.now().isoformat()}")

    # Stable V1/V2 stages are safely reused when checkpointed.
    stage_01_ingest()
    stage_02_family_dedup()
    stage_02b_descriptives()
    stage_03_patent_embeddings()
    stage_04_umap()
    stage_05_hdbscan_tuning()
    stage_06_base_bertopic()
    stage_07_reference_fields()
    stage_08_extract_mechanism_functions()
    stage_09_normalize_score_functions()
    stage_10_circularity_relevance()
    stage_11_zeroshot_assignment()
    stage_12_field_sentence_corpus()
    stage_13_cluster_metrics()
    stage_14_key_cluster_analysis()
    stage_15_validation_and_llm_prompts()
    stage_16_finalize()

    elapsed = (time.time() - start) / 60
    R.emit(f"\nFINAL V3 PIPELINE COMPLETE in {elapsed:.1f} minutes.")
    R.emit(f"MASTER REPORT: {MASTER_REPORT_PATH}")
    R.emit(f"FINAL OPPORTUNITY TABLE: {FINAL_OPPORTUNITY_TABLE}")
    R.emit(f"FINAL QA: {FINAL_QA_PATH}")
    R.emit(f"OUTPUT ROOT: {OUTPUT_ROOT}")


# %% CELL: FINAL_V4_EVIDENCE_GATED
# ============================================================
# FINAL V4 — EVIDENCE-GATED TWO-STAGE OPPORTUNITY ASSIGNMENT
#
# This version replaces direct Document -> (Field × Function) semantic labeling
# with a defensible two-stage evidence architecture:
#
#   1) FIELD:
#      A patent can enter a reference field only if that representative CPC class
#      is actually present in the patent-family CPC evidence.
#      When multiple representative fields are present, document-to-field semantic
#      similarity is used ONLY as a tie-breaker.
#
#   2) FUNCTION:
#      First preference = directly extracted normalized Action–Object function.
#      If no selected function was directly extracted, sentence-level semantic
#      matching is allowed only when it passes an empirically calibrated similarity
#      AND top-1/top-2 margin threshold.
#
#   3) OPPORTUNITY:
#      Only patents with BOTH field evidence and function evidence enter:
#         Opportunity = Evidence-supported Field ∩ Evidence-supported Function
#
# This directly solves the prior problem where a document could be semantically
# close to a combined Field×Function label without sentence/function evidence.
# ============================================================

PIPELINE_REVISION = "FINAL_V4_1_QA_FIXED_2026_09_14"

PRIMARY_MIN_CLUSTER_SIZE = 10
FUNCTION_CALIBRATION_MIN_PRECISION = 0.85
FUNCTION_CALIBRATION_MIN_DOCS = 20
FUNCTION_SIM_THRESHOLD_FALLBACK = 0.45
FUNCTION_MARGIN_THRESHOLD_FALLBACK = 0.02
FUNCTION_THRESHOLD_GRID = [round(x, 3) for x in np.arange(0.30, 0.701, 0.025)]
FUNCTION_MARGIN_GRID = [round(x, 3) for x in np.arange(0.00, 0.101, 0.01)]

MECHANISM_MIN_FAMILY_SUPPORT = 2
KEY_PATENTS_PER_CLUSTER = 10
KEY_SENTENCES_PER_PATENT = 5

# V4-specific artifacts
V4_SENTENCES = DATA_DIR / "11_all_patent_sentences_FINAL_V4.parquet"
V4_SENT_EMB = EMBED_DIR / "11_all_patent_sentence_embeddings_FINAL_V4.npy"
V4_FUNCTION_CALIBRATION = DATA_DIR / "11_function_semantic_calibration_FINAL_V4.parquet"
V4_LABELS = DATA_DIR / "11_evidence_gated_labels_FINAL_V4.parquet"
ZS_LABELS = V4_LABELS
V4_ASSIGN = DATA_DIR / "11_evidence_gated_assignments_FINAL_V4.parquet"
ZS_ASSIGN = V4_ASSIGN

CLUSTER_METRICS = DATA_DIR / "13_cluster_metrics_FINAL_V4.parquet"

KEY_PATENTS_PATH = DATA_DIR / "14_key_patents_FINAL_V4.parquet"
KEY_SENTENCES_PATH = DATA_DIR / "14_key_sentences_FINAL_V4.parquet"
KEY_NODES_PATH = DATA_DIR / "14_key_nodes_FINAL_V4.parquet"
KEY_MECHANISMS_SUPPORTED_PATH = DATA_DIR / "14_supported_mechanisms_FINAL_V4.parquet"
KEY_MECHANISMS_SINGLETON_PATH = DATA_DIR / "14_singleton_mechanisms_FINAL_V4.parquet"

FINAL_OPPORTUNITY_TABLE = DATA_DIR / "15_FINAL_OPPORTUNITY_TABLE_V4.parquet"
FINAL_QA_PATH = DATA_DIR / "15_FINAL_QA_V4.parquet"
LLM_PROMPT_INDEX = DATA_DIR / "15_llm_prompt_index_FINAL_V4.parquet"

NETWORK_DIR_V4 = NETWORK_DIR / "FINAL_V4"
NETWORK_DIR_V4.mkdir(parents=True, exist_ok=True)

V4_MIGRATION_MARKER = CHECKPOINT_DIR / "FINAL_V4_MIGRATED.json"

def _archive_superseded_v3_outputs_once():
    if V4_MIGRATION_MARKER.exists():
        return

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    archive = ARCHIVE_DIR / f"pre_FINAL_V4_{stamp}"
    archive.mkdir(parents=True, exist_ok=True)

    prefixes = ("11_", "13_", "14_", "15_", "16_")

    for root in [TABLE_DIR, TABLE_TEXT_DIR, FIG_DIR]:
        dst = archive / root.name
        dst.mkdir(parents=True, exist_ok=True)
        for p in list(root.glob("*")):
            if p.is_file() and p.name.startswith(prefixes):
                shutil.move(str(p), str(dst / p.name))

    dst = archive / "data"
    dst.mkdir(parents=True, exist_ok=True)
    for p in list(DATA_DIR.glob("*")):
        if p.is_file() and p.name.startswith(("11_", "13_", "14_", "15_", "16_")):
            shutil.move(str(p), str(dst / p.name))

    if NETWORK_DIR.exists():
        ndst = archive / "networks"
        ndst.mkdir(parents=True, exist_ok=True)
        for p in list(NETWORK_DIR.iterdir()):
            if p.name == "FINAL_V4":
                continue
            shutil.move(str(p), str(ndst / p.name))

    if LLM_DIR.exists():
        ldst = archive / "llm"
        ldst.mkdir(parents=True, exist_ok=True)
        for p in list(LLM_DIR.iterdir()):
            shutil.move(str(p), str(ldst / p.name))

    atomic_write_json(V4_MIGRATION_MARKER, {
        "revision": PIPELINE_REVISION,
        "migrated_at": datetime.now(timezone.utc).isoformat(),
        "archive": str(archive),
        "note": "Superseded V3 downstream outputs archived; stable preprocessing/embeddings/reference fields/functions/ROS/TN corpus retained."
    })
    R.emit(f"[FINAL V4 migration] Archived superseded downstream outputs -> {archive}")

_archive_superseded_v3_outputs_once()

R.update_section(
    "v4_final_architecture",
    "Final evidence-gated opportunity architecture",
    f"""
The final analysis replaces direct combined-label assignment with an **evidence-gated two-stage architecture**.
Reference-field membership requires observed CPC evidence in the patent family. Functional membership uses
direct normalized Action–Object extraction where available; otherwise sentence-level semantic matching is allowed
only when it passes empirically calibrated similarity and top-1/top-2 margin thresholds.

A patent enters an opportunity cluster only when **both** evidence gates are satisfied:
`Opportunity = supported reference field ∩ supported function`.
"""
)


# ============================================================
# Sentence evidence + function calibration helpers
# ============================================================

def _build_sentence_table(df):
    rows = []
    sid = 0
    for _, r in tqdm(df.iterrows(), total=len(df), desc="Building sentence evidence corpus"):
        sents = sentence_split_simple(r["document_text"])
        if not sents:
            txt = str(r["document_text"]).strip()
            if txt:
                sents = [txt[:1000]]
        for sent in sents:
            rows.append({
                "sentence_id": sid,
                "doc_idx": int(r["doc_idx"]),
                "family_id": r["family_id"],
                "sentence": sent,
            })
            sid += 1
    return pd.DataFrame(rows)

def _function_calibration_search(calibration_rows):
    """
    calibration_rows columns:
      true_function, predicted_function, best_similarity, margin
    """
    if len(calibration_rows) < FUNCTION_CALIBRATION_MIN_DOCS:
        return (
            FUNCTION_SIM_THRESHOLD_FALLBACK,
            FUNCTION_MARGIN_THRESHOLD_FALLBACK,
            pd.DataFrame([{
                "similarity_threshold": FUNCTION_SIM_THRESHOLD_FALLBACK,
                "margin_threshold": FUNCTION_MARGIN_THRESHOLD_FALLBACK,
                "accepted": 0,
                "correct_accepted": 0,
                "precision": np.nan,
                "recall": np.nan,
                "f1": np.nan,
                "selection_note": "Fallback because calibration subset was too small"
            }])
        )

    rows = []
    total = len(calibration_rows)

    for sth in FUNCTION_THRESHOLD_GRID:
        for mth in FUNCTION_MARGIN_GRID:
            accepted = (
                (calibration_rows["best_similarity"] >= sth)
                & (calibration_rows["margin"] >= mth)
            )
            n_acc = int(accepted.sum())
            correct = (
                accepted
                & (
                    calibration_rows["predicted_function"]
                    == calibration_rows["true_function"]
                )
            )
            n_correct = int(correct.sum())

            precision = n_correct / n_acc if n_acc else np.nan
            recall = n_correct / total if total else np.nan
            if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0:
                f1 = 2 * precision * recall / (precision + recall)
            else:
                f1 = np.nan

            rows.append({
                "similarity_threshold": sth,
                "margin_threshold": mth,
                "accepted": n_acc,
                "correct_accepted": n_correct,
                "precision": precision,
                "recall": recall,
                "f1": f1,
            })

    grid = pd.DataFrame(rows)

    min_accepted = max(
        10,
        int(round(0.10 * total))
    )
    admissible = grid[
        (grid["precision"] >= FUNCTION_CALIBRATION_MIN_PRECISION)
        & (grid["accepted"] >= min_accepted)
    ].copy()

    if len(admissible):
        chosen = admissible.sort_values(
            ["recall","precision","similarity_threshold","margin_threshold"],
            ascending=[False,False,False,False]
        ).iloc[0]
        note = (
            f"Highest recall among configurations with precision >= "
            f"{FUNCTION_CALIBRATION_MIN_PRECISION:.0%} and accepted >= {min_accepted}"
        )
    else:
        finite = grid[grid["f1"].notna()].copy()
        if len(finite):
            chosen = finite.sort_values(
                ["f1","precision","recall"],
                ascending=[False,False,False]
            ).iloc[0]
            note = "No grid point met the precision target; selected maximum F1"
        else:
            return (
                FUNCTION_SIM_THRESHOLD_FALLBACK,
                FUNCTION_MARGIN_THRESHOLD_FALLBACK,
                grid
            )

    grid["selected"] = (
        np.isclose(
            grid["similarity_threshold"],
            float(chosen["similarity_threshold"])
        )
        & np.isclose(
            grid["margin_threshold"],
            float(chosen["margin_threshold"])
        )
    )
    grid["selection_note"] = ""
    grid.loc[grid["selected"], "selection_note"] = note

    return (
        float(chosen["similarity_threshold"]),
        float(chosen["margin_threshold"]),
        grid
    )


# ============================================================
# FINAL Stage 11 — two-stage field/function evidence gating
# ============================================================

def stage_11_zeroshot_assignment():
    stage = "11_evidence_gated_assignment_FINAL_V4"
    stage_banner(stage)

    required = [
        V4_SENTENCES,
        V4_SENT_EMB,
        V4_FUNCTION_CALIBRATION,
        V4_LABELS,
        V4_ASSIGN,
    ]
    if stage_done(stage, required):
        R.emit("FINAL V4 checkpoint found — evidence-gated assignment skipped.")
        return

    df = pd.read_parquet(FAMILY_PARQUET)
    doc_emb = np.asarray(
        np.load(PATENT_EMB, mmap_mode="r"),
        dtype=np.float32
    )
    model = get_embedder()

    # --------------------------------------------------------
    # A. Representative field set
    # --------------------------------------------------------
    reps_fields = pd.read_parquet(REP_FIELDS_PATH)
    fields = (
        reps_fields.sort_values("reference_field_score", ascending=False)
        .drop_duplicates("cpc_class")
        .copy()
    )
    if N_REP_REFERENCE_FIELDS:
        fields = fields.head(N_REP_REFERENCE_FIELDS)

    fields = fields[[
        "cpc_class","cpc_title","reference_field_score"
    ]].reset_index(drop=True)

    field_texts = [
        f"{r.cpc_class}: {r.cpc_title}"
        for r in fields.itertuples()
    ]
    field_vecs = model.encode(
        field_texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype("float32")

    field_idx = {
        row["cpc_class"]: i
        for i, row in fields.iterrows()
    }

    # --------------------------------------------------------
    # B. Representative function set
    # --------------------------------------------------------
    functions = pd.read_parquet(REP_FUNCTIONS).copy()
    functions = functions[[
        "function_rank","prototype_function",
        "normalized_frequency","normalized_patent_families",
        "function_score"
    ]].reset_index(drop=True)

    fn_names = functions["prototype_function"].tolist()
    fn_index = {
        fn: i for i, fn in enumerate(fn_names)
    }

    fn_vecs = model.encode(
        fn_names,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype("float32")

    # --------------------------------------------------------
    # C. Build / embed all patent sentences
    # --------------------------------------------------------
    if V4_SENTENCES.exists():
        sent_df = pd.read_parquet(V4_SENTENCES)
    else:
        sent_df = _build_sentence_table(df)
        sent_df.to_parquet(V4_SENTENCES, index=False)
        save_table(
            sent_df,
            "11_all_patent_sentences_FINAL_V4",
            print_full=PRINT_FULL_ANALYTICAL_TABLES
        )

    if V4_SENT_EMB.exists():
        sent_emb = np.asarray(
            np.load(V4_SENT_EMB, mmap_mode="r"),
            dtype=np.float32
        )
    else:
        path = encode_texts_sharded(
            sent_df["sentence"].tolist(),
            "11_all_patent_sentence_embeddings_FINAL_V4",
            shard_size=5000
        )
        if path != V4_SENT_EMB:
            shutil.copy2(path, V4_SENT_EMB)
        sent_emb = np.asarray(
            np.load(V4_SENT_EMB, mmap_mode="r"),
            dtype=np.float32
        )

    # Sentence -> function similarities; only 8 selected functions, so compact.
    sent_fn_sims = (
        sent_emb @ fn_vecs.T
    ).astype("float32")

    # For each document and each selected function:
    # maximum sentence similarity and the evidence sentence id.
    doc_fn_rows = []
    for doc_idx, grp in sent_df.groupby("doc_idx", sort=False):
        ids = grp["sentence_id"].to_numpy(dtype=int)
        local = sent_fn_sims[ids]
        best_local_pos = np.argmax(local, axis=0)
        best_sims = local[best_local_pos, np.arange(len(fn_names))]
        best_sentence_ids = ids[best_local_pos]

        rec = {
            "doc_idx": int(doc_idx)
        }
        for j, fn in enumerate(fn_names):
            rec[f"fn_sim_{j}"] = float(best_sims[j])
            rec[f"fn_sid_{j}"] = int(best_sentence_ids[j])
        doc_fn_rows.append(rec)

    doc_fn = pd.DataFrame(doc_fn_rows).set_index("doc_idx")

    # --------------------------------------------------------
    # D. Direct parser evidence
    # --------------------------------------------------------
    rawf = pd.read_parquet(RAW_FUNCTIONS)
    fmap = pd.read_parquet(FUNCTION_MAP)

    rawf = rawf.merge(
        fmap[[
            "action_object",
            "prototype_function"
        ]].drop_duplicates(),
        on="action_object",
        how="left"
    )
    rawf = rawf[
        rawf["prototype_function"].isin(fn_names)
    ].copy()

    direct_candidates = (
        rawf.groupby(
            ["doc_idx","prototype_function"]
        )
        .agg(
            direct_instances=("family_id","size"),
            direct_sentences=("sentence","nunique"),
            first_direct_sentence=("sentence","first"),
            first_mechanism=("mechanism","first"),
        )
        .reset_index()
    )

    # Documents with exactly one direct representative function are used as a
    # silver-standard calibration subset for sentence-level fallback.
    direct_function_sets = (
        direct_candidates.groupby("doc_idx")["prototype_function"]
        .apply(lambda s: sorted(set(s)))
        .to_dict()
    )

    calibration_rows = []
    for doc_idx, funcs in direct_function_sets.items():
        if len(funcs) != 1:
            continue
        if doc_idx not in doc_fn.index:
            continue

        true_fn = funcs[0]
        sims = np.array([
            doc_fn.loc[doc_idx, f"fn_sim_{j}"]
            for j in range(len(fn_names))
        ], dtype=float)

        order = np.argsort(sims)
        best_j = int(order[-1])
        second_j = int(order[-2]) if len(order) > 1 else best_j
        calibration_rows.append({
            "doc_idx": int(doc_idx),
            "true_function": true_fn,
            "predicted_function": fn_names[best_j],
            "best_similarity": float(sims[best_j]),
            "second_similarity": float(sims[second_j]),
            "margin": float(
                sims[best_j] - sims[second_j]
            ),
            "correct_top1": fn_names[best_j] == true_fn,
        })

    calibration = pd.DataFrame(calibration_rows)

    sim_th, margin_th, calibration_grid = _function_calibration_search(
        calibration
    )
    calibration.to_parquet(
        V4_FUNCTION_CALIBRATION,
        index=False
    )
    save_table(
        calibration,
        "11_function_calibration_silver_standard_FINAL_V4",
        print_full=True
    )
    save_table(
        calibration_grid,
        "11_function_calibration_grid_FINAL_V4",
        print_full=True
    )

    # --------------------------------------------------------
    # E. Evidence-gated assignment per patent
    # --------------------------------------------------------
    sent_lookup = dict(zip(
        sent_df["sentence_id"],
        sent_df["sentence"]
    ))
    function_score_map = dict(zip(
        functions["prototype_function"],
        functions["function_score"]
    ))

    # direct evidence lookup
    direct_grouped = {
        int(doc_idx): grp.copy()
        for doc_idx, grp in direct_candidates.groupby("doc_idx")
    }

    assignment_rows = []

    for _, r in tqdm(
        df.iterrows(),
        total=len(df),
        desc="Evidence-gated Field ∩ Function assignment"
    ):
        did = int(r["doc_idx"])

        # ---------- FIELD GATE: actual CPC evidence ----------
        patent_classes = sorted(set(filter(
            None,
            (
                cpc_class(x)
                for x in parse_cpc_codes(
                    r["cpc_classifications"]
                )
            )
        )))

        eligible_fields = [
            cc for cc in patent_classes
            if cc in field_idx
        ]

        field_assigned = False
        field_code = None
        field_name = None
        field_sim = np.nan
        field_margin = np.nan
        field_evidence = ""

        if eligible_fields:
            sims = []
            for cc in eligible_fields:
                j = field_idx[cc]
                sims.append(
                    (
                        cc,
                        float(np.dot(doc_emb[did], field_vecs[j]))
                    )
                )
            sims.sort(
                key=lambda x: x[1],
                reverse=True
            )
            field_code, field_sim = sims[0]
            field_name = fields.loc[
                fields["cpc_class"] == field_code,
                "cpc_title"
            ].iloc[0]
            if len(sims) > 1:
                field_margin = float(
                    sims[0][1] - sims[1][1]
                )
            field_evidence = (
                "Observed representative CPC class in patent family"
            )
            field_assigned = True

        # ---------- FUNCTION GATE ----------
        function_assigned = False
        function_name = None
        function_source = None
        function_sim = np.nan
        function_margin = np.nan
        evidence_sentence_id = None
        evidence_sentence = ""
        evidence_mechanism = ""
        direct_instance_count = 0

        if did in doc_fn.index:
            sims = np.array([
                doc_fn.loc[did, f"fn_sim_{j}"]
                for j in range(len(fn_names))
            ], dtype=float)

            order = np.argsort(sims)
            global_best_j = int(order[-1])
            global_second_j = int(order[-2]) if len(order) > 1 else global_best_j
            global_margin = float(
                sims[global_best_j]
                - sims[global_second_j]
            )

            if did in direct_grouped:
                dg = direct_grouped[did].copy()

                # Choose among DIRECTLY EXTRACTED selected functions using
                # sentence-semantic support first, then direct count, then function score.
                scored = []
                for _, dr in dg.iterrows():
                    fn = dr["prototype_function"]
                    j = fn_index[fn]
                    scored.append({
                        "function": fn,
                        "fn_j": j,
                        "sentence_similarity": float(sims[j]),
                        "direct_instances": int(dr["direct_instances"]),
                        "function_score": float(function_score_map.get(fn, 0.0)),
                        "direct_sentence": dr["first_direct_sentence"],
                        "direct_mechanism": dr["first_mechanism"],
                    })
                sfd = pd.DataFrame(scored).sort_values(
                    [
                        "sentence_similarity",
                        "direct_instances",
                        "function_score"
                    ],
                    ascending=[False,False,False]
                )
                best = sfd.iloc[0]
                function_name = best["function"]
                j = int(best["fn_j"])
                function_sim = float(sims[j])

                other = [
                    sims[k]
                    for k in range(len(fn_names))
                    if k != j
                ]
                second = max(other) if other else function_sim
                function_margin = float(
                    function_sim - second
                )

                function_source = "direct_parser"
                direct_instance_count = int(
                    best["direct_instances"]
                )
                evidence_sentence = str(
                    best["direct_sentence"]
                )
                evidence_mechanism = str(
                    best["direct_mechanism"]
                    if pd.notna(best["direct_mechanism"])
                    else ""
                )
                # sentence id closest by exact text if available
                matches = sent_df[
                    (sent_df["doc_idx"] == did)
                    & (
                        sent_df["sentence"]
                        == evidence_sentence
                    )
                ]
                if len(matches):
                    evidence_sentence_id = int(
                        matches.iloc[0]["sentence_id"]
                    )
                else:
                    evidence_sentence_id = int(
                        doc_fn.loc[
                            did,
                            f"fn_sid_{j}"
                        ]
                    )
                    evidence_sentence = sent_lookup.get(
                        evidence_sentence_id,
                        evidence_sentence
                    )
                function_assigned = True

            else:
                # Sentence-semantic fallback: must pass BOTH calibrated gates.
                best_j = global_best_j
                if (
                    sims[best_j] >= sim_th
                    and global_margin >= margin_th
                ):
                    function_name = fn_names[best_j]
                    function_sim = float(
                        sims[best_j]
                    )
                    function_margin = global_margin
                    function_source = "semantic_sentence"
                    evidence_sentence_id = int(
                        doc_fn.loc[
                            did,
                            f"fn_sid_{best_j}"
                        ]
                    )
                    evidence_sentence = sent_lookup.get(
                        evidence_sentence_id,
                        ""
                    )
                    function_assigned = True

        # ---------- INTERSECTION ----------
        assigned = (
            field_assigned
            and function_assigned
        )

        assignment_rows.append({
            "doc_idx": did,
            "family_id": r["family_id"],
            "lens_id": r.get("lens_id"),
            "title": r.get("title"),

            "field_assigned": field_assigned,
            "field_cpc_class": field_code,
            "reference_field": field_name,
            "field_similarity": field_sim,
            "field_margin": field_margin,
            "field_evidence": field_evidence,

            "function_assigned": function_assigned,
            "function": function_name,
            "function_evidence_source": function_source,
            "function_similarity": function_sim,
            "function_margin": function_margin,
            "direct_function_instances": direct_instance_count,
            "function_evidence_sentence_id": evidence_sentence_id,
            "function_evidence_sentence": evidence_sentence,
            "function_evidence_mechanism": evidence_mechanism,

            "assigned": assigned,
        })

    out = pd.DataFrame(assignment_rows)

    # Create deterministic Field × Function labels.
    label_rows = []
    lid = 0
    for _, fr in fields.iterrows():
        for _, fnr in functions.iterrows():
            label_rows.append({
                "label_id": lid,
                "cpc_class": fr["cpc_class"],
                "reference_field": fr["cpc_title"],
                "function_rank": int(fnr["function_rank"]),
                "function": fnr["prototype_function"],
                "label_text": (
                    f"{fr['cpc_title']} that "
                    f"{fnr['prototype_function']}"
                ),
            })
            lid += 1

    labels = pd.DataFrame(label_rows)
    labels.to_parquet(
        V4_LABELS,
        index=False
    )
    save_table(
        labels,
        "11_evidence_gated_labels_FINAL_V4",
        print_full=True
    )

    label_map = {
        (r["cpc_class"], r["function"]):
        (
            int(r["label_id"]),
            r["label_text"]
        )
        for _, r in labels.iterrows()
    }

    label_ids = []
    label_texts = []
    combined_fit = []
    combined_margin = []

    for _, r in out.iterrows():
        if not r["assigned"]:
            label_ids.append(-1)
            label_texts.append(None)
            combined_fit.append(np.nan)
            combined_margin.append(np.nan)
            continue

        lid, ltxt = label_map[
            (
                r["field_cpc_class"],
                r["function"]
            )
        ]
        label_ids.append(lid)
        label_texts.append(ltxt)

        # Combined fit is descriptive only; assignment itself is evidence-gated.
        combined_fit.append(
            float(
                np.nanmean([
                    r["field_similarity"],
                    r["function_similarity"]
                ])
            )
        )

        margins = [
            x for x in [
                r["field_margin"],
                r["function_margin"]
            ]
            if pd.notna(x)
        ]
        combined_margin.append(
            float(min(margins))
            if margins else np.nan
        )

    out["zeroshot_label_id"] = label_ids
    out["label_id"] = label_ids
    out["label_text"] = label_texts
    out["cpc_class"] = out["field_cpc_class"]
    out["zeroshot_similarity"] = combined_fit
    out["semantic_margin"] = combined_margin

    out.to_parquet(
        V4_ASSIGN,
        index=False
    )
    save_table(
        out,
        "11_evidence_gated_assignments_FINAL_V4",
        print_full=True
    )

    # --------------------------------------------------------
    # F. Assignment funnel + diagnostics
    # --------------------------------------------------------
    total = len(out)
    field_n = int(out["field_assigned"].sum())
    direct_n = int(
        (
            out["function_evidence_source"]
            == "direct_parser"
        ).sum()
    )
    semantic_n = int(
        (
            out["function_evidence_source"]
            == "semantic_sentence"
        ).sum()
    )
    fn_n = int(out["function_assigned"].sum())
    final_n = int(out["assigned"].sum())

    funnel = pd.DataFrame([
        {
            "stage": "Total patent families",
            "count": total,
            "share_of_total": 1.0,
        },
        {
            "stage": "Field evidence: representative CPC class observed",
            "count": field_n,
            "share_of_total": field_n / total,
        },
        {
            "stage": "Function evidence: any accepted source",
            "count": fn_n,
            "share_of_total": fn_n / total,
        },
        {
            "stage": "Direct parser function evidence",
            "count": direct_n,
            "share_of_total": direct_n / total,
        },
        {
            "stage": "Semantic sentence fallback function evidence",
            "count": semantic_n,
            "share_of_total": semantic_n / total,
        },
        {
            "stage": "Final Field ∩ Function intersection",
            "count": final_n,
            "share_of_total": final_n / total,
        },
    ])
    save_table(
        funnel,
        "11_assignment_funnel_FINAL_V4",
        print_full=True
    )

    assigned = out[out["assigned"]].copy()
    heat = pd.crosstab(
        assigned["reference_field"],
        assigned["function"]
    )
    heat.to_csv(
        TABLE_DIR / "11_field_function_intersection_counts_FINAL_V4.csv"
    )

    fig, ax = plt.subplots(
        figsize=(
            max(10, 1.25*max(1, heat.shape[1])),
            max(6, .65*max(1, heat.shape[0]))
        )
    )
    im = ax.imshow(
        heat.values,
        aspect="auto"
    )
    ax.set_xticks(
        np.arange(heat.shape[1])
    )
    ax.set_xticklabels(
        heat.columns,
        rotation=55,
        ha="right",
        fontsize=8
    )
    ax.set_yticks(
        np.arange(heat.shape[0])
    )
    ax.set_yticklabels(
        heat.index,
        fontsize=8
    )
    ax.set_xlabel(
        "Evidence-supported function"
    )
    ax.set_ylabel(
        "Observed representative CPC field"
    )
    ax.set_title(
        "Final evidence-gated Field × Function intersections"
    )
    fig.colorbar(
        im,
        ax=ax,
        label="Patent families"
    )

    top_cells = (
        assigned.groupby([
            "reference_field","function"
        ])
        .size()
        .reset_index(name="patent_families")
        .sort_values(
            "patent_families",
            ascending=False
        )
    )

    fig_text = (
        f"Final evidence-gated assignment retained {final_n:,}/{total:,} "
        f"patent families ({final_n/total:.1%}).\n"
        f"Field evidence coverage={field_n/total:.1%}; "
        f"function evidence coverage={fn_n/total:.1%}; "
        f"direct parser function evidence={direct_n:,}; "
        f"semantic sentence fallback={semantic_n:,}.\n"
        f"Calibrated semantic fallback thresholds: similarity>={sim_th:.3f}, "
        f"margin>={margin_th:.3f}.\n\n"
        "Largest supported intersections:\n"
        + top_cells.head(25).to_string(index=False)
    )
    save_figure(
        fig,
        "11_field_function_intersections_FINAL_V4",
        fig_text,
        top_cells
    )

    # Calibration diagnostic figure
    if len(calibration_grid):
        pivot_best = (
            calibration_grid.sort_values(
                ["selected","f1"],
                ascending=[False,False]
            )
            .head(25)
        )
        fig, ax = plt.subplots(
            figsize=(10,6)
        )
        for mth, grp in calibration_grid.groupby(
            "margin_threshold"
        ):
            if mth in {
                0.0,
                margin_th,
                0.02,
                0.05,
                0.10
            }:
                ax.plot(
                    grp["similarity_threshold"],
                    grp["precision"],
                    marker="o",
                    label=f"precision, margin≥{mth:.2f}"
                )
        ax.axvline(
            sim_th,
            linestyle="--",
            label=f"selected similarity={sim_th:.3f}"
        )
        ax.set_ylim(0,1.05)
        ax.set_xlabel(
            "Sentence-function similarity threshold"
        )
        ax.set_ylabel(
            "Calibration precision"
        )
        ax.set_title(
            "Function semantic-fallback calibration"
        )
        ax.legend(fontsize=8)
        ax.grid(alpha=.25)

        save_figure(
            fig,
            "11_function_semantic_calibration_FINAL_V4",
            f"Silver-standard calibration docs={len(calibration):,}. "
            f"Selected similarity threshold={sim_th:.3f}, margin threshold={margin_th:.3f}.\n\n"
            + pivot_best.to_string(index=False),
            calibration_grid
        )

    body = f"""
The final assignment architecture uses two independent evidence gates.

**Field gate.** A patent can enter a representative field only when the corresponding CPC class is explicitly
present in the patent-family classifications. Semantic similarity is used only to resolve ties when more than one
representative field is present.

**Function gate.** Direct normalized Action–Object extraction has priority. Patents without direct selected-function
evidence may enter through sentence-level semantic evidence only when both a similarity threshold
(**{sim_th:.3f}**) and a top-1/top-2 margin threshold (**{margin_th:.3f}**) are satisfied. These thresholds are
calibrated against patents with exactly one directly extracted selected function.

The final opportunity corpus contains **{final_n:,}/{total:,} patent families ({final_n/total:.1%})** that pass
both gates. This makes every opportunity membership traceable to observed CPC field evidence and explicit
function-level textual evidence.
"""
    R.update_section(
        "11_zeroshot",
        "Final evidence-gated Field × Function assignment",
        body
    )
    R.emit(body)

    mark_stage_done(
        stage,
        required,
        extra={
            "field_evidence_patents": field_n,
            "function_evidence_patents": fn_n,
            "direct_function_patents": direct_n,
            "semantic_fallback_patents": semantic_n,
            "final_assigned_patents": final_n,
            "function_similarity_threshold": sim_th,
            "function_margin_threshold": margin_th,
            "calibration_docs": len(calibration),
        }
    )


# ============================================================
# FINAL Stage 13 — opportunity evaluation on evidence intersections
# ============================================================

def stage_13_cluster_metrics():
    stage = "13_cluster_metrics_FINAL_V4"
    stage_banner(stage)

    if stage_done(stage, [CLUSTER_METRICS]):
        R.emit("FINAL V4 checkpoint found — cluster metrics skipped.")
        return

    df = pd.read_parquet(FAMILY_PARQUET)
    assign = pd.read_parquet(V4_ASSIGN)
    ros = pd.read_parquet(ROS_PATH)
    labels = pd.read_parquet(V4_LABELS)

    emb = np.asarray(
        np.load(PATENT_EMB, mmap_mode="r"),
        dtype=np.float32
    )
    sents = pd.read_parquet(FIELD_SENTENCES)
    sent_emb = np.asarray(
        np.load(FIELD_SENT_EMB, mmap_mode="r"),
        dtype=np.float32
    )

    work = (
        df.merge(
            assign[[
                "doc_idx",
                "zeroshot_label_id",
                "zeroshot_similarity",
                "semantic_margin",
                "field_similarity",
                "function_similarity",
                "function_evidence_source",
                "assigned"
            ]],
            on="doc_idx"
        )
        .merge(
            ros[[
                "doc_idx",
                "recycling_orientation_raw"
            ]],
            on="doc_idx"
        )
    )
    work = work[
        work["assigned"]
    ].copy()

    analysis_dt = pd.Timestamp(
        ANALYSIS_DATE
    )
    pub = pd.to_datetime(
        work["family_earliest_publication_date"],
        errors="coerce"
    )
    work["citation_age_years"] = (
        (analysis_dt - pub).dt.days
        / 365.25
    ).clip(
        lower=1.0
    ).fillna(1.0)

    work["cited_by_patent_count"] = pd.to_numeric(
        work["cited_by_patent_count"],
        errors="coerce"
    ).fillna(0)

    work["simple_family_size"] = pd.to_numeric(
        work["simple_family_size"],
        errors="coerce"
    ).fillna(1).clip(lower=1)

    cluster_ids = sorted(
        work["zeroshot_label_id"]
        .unique()
        .astype(int)
    )

    centroids = {}
    for cid in cluster_ids:
        ids = work.loc[
            work["zeroshot_label_id"] == cid,
            "doc_idx"
        ].astype(int).values

        centroids[cid] = l2_normalize(
            emb[ids].mean(
                axis=0,
                keepdims=True
            )
        )[0]

    model = get_embedder()
    target_vec = model.encode(
        [TARGET_TECHNOLOGY],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0].astype("float32")

    rows = []
    for cid, grp in work.groupby(
        "zeroshot_label_id"
    ):
        cid = int(cid)
        P = len(grp)
        citation_rate = (
            grp["cited_by_patent_count"]
            / grp["citation_age_years"]
        )

        direct_share = float(
            (
                grp["function_evidence_source"]
                == "direct_parser"
            ).mean()
        )

        rows.append({
            "cluster_id": cid,
            "cluster_size": int(P),
            "size_factor": float(
                1 + math.log(max(P,1))
            ),
            "median_age_normalized_forward_citations": float(
                np.median(citation_rate)
            ),
            "median_simple_family_size": float(
                np.median(
                    grp["simple_family_size"]
                )
            ),
            "mean_field_similarity": float(
                grp["field_similarity"].mean()
            ),
            "mean_function_similarity": float(
                grp["function_similarity"].mean()
            ),
            "mean_combined_fit": float(
                grp["zeroshot_similarity"].mean()
            ),
            "mean_semantic_margin": float(
                grp["semantic_margin"].dropna().mean()
            ),
            "direct_function_evidence_share": direct_share,
            "semantic_fallback_share": float(
                1 - direct_share
            ),
            "TR_raw_similarity": float(
                np.dot(
                    centroids[cid],
                    target_vec
                )
            ),
            "ROS_raw_mean": float(
                grp["recycling_orientation_raw"].mean()
            ),
        })

    metrics = pd.DataFrame(rows)

    # Correct TV: z-score medians first; cluster-size factor second.
    metrics["citation_median_z"] = safe_z(
        metrics[
            "median_age_normalized_forward_citations"
        ]
    ).values

    metrics["family_median_z"] = safe_z(
        metrics[
            "median_simple_family_size"
        ]
    ).values

    metrics["citation_component"] = (
        metrics["citation_median_z"]
        * metrics["size_factor"]
    )
    metrics["family_component"] = (
        metrics["family_median_z"]
        * metrics["size_factor"]
    )
    metrics["TV"] = (
        metrics["citation_component"]
        + metrics["family_component"]
    ) / 2

    metrics["TR"] = safe_z(
        metrics["TR_raw_similarity"]
    ).values
    metrics["ROS"] = safe_z(
        metrics["ROS_raw_mean"]
    ).values

    # TN recomputed for final intersection centroids.
    C = np.stack([
        centroids[cid]
        for cid in cluster_ids
    ]).astype("float32")

    sim_matrix = (
        C @ sent_emb.T
    )
    np.save(
        EMBED_DIR / "13_tn_cluster_sentence_similarity_FINAL_V4.npy",
        sim_matrix.astype("float32")
    )

    q3 = float(
        np.quantile(
            sim_matrix,
            0.75
        )
    )

    sent_pub = pd.to_datetime(
        sents["publication_date"],
        errors="coerce"
    )
    sent_age = (
        (analysis_dt - sent_pub).dt.days
        / 365.25
    ).clip(
        lower=1.0
    ).fillna(1.0).values

    weights = 1.0 / sent_age

    tn_rows = []
    for i, cid in enumerate(
        cluster_ids
    ):
        mask = (
            sim_matrix[i] >= q3
        )
        tn_rows.append({
            "cluster_id": cid,
            "TN_overlap_raw": float(
                weights[mask].sum()
            ),
            "TN_matching_sentences": int(
                mask.sum()
            ),
        })

    tn = pd.DataFrame(
        tn_rows
    )
    tn["TN"] = -safe_z(
        tn["TN_overlap_raw"]
    ).values

    metrics = metrics.merge(
        tn,
        on="cluster_id",
        how="left"
    ).merge(
        labels[[
            "label_id",
            "label_text",
            "cpc_class",
            "reference_field",
            "function"
        ]],
        left_on="cluster_id",
        right_on="label_id",
        how="left"
    )

    metrics["key_cluster_all"] = (
        (metrics["TV"] > 0)
        & (metrics["TR"] > 0)
    )

    metrics["primary_key_cluster"] = (
        metrics["key_cluster_all"]
        & (
            metrics["cluster_size"]
            >= PRIMARY_MIN_CLUSTER_SIZE
        )
    )

    metrics["key_cluster"] = (
        metrics["primary_key_cluster"]
    )

    metrics["TV_plus_TR"] = (
        metrics["TV"]
        + metrics["TR"]
    )

    metrics["novelty_orientation"] = np.where(
        metrics["TN"] >= 0,
        "exploration-oriented",
        "exploitation-oriented"
    )

    metrics["recycling_orientation"] = np.where(
        metrics["ROS"] >= 0,
        "end-of-life-recycling-oriented",
        "repair/service-leaning"
    )

    metrics["cluster_support_class"] = np.where(
        metrics["cluster_size"]
        >= PRIMARY_MIN_CLUSTER_SIZE,
        "primary-support",
        "low-support exploratory"
    )

    metrics = metrics.sort_values(
        [
            "primary_key_cluster",
            "key_cluster_all",
            "TV_plus_TR"
        ],
        ascending=[
            False,
            False,
            False
        ]
    ).reset_index(drop=True)

    metrics.to_parquet(
        CLUSTER_METRICS,
        index=False
    )
    save_table(
        metrics,
        "13_cluster_metrics_FINAL_V4",
        print_full=True
    )

    primary = metrics[
        metrics["primary_key_cluster"]
    ].copy()

    emerging = metrics[
        metrics["key_cluster_all"]
        & (~metrics["primary_key_cluster"])
    ].copy()

    fig, ax = plt.subplots(
        figsize=(10,8)
    )
    sc = ax.scatter(
        metrics["TR"],
        metrics["TV"],
        c=metrics["TN"],
        s=45 + 4*np.sqrt(
            metrics["cluster_size"]
        ),
        alpha=.75
    )
    ax.axhline(
        0,
        linewidth=1
    )
    ax.axvline(
        0,
        linewidth=1
    )
    ax.set_xlabel(
        "TR (z-score)"
    )
    ax.set_ylabel(
        "TV (corrected)"
    )
    ax.set_title(
        "Evidence-gated technology-opportunity map"
    )
    fig.colorbar(
        sc,
        ax=ax,
        label="TN"
    )

    for _, r in primary.iterrows():
        ax.annotate(
            f"P{int(r['cluster_id'])}",
            (r["TR"], r["TV"]),
            fontsize=8
        )

    for _, r in emerging.iterrows():
        ax.annotate(
            f"E{int(r['cluster_id'])}",
            (r["TR"], r["TV"]),
            fontsize=8
        )

    text = (
        f"Evidence-gated evaluated intersections={len(metrics)}; "
        f"TN Q3={q3:.4f}; "
        f"high-TV/high-TR={int(metrics['key_cluster_all'].sum())}; "
        f"primary n>={PRIMARY_MIN_CLUSTER_SIZE}: {len(primary)}; "
        f"emerging: {len(emerging)}.\n\n"
        "PRIMARY:\n"
        + (
            primary[[
                "cluster_id",
                "label_text",
                "cluster_size",
                "TV","TR","TN","ROS",
                "direct_function_evidence_share",
                "mean_function_similarity"
            ]].to_string(index=False)
            if len(primary)
            else "None"
        )
        + "\n\nEMERGING:\n"
        + (
            emerging[[
                "cluster_id",
                "label_text",
                "cluster_size",
                "TV","TR","TN","ROS",
                "direct_function_evidence_share",
                "mean_function_similarity"
            ]].to_string(index=False)
            if len(emerging)
            else "None"
        )
    )
    save_figure(
        fig,
        "13_TV_TR_scatter_FINAL_V4",
        text,
        metrics
    )

    body = f"""
TV, TR, TN, and ROS were recomputed after evidence gating, so the evaluated opportunity clusters now contain only
patents with observed representative-CPC field evidence and accepted function-level textual evidence.
The final screen identified **{int(metrics['key_cluster_all'].sum())} high-TV/high-TR opportunity intersections**,
including **{len(primary)} primary opportunities** with at least **{PRIMARY_MIN_CLUSTER_SIZE} patent families**
and **{len(emerging)} smaller emerging/exploratory opportunities**.
"""
    R.update_section(
        "13_metrics",
        "Evidence-gated technology-opportunity evaluation",
        body
    )
    R.emit(body)

    mark_stage_done(
        stage,
        [CLUSTER_METRICS],
        extra={
            "tn_q3": q3,
            "high_tv_tr": int(
                metrics["key_cluster_all"].sum()
            ),
            "primary": len(primary),
            "emerging": len(emerging),
        }
    )


# ============================================================
# FINAL Stage 14 — key patents use actual field+function evidence
# ============================================================

_GENERIC_MECH = [
    r"\bmeans? of (the )?design\b",
    r"\baccording to\b",
    r"\bthe present invention\b",
    r"\bthe invention\b",
    r"\bthe method\b",
    r"\bthe device\b",
    r"\bthe system\b",
]

def _v4_clean_mechanism(x):
    x = clean_phrase(x, 220).lower()
    x = re.sub(
        r"^(by|via|through|using|with|upon|under)\s+",
        "",
        x
    ).strip()
    if len(x) < 8:
        return ""
    for pat in _GENERIC_MECH:
        if re.search(
            pat,
            x,
            flags=re.I
        ):
            return ""
    return x

def _regex_mechanism_from_sentence(sentence):
    s = clean_phrase(
        sentence,
        500
    )
    m = re.search(
        r"\b(?:by|via|through|using|with|upon|under|in response to)\b\s+([^.;]{4,150})",
        s,
        flags=re.I
    )
    if not m:
        return ""
    return _v4_clean_mechanism(
        m.group(1)
    )

def _mean_top(values, n=3):
    vals = sorted(
        [
            float(v)
            for v in values
            if pd.notna(v)
        ],
        reverse=True
    )
    if not vals:
        return 0.0
    return float(
        np.mean(
            vals[:n]
        )
    )

def stage_14_key_cluster_analysis():
    stage = "14_key_cluster_analysis_FINAL_V4"
    stage_banner(stage)

    required = [
        KEY_PATENTS_PATH,
        KEY_SENTENCES_PATH,
        KEY_NODES_PATH,
        KEY_MECHANISMS_SUPPORTED_PATH,
        KEY_MECHANISMS_SINGLETON_PATH,
    ]
    if stage_done(stage, required):
        R.emit("FINAL V4 checkpoint found — key-cluster analysis skipped.")
        return

    df = pd.read_parquet(FAMILY_PARQUET)
    assign = pd.read_parquet(V4_ASSIGN)
    metrics = pd.read_parquet(CLUSTER_METRICS)

    work = df.merge(
        assign[[
            "doc_idx",
            "zeroshot_label_id",
            "field_cpc_class",
            "reference_field",
            "field_similarity",
            "function",
            "function_evidence_source",
            "function_similarity",
            "function_margin",
            "function_evidence_sentence",
            "function_evidence_mechanism",
            "zeroshot_similarity",
            "semantic_margin",
            "assigned"
        ]],
        on="doc_idx"
    )
    work = work[
        work["assigned"]
    ].copy()

    keys = metrics[
        metrics["key_cluster_all"]
    ].sort_values(
        [
            "primary_key_cluster",
            "TV_plus_TR"
        ],
        ascending=[
            False,
            False
        ]
    )

    all_nodes = []
    all_key_patents = []
    all_key_sentences = []
    all_supported = []
    all_singletons = []
    evidence_summary_rows = []

    for _, kr in keys.iterrows():
        cid = int(
            kr["cluster_id"]
        )
        cdir = (
            NETWORK_DIR_V4
            / f"cluster_{cid:03d}"
        )
        cdir.mkdir(
            parents=True,
            exist_ok=True
        )
        done = (
            cdir / "DONE.json"
        )

        if done.exists():
            R.emit(
                f"[FINAL V4 cluster {cid}] cached."
            )
            for fn, collector in [
                ("nodes.parquet", all_nodes),
                ("key_patents.parquet", all_key_patents),
                ("key_sentences.parquet", all_key_sentences),
                ("supported_mechanisms.parquet", all_supported),
                ("singleton_mechanisms.parquet", all_singletons),
            ]:
                p = cdir / fn
                if p.exists():
                    collector.append(
                        pd.read_parquet(p)
                    )
            meta = json.loads(
                done.read_text(
                    encoding="utf-8"
                )
            )
            evidence_summary_rows.append(
                meta["evidence_summary"]
            )
            continue

        cluster_docs = work[
            work["zeroshot_label_id"]
            == cid
        ].copy()

        doc_ids = set(
            cluster_docs[
                "doc_idx"
            ].astype(int)
        )

        # CPC subclass evidence and network
        cpc_by_doc = {}
        for _, r in cluster_docs.iterrows():
            subclasses = sorted(set(filter(
                None,
                (
                    cpc_subclass(x)
                    for x in parse_cpc_codes(
                        r["cpc_classifications"]
                    )
                )
            )))
            cpc_by_doc[
                int(r["doc_idx"])
            ] = subclasses

        cpc_G = build_cooccurrence([
            cpc_by_doc.get(
                did,
                []
            )
            for did in doc_ids
        ])
        cpc_nodes, cpc_edges = network_tables(
            cpc_G,
            cid,
            "CPC_subclass"
        )
        all_nodes.append(
            cpc_nodes
        )

        cpc_cent = (
            cpc_nodes
            .set_index("node")[
                "centrality"
            ]
            .to_dict()
            if len(cpc_nodes)
            else {}
        )

        # Mechanisms come from the actual function-evidence sentence.
        mech_rows = []
        for _, r in cluster_docs.iterrows():
            mech = _v4_clean_mechanism(
                r[
                    "function_evidence_mechanism"
                ]
            )
            if not mech:
                mech = _regex_mechanism_from_sentence(
                    r[
                        "function_evidence_sentence"
                    ]
                )
            if mech:
                mech_rows.append({
                    "cluster_id": cid,
                    "doc_idx": int(
                        r["doc_idx"]
                    ),
                    "family_id": r["family_id"],
                    "mechanism": mech,
                    "evidence_sentence": r[
                        "function_evidence_sentence"
                    ],
                })

        mech_df = pd.DataFrame(
            mech_rows
        )

        if len(mech_df):
            mech_support = (
                mech_df.groupby(
                    "mechanism"
                )
                .agg(
                    patent_families=(
                        "family_id",
                        "nunique"
                    ),
                    instances=(
                        "family_id",
                        "size"
                    ),
                    example_sentence=(
                        "evidence_sentence",
                        "first"
                    ),
                )
                .reset_index()
                .sort_values(
                    [
                        "patent_families",
                        "instances"
                    ],
                    ascending=[
                        False,
                        False
                    ]
                )
            )
            mech_support.insert(
                0,
                "cluster_id",
                cid
            )

            supported = mech_support[
                mech_support[
                    "patent_families"
                ] >= MECHANISM_MIN_FAMILY_SUPPORT
            ].copy()

            singleton = mech_support[
                mech_support[
                    "patent_families"
                ] < MECHANISM_MIN_FAMILY_SUPPORT
            ].copy()
        else:
            cols = [
                "cluster_id",
                "mechanism",
                "patent_families",
                "instances",
                "example_sentence"
            ]
            supported = pd.DataFrame(
                columns=cols
            )
            singleton = pd.DataFrame(
                columns=cols
            )

        supported.to_parquet(
            cdir / "supported_mechanisms.parquet",
            index=False
        )
        singleton.to_parquet(
            cdir / "singleton_mechanisms.parquet",
            index=False
        )

        all_supported.append(
            supported
        )
        all_singletons.append(
            singleton
        )

        # Patent score:
        # 1) CPC centrality support
        # 2) function-evidence similarity
        # 3) combined field/function fit
        # Direct-parser patents are preferred as evidence tier 1;
        # calibrated semantic-fallback patents remain eligible as tier 0.
        p_rows = []
        for _, r in cluster_docs.iterrows():
            did = int(
                r["doc_idx"]
            )
            cpcs = cpc_by_doc.get(
                did,
                []
            )

            cpc_component_raw = _mean_top(
                [
                    cpc_cent.get(
                        x,
                        0.0
                    )
                    for x in cpcs
                ],
                n=3
            )

            source = r[
                "function_evidence_source"
            ]
            evidence_tier = (
                1
                if source
                == "direct_parser"
                else 0
            )

            p_rows.append({
                "cluster_id": cid,
                "cluster_type": (
                    "primary"
                    if bool(
                        kr[
                            "primary_key_cluster"
                        ]
                    )
                    else "emerging"
                ),
                "cluster_label": kr[
                    "label_text"
                ],
                "doc_idx": did,
                "family_id": r[
                    "family_id"
                ],
                "lens_id": r.get(
                    "lens_id"
                ),
                "title": r.get(
                    "title"
                ),
                "applicants": r.get(
                    "applicants"
                ),
                "owners": r.get(
                    "owners"
                ),
                "publication_date": r.get(
                    "publication_date"
                ),
                "field_cpc_class": r[
                    "field_cpc_class"
                ],
                "field_similarity": float(
                    r[
                        "field_similarity"
                    ]
                ),
                "function": r[
                    "function"
                ],
                "function_evidence_source": source,
                "function_similarity": float(
                    r[
                        "function_similarity"
                    ]
                ),
                "function_margin": float(
                    r[
                        "function_margin"
                    ]
                ),
                "function_evidence_sentence": r[
                    "function_evidence_sentence"
                ],
                "combined_fit": float(
                    r[
                        "zeroshot_similarity"
                    ]
                ),
                "combined_margin": float(
                    r[
                        "semantic_margin"
                    ]
                )
                if pd.notna(
                    r[
                        "semantic_margin"
                    ]
                )
                else np.nan,
                "evidence_tier": evidence_tier,
                "cpc_component_raw": cpc_component_raw,
                "cpc_subclasses": " ; ".join(
                    cpcs
                ),
            })

        ptab = pd.DataFrame(
            p_rows
        )

        ptab[
            "cpc_component_z"
        ] = safe_z(
            ptab[
                "cpc_component_raw"
            ]
        ).values

        ptab[
            "function_similarity_z"
        ] = safe_z(
            ptab[
                "function_similarity"
            ]
        ).values

        ptab[
            "combined_fit_z"
        ] = safe_z(
            ptab[
                "combined_fit"
            ]
        ).values

        ptab[
            "key_patent_score"
        ] = (
            ptab[
                "cpc_component_z"
            ]
            + ptab[
                "function_similarity_z"
            ]
            + ptab[
                "combined_fit_z"
            ]
        ) / 3

        ptab = ptab.sort_values(
            [
                "evidence_tier",
                "key_patent_score",
                "function_similarity"
            ],
            ascending=[
                False,
                False,
                False
            ]
        )

        keyp = ptab.head(
            KEY_PATENTS_PER_CLUSTER
        ).copy()

        keyp.to_parquet(
            cdir / "key_patents.parquet",
            index=False
        )
        all_key_patents.append(
            keyp
        )

        # Key sentence is the exact sentence that justified function membership.
        ks_rows = []
        for _, p in keyp.iterrows():
            ks_rows.append({
                "cluster_id": cid,
                "cluster_type": p[
                    "cluster_type"
                ],
                "doc_idx": int(
                    p["doc_idx"]
                ),
                "family_id": p[
                    "family_id"
                ],
                "lens_id": p[
                    "lens_id"
                ],
                "title": p[
                    "title"
                ],
                "key_function": p[
                    "function"
                ],
                "function_evidence_source": p[
                    "function_evidence_source"
                ],
                "function_similarity": p[
                    "function_similarity"
                ],
                "sentence": p[
                    "function_evidence_sentence"
                ],
                "key_patent_score": p[
                    "key_patent_score"
                ],
            })

        ksent = pd.DataFrame(
            ks_rows
        )
        ksent.to_parquet(
            cdir / "key_sentences.parquet",
            index=False
        )
        all_key_sentences.append(
            ksent
        )

        cpc_nodes.to_parquet(
            cdir / "nodes.parquet",
            index=False
        )
        cpc_edges.to_parquet(
            cdir / "edges.parquet",
            index=False
        )

        plot_network(
            cpc_G,
            cpc_nodes,
            f"14_cluster_{cid:03d}_cpc_network_FINAL_V4",
            f"Cluster {cid}: CPC-subclass co-occurrence"
        )

        if len(supported):
            plotm = supported.head(
                15
            ).sort_values(
                "patent_families"
            )
            fig, ax = plt.subplots(
                figsize=(
                    11,
                    max(
                        6,
                        .42*len(plotm)
                    )
                )
            )
            ax.barh(
                plotm[
                    "mechanism"
                ],
                plotm[
                    "patent_families"
                ]
            )
            ax.set_xlabel(
                "Supporting patent families"
            )
            ax.set_title(
                f"Cluster {cid}: recurrent mechanisms"
            )
            save_figure(
                fig,
                f"14_cluster_{cid:03d}_supported_mechanisms_FINAL_V4",
                supported.head(
                    20
                ).to_string(
                    index=False
                ),
                supported
            )

        direct_share = float(
            (
                cluster_docs[
                    "function_evidence_source"
                ]
                == "direct_parser"
            ).mean()
        )

        evidence_summary = {
            "cluster_id": cid,
            "cluster_type": (
                "primary"
                if bool(
                    kr[
                        "primary_key_cluster"
                    ]
                )
                else "emerging"
            ),
            "cluster_label": kr[
                "label_text"
            ],
            "cluster_size": int(
                len(
                    cluster_docs
                )
            ),
            "direct_function_evidence_share": direct_share,
            "semantic_fallback_share": float(
                1 - direct_share
            ),
            "mean_function_similarity": float(
                cluster_docs[
                    "function_similarity"
                ].mean()
            ),
            "mean_field_similarity": float(
                cluster_docs[
                    "field_similarity"
                ].mean()
            ),
            "key_patents": int(
                len(keyp)
            ),
            "supported_mechanisms": int(
                len(supported)
            ),
            "singleton_mechanisms": int(
                len(singleton)
            ),
            "TV": float(
                kr["TV"]
            ),
            "TR": float(
                kr["TR"]
            ),
            "TN": float(
                kr["TN"]
            ),
            "ROS": float(
                kr["ROS"]
            ),
        }

        evidence_summary_rows.append(
            evidence_summary
        )

        atomic_write_json(
            done,
            {
                "evidence_summary": evidence_summary,
                "top_key_patents": (
                    keyp[[
                        "lens_id",
                        "title",
                        "function_evidence_source",
                        "function_similarity",
                        "key_patent_score"
                    ]]
                    .head(5)
                    .to_dict(
                        "records"
                    )
                ),
                "supported_mechanisms": (
                    supported.head(
                        10
                    ).to_dict(
                        "records"
                    )
                ),
            }
        )

        R.emit(
            json.dumps(
                evidence_summary,
                indent=2,
                ensure_ascii=False
            )
        )

    nodes_all = (
        pd.concat(
            all_nodes,
            ignore_index=True
        )
        if all_nodes
        else pd.DataFrame()
    )
    kp_all = (
        pd.concat(
            all_key_patents,
            ignore_index=True
        )
        if all_key_patents
        else pd.DataFrame()
    )
    ks_all = (
        pd.concat(
            all_key_sentences,
            ignore_index=True
        )
        if all_key_sentences
        else pd.DataFrame()
    )
    supp_all = (
        pd.concat(
            all_supported,
            ignore_index=True
        )
        if all_supported
        else pd.DataFrame()
    )
    single_all = (
        pd.concat(
            all_singletons,
            ignore_index=True
        )
        if all_singletons
        else pd.DataFrame()
    )

    nodes_all.to_parquet(
        KEY_NODES_PATH,
        index=False
    )
    kp_all.to_parquet(
        KEY_PATENTS_PATH,
        index=False
    )
    ks_all.to_parquet(
        KEY_SENTENCES_PATH,
        index=False
    )
    supp_all.to_parquet(
        KEY_MECHANISMS_SUPPORTED_PATH,
        index=False
    )
    single_all.to_parquet(
        KEY_MECHANISMS_SINGLETON_PATH,
        index=False
    )

    save_table(
        nodes_all,
        "14_key_nodes_FINAL_V4",
        print_full=True
    )
    save_table(
        kp_all,
        "14_key_patents_FINAL_V4",
        print_full=True
    )
    save_table(
        ks_all,
        "14_key_sentences_FINAL_V4",
        print_full=True
    )
    save_table(
        supp_all,
        "14_supported_mechanisms_FINAL_V4",
        print_full=True
    )
    save_table(
        single_all,
        "14_singleton_mechanisms_FINAL_V4",
        print_full=True
    )

    evidence_df = pd.DataFrame(
        evidence_summary_rows
    )

    save_table(
        evidence_df,
        "14_cluster_evidence_support_FINAL_V4",
        print_full=True
    )

    # enrich cluster metrics with evidence summary
    m = pd.read_parquet(
        CLUSTER_METRICS
    )
    if len(evidence_df):
        m = m.merge(
            evidence_df[[
                "cluster_id",
                "key_patents",
                "supported_mechanisms",
                "singleton_mechanisms"
            ]],
            on="cluster_id",
            how="left"
        )

    m.to_parquet(
        CLUSTER_METRICS,
        index=False
    )
    save_table(
        m,
        "13_cluster_metrics_WITH_EVIDENCE_FINAL_V4",
        print_full=True
    )

    body = f"""
Key-patent evidence is now fully aligned with the evidence-gated opportunity membership.
Every key patent already has an observed representative-CPC field and a traceable function-evidence sentence.
Direct-parser function evidence is ranked ahead of calibrated semantic fallback evidence. Within each evidence tier,
patents are ranked using CPC-network centrality, sentence-function similarity, and combined field/function fit.

Recurrent mechanisms require support from at least **{MECHANISM_MIN_FAMILY_SUPPORT} patent families**; one-family
mechanisms remain available as case examples rather than being described as dominant mechanisms.
"""
    R.update_section(
        "14_networks",
        "Final evidence-aligned key patents and mechanisms",
        body
    )
    R.emit(body)

    mark_stage_done(
        stage,
        required,
        extra={
            "clusters_analyzed": int(
                len(keys)
            ),
            "key_patents": int(
                len(kp_all)
            ),
            "key_sentences": int(
                len(ks_all)
            ),
            "supported_mechanisms": int(
                len(supp_all)
            ),
        }
    )


# ============================================================
# FINAL Stage 15 — final opportunity table and QA
# ============================================================

def stage_15_validation_and_llm_prompts():
    stage = "15_validation_llm_prompts_FINAL_V4"
    stage_banner(stage)

    marker = (
        CHECKPOINT_DIR
        / "15_validation_llm_prompts_FINAL_V4.done"
    )

    required = [
        marker,
        FINAL_OPPORTUNITY_TABLE,
        FINAL_QA_PATH,
        LLM_PROMPT_INDEX,
    ]

    if stage_done(
        stage,
        required
    ):
        R.emit(
            "FINAL V4 checkpoint found — final validation package skipped."
        )
        return

    metrics = pd.read_parquet(
        CLUSTER_METRICS
    )
    assign = pd.read_parquet(
        V4_ASSIGN
    )
    keyp = pd.read_parquet(
        KEY_PATENTS_PATH
    )
    keysent = pd.read_parquet(
        KEY_SENTENCES_PATH
    )
    nodes = pd.read_parquet(
        KEY_NODES_PATH
    )
    supported_mech = pd.read_parquet(
        KEY_MECHANISMS_SUPPORTED_PATH
    )
    calibration = pd.read_parquet(
        V4_FUNCTION_CALIBRATION
    )

    final_opp = metrics[
        metrics["key_cluster_all"]
    ].copy()

    final_opp[
        "opportunity_status"
    ] = np.where(
        final_opp[
            "primary_key_cluster"
        ],
        "Primary",
        "Emerging / exploratory"
    )

    final_opp[
        "interpretation"
    ] = np.select(
        [
            (
                final_opp["TN"] >= 0
            )
            & (
                final_opp["ROS"] >= 0
            ),
            (
                final_opp["TN"] >= 0
            )
            & (
                final_opp["ROS"] < 0
            ),
            (
                final_opp["TN"] < 0
            )
            & (
                final_opp["ROS"] >= 0
            ),
        ],
        [
            "Exploration-oriented and end-of-life-recycling-oriented",
            "Exploration-oriented but repair/service-leaning",
            "Exploitation-oriented and end-of-life-recycling-oriented",
        ],
        default=(
            "Exploitation-oriented and repair/service-leaning"
        )
    )

    cols = [
        "opportunity_status",
        "cluster_id",
        "label_text",
        "reference_field",
        "function",
        "cluster_size",
        "TV","TR","TN","ROS",
        "TV_plus_TR",
        "mean_field_similarity",
        "mean_function_similarity",
        "mean_combined_fit",
        "mean_semantic_margin",
        "direct_function_evidence_share",
        "semantic_fallback_share",
        "key_patents",
        "supported_mechanisms",
        "interpretation",
    ]

    final_opp = final_opp[
        [
            c
            for c in cols
            if c in final_opp.columns
        ]
    ].sort_values(
        [
            "opportunity_status",
            "TV_plus_TR"
        ],
        ascending=[
            True,
            False
        ]
    )

    final_opp.to_parquet(
        FINAL_OPPORTUNITY_TABLE,
        index=False
    )
    save_table(
        final_opp,
        "15_FINAL_OPPORTUNITY_TABLE_V4",
        print_full=True
    )

    # ---------------- QA ----------------
    assigned = assign[
        assign["assigned"]
    ].copy()

    structural_field_ok = (
        assigned[
            "field_cpc_class"
        ].notna()
        & assigned[
            "field_evidence"
        ].notna()
    ).all() if len(assigned) else True

    structural_function_ok = (
        assigned[
            "function"
        ].notna()
        & assigned[
            "function_evidence_sentence"
        ].fillna("")
        .astype(str)
        .str.len()
        .gt(0)
    ).all() if len(assigned) else True

    key_sentence_ok = (
        keyp[
            "function_evidence_sentence"
        ].fillna("")
        .astype(str)
        .str.len()
        .gt(0)
    ).all() if len(keyp) else True

    direct_share = float(
        (
            assigned[
                "function_evidence_source"
            ]
            == "direct_parser"
        ).mean()
    ) if len(assigned) else np.nan

    calibration_accuracy = float(
        calibration[
            "correct_top1"
        ].mean()
    ) if len(calibration) else np.nan

    primary_count = int(
        metrics[
            "primary_key_cluster"
        ].sum()
    )
    emerging_count = int(
        (
            metrics[
                "key_cluster_all"
            ]
            & ~metrics[
                "primary_key_cluster"
            ]
        ).sum()
    )

    # FINAL V4.1: explicit QA schema for Parquet compatibility.
    # Boolean, numeric, and display values are stored in separate typed columns.
    qa = pd.DataFrame([
        {
            "check": "Every final assignment has observed CPC field evidence",
            "value_type": "boolean",
            "value_boolean": bool(structural_field_ok),
            "value_numeric": np.nan,
            "value_display": str(bool(structural_field_ok)),
            "status": ("PASS" if structural_field_ok else "FAIL"),
            "criterion": "100% required",
        },
        {
            "check": "Every final assignment has traceable function-evidence sentence",
            "value_type": "boolean",
            "value_boolean": bool(structural_function_ok),
            "value_numeric": np.nan,
            "value_display": str(bool(structural_function_ok)),
            "status": ("PASS" if structural_function_ok else "FAIL"),
            "criterion": "100% required",
        },
        {
            "check": "Every key patent has traceable function sentence",
            "value_type": "boolean",
            "value_boolean": bool(key_sentence_ok),
            "value_numeric": np.nan,
            "value_display": str(bool(key_sentence_ok)),
            "status": ("PASS" if key_sentence_ok else "FAIL"),
            "criterion": "100% required",
        },
        {
            "check": "Final evidence-gated assignment rate",
            "value_type": "numeric",
            "value_boolean": pd.NA,
            "value_numeric": float(len(assigned) / max(len(assign), 1)),
            "value_display": f"{len(assigned) / max(len(assign), 1):.6f}",
            "status": "INFO",
            "criterion": "Reported descriptively; no artificial target percentage",
        },
        {
            "check": "Direct-parser share among final assignments",
            "value_type": "numeric",
            "value_boolean": pd.NA,
            "value_numeric": float(direct_share) if pd.notna(direct_share) else np.nan,
            "value_display": f"{direct_share:.6f}" if pd.notna(direct_share) else "NA",
            "status": "INFO",
            "criterion": "Semantic fallback is separately identified and calibrated",
        },
        {
            "check": "Silver-standard sentence-function top-1 accuracy",
            "value_type": "numeric",
            "value_boolean": pd.NA,
            "value_numeric": float(calibration_accuracy) if pd.notna(calibration_accuracy) else np.nan,
            "value_display": f"{calibration_accuracy:.6f}" if pd.notna(calibration_accuracy) else "NA",
            "status": (
                "PASS"
                if pd.notna(calibration_accuracy) and calibration_accuracy >= 0.70
                else "WARN"
            ),
            "criterion": ">=70% desirable; calibration grid additionally enforces high precision for accepted fallback",
        },
        {
            "check": "Primary opportunity count",
            "value_type": "integer",
            "value_boolean": pd.NA,
            "value_numeric": float(primary_count),
            "value_display": str(int(primary_count)),
            "status": "INFO",
            "criterion": f"Primary status requires TV>0, TR>0, n>={PRIMARY_MIN_CLUSTER_SIZE}",
        },
        {
            "check": "Emerging opportunity count",
            "value_type": "integer",
            "value_boolean": pd.NA,
            "value_numeric": float(emerging_count),
            "value_display": str(int(emerging_count)),
            "status": "INFO",
            "criterion": f"Emerging status: TV>0, TR>0, n<{PRIMARY_MIN_CLUSTER_SIZE}",
        },
    ])

    qa["value_boolean"] = qa["value_boolean"].astype("boolean")
    qa["value_numeric"] = pd.to_numeric(qa["value_numeric"], errors="coerce").astype("float64")
    qa["value_display"] = qa["value_display"].astype(str)

    qa.to_parquet(
        FINAL_QA_PATH,
        index=False
    )
    save_table(
        qa,
        "15_FINAL_QA_V4_1",
        print_full=True
    )

    failures = qa[
        qa["status"]
        == "FAIL"
    ]
    if len(failures):
        raise RuntimeError(
            "FINAL V4 STRUCTURAL QA FAILED:\n"
            + failures.to_string(
                index=False
            )
        )

    validation = pd.DataFrame([
        {
            "validation_component": "Reference-field assignment",
            "status": "Evidence-gated",
            "basis": "Representative CPC class must be observed in the patent-family classifications.",
        },
        {
            "validation_component": "Function assignment",
            "status": "Evidence-gated",
            "basis": "Direct normalized function extraction preferred; sentence-semantic fallback must pass calibrated similarity and margin thresholds.",
        },
        {
            "validation_component": "Opportunity membership",
            "status": "Evidence-gated",
            "basis": "Patent enters an opportunity only when both field and function evidence are present.",
        },
        {
            "validation_component": "Key-patent traceability",
            "status": "Completed",
            "basis": "Each key patent carries the exact sentence used as its function evidence.",
        },
        {
            "validation_component": "Historical predictive validation",
            "status": "Not claimed",
            "basis": "Current export lacks dated forward-citation events; historical use of cumulative counts would leak future information.",
        },
        {
            "validation_component": "Expert validation",
            "status": "External human step",
            "basis": "Experts can evaluate transferability and feasibility using the saved key patents, function sentences, CPC evidence, and mechanism tables.",
        },
    ])

    save_table(
        validation,
        "15_validation_readiness_FINAL_V4",
        print_full=True
    )

    # ---------------- Final LLM prompt package ----------------
    primary = metrics[
        metrics[
            "primary_key_cluster"
        ]
    ].sort_values(
        "TV_plus_TR",
        ascending=False
    ).head(
        N_LLM_CLUSTERS
    )

    prompt_rows = []

    for _, r in primary.iterrows():
        cid = int(
            r["cluster_id"]
        )

        ref_nodes = nodes[
            (
                nodes[
                    "cluster_id"
                ] == cid
            )
            & (
                nodes[
                    "network_type"
                ]
                == "CPC_subclass"
            )
        ].head(5)

        mechs = supported_mech[
            supported_mech[
                "cluster_id"
            ] == cid
        ].head(8)

        kp = keyp[
            keyp[
                "cluster_id"
            ] == cid
        ].head(8)

        sent = keysent[
            keysent[
                "cluster_id"
            ] == cid
        ].head(15)

        prompt = f"""
You are a technology innovation strategist supporting end-of-life recycling and product-recyclability R&D.

Use ONLY the supplied evidence-gated opportunity cluster.

TARGET TECHNOLOGY
{TARGET_TECHNOLOGY}

FIELD OF INTEREST
{FIELD_OF_INTEREST}

OPPORTUNITY
Cluster ID: {cid}
Reference field: {r['reference_field']}
Function: {r['function']}
TV={r['TV']:.4f}; TR={r['TR']:.4f}; TN={r['TN']:.4f}; ROS={r['ROS']:.4f}
Cluster size={int(r['cluster_size'])}
Direct-parser function evidence share={r['direct_function_evidence_share']:.1%}

KEY CPC SUBCLASSES
{ref_nodes[['node','frequency','centrality']].to_string(index=False) if len(ref_nodes) else 'None'}

RECURRENT MECHANISMS
{mechs[['mechanism','patent_families','instances']].to_string(index=False) if len(mechs) else 'No mechanism met the >=2-family recurrence rule.'}

KEY PATENTS
{kp[['lens_id','title','function_evidence_source','function_similarity','key_patent_score']].to_string(index=False) if len(kp) else 'None'}

TRACEABLE FUNCTION-EVIDENCE SENTENCES
{sent[['lens_id','key_function','function_evidence_source','function_similarity','sentence']].to_string(index=False) if len(sent) else 'None'}

Generate exactly {N_SOLUTIONS_PER_CLUSTER} technically distinct cross-field concepts.
For each concept report:
- title
- core innovation
- transferred function
- adapted mechanism
- technical feasibility
- end-of-life recycling/recyclability contribution
- differentiation from the supplied evidence
- implementation pathway
- technical challenges
- exact Lens IDs / evidence sentences used

Do not claim patentability, freedom-to-operate, or legal clearance.
""".strip()

        p = (
            LLM_DIR
            / f"cluster_{cid:03d}_FINAL_V4_prompt.txt"
        )
        atomic_write_text(
            p,
            prompt
        )

        R.emit(
            f"\n===== FINAL V4 LLM PROMPT — CLUSTER {cid} =====\n{prompt}\n"
        )

        prompt_rows.append({
            "cluster_id": cid,
            "label_text": r[
                "label_text"
            ],
            "prompt_path": str(p),
            "TV": r["TV"],
            "TR": r["TR"],
            "TN": r["TN"],
            "ROS": r["ROS"],
        })

    prompt_index = pd.DataFrame(
        prompt_rows
    )
    prompt_index.to_parquet(
        LLM_PROMPT_INDEX,
        index=False
    )
    save_table(
        prompt_index,
        "15_llm_prompt_index_FINAL_V4",
        print_full=True
    )

    body = f"""
The final QA confirms that every retained opportunity membership has explicit CPC field evidence and a traceable
function-evidence sentence, and every key patent retains that sentence. The final manuscript-ready set contains
**{primary_count} primary opportunities** and **{emerging_count} smaller emerging/exploratory opportunities**.

No target assignment percentage is imposed. Coverage is reported as an empirical consequence of evidence gating.
"""
    R.update_section(
        "15_validation_llm",
        "Final evidence-gated QA and manuscript-ready opportunities",
        body
    )
    R.emit(body)

    marker.write_text(
        "done",
        encoding="utf-8"
    )
    mark_stage_done(
        stage,
        required,
        extra={
            "primary_opportunities": primary_count,
            "emerging_opportunities": emerging_count,
            "final_assignment_rate": (
                len(assigned)
                / max(len(assign),1)
            ),
            "structural_failures": int(
                len(failures)
            ),
        }
    )


# ============================================================
# FINAL V4 run order
# ============================================================

def run_pipeline():
    start = time.time()

    R.emit(
        "\nBEGINNING / RESUMING FINAL V4 EVIDENCE-GATED RECYCLE TOA PIPELINE"
    )
    R.emit(
        f"Revision: {PIPELINE_REVISION}"
    )
    R.emit(
        f"Start time: {datetime.now().isoformat()}"
    )

    # Reuse stable, expensive stages.
    stage_01_ingest()
    stage_02_family_dedup()
    stage_02b_descriptives()
    stage_03_patent_embeddings()
    stage_04_umap()
    stage_05_hdbscan_tuning()
    stage_06_base_bertopic()
    stage_07_reference_fields()
    stage_08_extract_mechanism_functions()
    stage_09_normalize_score_functions()
    stage_10_circularity_relevance()

    # Final evidence-gated downstream analysis.
    stage_11_zeroshot_assignment()
    stage_12_field_sentence_corpus()
    stage_13_cluster_metrics()
    stage_14_key_cluster_analysis()
    stage_15_validation_and_llm_prompts()
    stage_16_finalize()

    elapsed = (
        time.time()
        - start
    ) / 60

    R.emit(
        f"\nFINAL V4 PIPELINE COMPLETE in {elapsed:.1f} minutes."
    )
    R.emit(
        f"MASTER REPORT: {MASTER_REPORT_PATH}"
    )
    R.emit(
        f"FINAL OPPORTUNITY TABLE: {FINAL_OPPORTUNITY_TABLE}"
    )
    R.emit(
        f"FINAL QA: {FINAL_QA_PATH}"
    )
    R.emit(
        f"OUTPUT ROOT: {OUTPUT_ROOT}"
    )


# %% CELL: RUN
# ============================================================
# RUN / RESUME THE COMPLETE PIPELINE
# ============================================================

def run_pipeline():
    start = time.time()
    R.emit("\nBEGINNING / RESUMING RECYCLE TOA PIPELINE")
    R.emit(f"Start time: {datetime.now().isoformat()}")

    stage_01_ingest()
    stage_02_family_dedup()
    stage_02b_descriptives()
    stage_03_patent_embeddings()
    stage_04_umap()
    stage_05_hdbscan_tuning()
    stage_06_base_bertopic()
    stage_07_reference_fields()
    stage_08_extract_mechanism_functions()
    stage_09_normalize_score_functions()
    stage_10_circularity_relevance()
    stage_11_zeroshot_assignment()
    stage_12_field_sentence_corpus()
    stage_13_cluster_metrics()
    stage_14_key_cluster_analysis()
    stage_15_validation_and_llm_prompts()
    stage_16_finalize()

    elapsed = (time.time() - start) / 60
    R.emit(f"\nPIPELINE COMPLETE / RESUMED SUCCESSFULLY in {elapsed:.1f} minutes.")
    R.emit(f"MASTER REPORT: {MASTER_REPORT_PATH}")
    R.emit(f"OUTPUT ROOT: {OUTPUT_ROOT}")

run_pipeline()
